1. 共用參數設定


In [ ]:
'''
Measure the elapsed time of every code cell, including cells that raise errors.
記錄每個 Code Cell 的執行時間，即使發生錯誤也顯示已耗費的時間。
'''
from time import perf_counter as _cell_perf_counter
from IPython import get_ipython as _cell_get_ipython

_CELL_TIMER_SHELL = _cell_get_ipython()
if _CELL_TIMER_SHELL is None:
    raise RuntimeError('請在 Jupyter Notebook 中執行本教材。')
_CELL_TIMER_STATE = {'started': _cell_perf_counter()}

def _cell_timer_begin(_info):
    _CELL_TIMER_STATE['started'] = _cell_perf_counter()

def _cell_timer_end(result):
    started = _CELL_TIMER_STATE.pop('started', None)
    if started is None:
        return
    minutes, seconds = divmod(_cell_perf_counter() - started, 60)
    elapsed = f'{int(minutes)} 分 {seconds:05.2f} 秒' if minutes else f'{seconds:.2f} 秒'
    status = '執行失敗' if result.error_before_exec or result.error_in_exec else '執行完成'
    print(f'⏱ 本格{status}，耗時：{elapsed}')

'''
Replace existing timers when the first cell is executed again.
重新執行第一格時，先取消舊計時器，避免重複顯示耗時。
'''
for _event_name, _callback in globals().get('_CELL_TIMER_CALLBACKS', ()):
    try:
        _CELL_TIMER_SHELL.events.unregister(_event_name, _callback)
    except ValueError:
        pass
_CELL_TIMER_CALLBACKS = (
    ('pre_run_cell', _cell_timer_begin),
    ('post_run_cell', _cell_timer_end),
)
for _event_name, _callback in _CELL_TIMER_CALLBACKS:
    _CELL_TIMER_SHELL.events.register(_event_name, _callback)

'''
Import standard-library tools for editable paths and local service settings.
匯入設定路徑與本機服務參數所需的 Python 內建工具。
'''
import os
from pathlib import Path
ROOT = Path('/kaggle/working')
COMFY = ROOT / 'ComfyUI-MiniMaxH3'

'''
Store large downloaded GGUF weights in scratch storage.
設定 GGUF 模型權重的暫存磁碟位置，減少工作空間占用。
'''
STORE = Path('/tmp/minimax_h3_models')
PORT = 8193  # 本機可用 Port
H3_COMFY_REF = 'v0.45.0'

'''
Set the Hugging Face cache directory before any model downloads.
在下載模型之前指定 Hugging Face 快取目錄，避免大型檔案占滿工作空間。
'''
HF_CACHE_DIR = Path('/tmp/minimax_h3_hf')
os.environ['HF_HOME'] = str(HF_CACHE_DIR)
os.environ['HF_HUB_CACHE'] = str(HF_CACHE_DIR / 'hub')
os.environ['HF_XET_CHUNK_CACHE_SIZE_BYTES'] = '0'

'''
Select the shared GGUF quantization. Larger formats may require CPU offload on T4.
選擇共用 GGUF 量化精度；越大的格式越可能需要 CPU Offload。
'''
QUANT = 'Q5_0'  # 'Q4_K', 'Q5_0', 'Q6_K', 'Q8_0'
DEFAULT_QUANT = QUANT

'''
Set GPU 0 memory budget in GiB; 14.5 is aggressive on a 15 GiB T4.
設定 GPU 0 的記憶體預算（GiB）；15 GiB T4 設為 14.5 剩餘緩衝很小，OOM 可調低。
'''
GPU0_MODEL_GIB = 14.5  # 14.5、14.0、13.5
GPU1_MODEL_GIB = 14.5  # 14.5、14.0、13.5

'''
Select the text-encoder weight placement strategy.
設定文字編碼器權重管理：dynamic 分批調度較節省 GPU 記憶體；streaming 使用時整體載入；resident 盡量常駐 GPU 記憶體。
'''
TEXT_MODE = 'dynamic'  # 'dynamic', 'streaming', 'resident'
TEXT_GPU = 'cuda:1'  # 'cuda:1'（預設）、'cuda:0'（需留意 GPU 記憶體用量）
NODE_CACHE_MODE = 'classic'  # 'classic', 'none'

'''
Select the optional remote FIFO API port; change only if the port conflicts.
指定選用 FIFO 遠端 API 的本機連接埠；只有發生 Port 衝突時才需要修改。
'''
REMOTE_PORT = 8765  # 例如 8765、8766，範圍 1024～65535
REMOTE_SERVICE_ID = '06_MiniMax_H3_影片生成_Kaggle'
REMOTE_ROOT = ROOT / 'remote_api_jobs' / REMOTE_SERVICE_ID

'''
Limit how many remote jobs wait for the single GPU worker.
設定 FIFO 最多等待的工作數量；超出時回傳 HTTP 429，client 會延遲重試。
'''
REMOTE_QUEUE_CAPACITY = 32  # 建議 12～32，依課堂人數調整
REMOTE_UPLOAD_CONCURRENCY = 2  # 建議 1～2
REMOTE_RESULT_TTL_HOURS = 24
REMOTE_RESULT_LIMIT_GIB = 8
REMOTE_MIN_FREE_GIB = 2

'''
Mark exceptionally long remote inference as stalled, without forcibly starting a second GPU job.
設定遠端推論超時警戒值；卡住時不強行啟動第二筆 GPU 推論。
'''
REMOTE_STALL_HOURS = 2
GGUF_REPO = 'robertbschuette/MiniMax-H3-GGUF'
R2V_REF_DIR = ROOT / 'minimax_h3_r2v_references'

'''
Validate user-editable local ports and remote queue limits.
確認教師設定的本機 Port 與遠端佇列限制有效，避免服務啟動後才失敗。
'''
assert 1024 <= REMOTE_PORT <= 65535, 'REMOTE_PORT 必須介於 1024～65535。'
assert 1 <= REMOTE_QUEUE_CAPACITY <= 1024, 'FIFO 佇列容量設定無效。'
assert 1 <= REMOTE_UPLOAD_CONCURRENCY <= 8, '上傳並發限制設定無效。'
assert REMOTE_RESULT_TTL_HOURS > 0 and REMOTE_RESULT_LIMIT_GIB > 0 and REMOTE_MIN_FREE_GIB >= 0
assert REMOTE_STALL_HOURS > 0
assert 1024 <= PORT <= 65535 and PORT != REMOTE_PORT
print('✓ 第 1 節完成：共用參數已設定，請繼續執行環境安裝。')


2. 安裝環境與匯入套件


In [ ]:
'''
Import the libraries needed for this section.
匯入本段程式所需的函式庫。
'''
import os
import subprocess
import sys
from pathlib import Path
import torch
assert torch.cuda.is_available() and torch.cuda.device_count() == 2, '請啟用 Kaggle T4 x2 GPU。'
for i in range(2):
    print(f'GPU {i}: {torch.cuda.get_device_name(i)}, {torch.cuda.get_device_properties(i).total_memory / 1024**3:.2f} GiB')

'''
Define the repositories needed for the local workflow.
定義本機 Workflow 需要的程式庫。
'''
repositories = {
    COMFY: 'https://github.com/Comfy-Org/ComfyUI.git',
    COMFY / 'custom_nodes/ComfyUI-GGUF': 'https://github.com/leejet/ComfyUI-GGUF.git',
    COMFY / 'custom_nodes/ComfyUI-ClipProj': 'https://github.com/nicolab28/ComfyUI-ClipProj.git',
}
old_commits = {}
for directory, url in repositories.items():
    if directory.exists() and not (directory / '.git').is_dir():
        raise RuntimeError(f'目錄不是 Git 倉庫：{directory}')
    if (directory / '.git').is_dir():
        old_commits[directory] = subprocess.check_output(['git', '-C', str(directory), 'rev-parse', 'HEAD'], text=True).strip()
    else:
        directory.parent.mkdir(parents=True, exist_ok=True)
        command = ['git', 'clone', '--depth', '1']

        '''
        Choose the release tag only for ComfyUI core.
        只有 ComfyUI 核心指定發行版標籤。
        '''
        if directory == COMFY:
            command += ['--branch', H3_COMFY_REF]
        subprocess.run(command + [url, str(directory)], check=True)

    '''
    The archived or patched GGUF loader may contain local compatibility changes.
    GGUF 載入器可能含有必要的本機相容性修正。
    '''
    if directory.name == 'ComfyUI-GGUF':
        dirty = subprocess.check_output(['git', '-C', str(directory), 'status', '--porcelain'], text=True).strip()
        if not dirty:
            subprocess.run(['git', '-C', str(directory), 'pull', '--ff-only', '-q'], check=True)
        else:
            print('GGUF 載入器已有本機修改，本次略過自動更新。')

'''
Retrieve the official ComfyUI release on repeat runs.
重跑時同樣取得官方 ComfyUI 發行版本。
'''
subprocess.run(['git', '-C', str(COMFY), 'fetch', '--depth', '1', 'origin', 'tag', H3_COMFY_REF], check=True)
subprocess.run(['git', '-C', str(COMFY), 'checkout', '--quiet', '--detach', H3_COMFY_REF], check=True)
H3_CORE_UPDATED = any(
    directory in old_commits and old_commits[directory] != subprocess.check_output(
        ['git', '-C', str(directory), 'rev-parse', 'HEAD'], text=True,
    ).strip()
    for directory in (COMFY, COMFY / 'custom_nodes/ComfyUI-GGUF')
)

'''
Update ClipProj to its current compatible implementation if this folder already existed.
若 ClipProj 資料夾原本就存在，同步更新到支援新版投影矩陣的程式。
'''
clipproj_dir = COMFY / 'custom_nodes/ComfyUI-ClipProj'
clipproj_before = subprocess.check_output(['git', '-C', str(clipproj_dir), 'rev-parse', 'HEAD'], text=True).strip()
subprocess.run(['git', '-C', str(clipproj_dir), 'fetch', '--quiet', '--depth', '1', 'origin', 'main'], check=True)

'''
Switch the local custom node to the retrieved revision for consistent code and weights.
切換到取得的版本，確保節點程式與投影矩陣相容。
'''
subprocess.run(['git', '-C', str(clipproj_dir), 'checkout', '--quiet', '--force', 'FETCH_HEAD'], check=True)
H3_CLIPPROJ_UPDATED = clipproj_before != subprocess.check_output(['git', '-C', str(clipproj_dir), 'rev-parse', 'HEAD'], text=True).strip()
clipproj_source = (clipproj_dir / 'clipproj_nodes.py').read_text(encoding='utf-8')

'''
Fail before loading models if an outdated fork lacks the required interfaces.
若外掛版本過舊且缺少必要介面，在載入模型前停止。
'''
assert 'def clip_types(' in clipproj_source and 'build_residual' in clipproj_source, 'ClipProj 缺少新版必要功能，請檢查安裝版本。'
constraint = ROOT / 'minimax_h3_torch_constraints.txt'
constraint.write_text(f'torch=={torch.__version__}\ntransformers>=4.50.3,<5.0\n', encoding='utf-8')

'''
Install ComfyUI dependencies and the GGUF backend.
安裝 ComfyUI 相依套件與 GGUF 後端。
'''
for requirements in [COMFY / 'requirements.txt', COMFY / 'custom_nodes/ComfyUI-GGUF/requirements.txt']:
    if requirements.is_file():
        subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', '-c', str(constraint), '-r', str(requirements)], check=True)

subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', '-c', str(constraint), 'huggingface_hub', 'hf_xet', 'requests', 'websocket-client', 'tqdm', 'ipywidgets>=8,<9'], check=True)
print('ComfyUI 與 MiniMax H3 節點安裝完成。')

'''
Import a small regex helper for the H3 GGUF architecture registration.
載入簡單的正規表示式工具，以註冊 H3 GGUF 架構。
'''
import re
gguf_loader_path = COMFY / 'custom_nodes/ComfyUI-GGUF/loader.py'
assert gguf_loader_path.is_file(), f'找不到 GGUF 載入器：{gguf_loader_path}'

'''
Read the exact installed GGUF implementation instead of assuming a fixed version.
讀取實際安裝的 GGUF 原始碼，不假設版本永遠固定。
'''
gguf_source = gguf_loader_path.read_text(encoding='utf-8')
arch_match = re.search(r'IMG_ARCH_LIST\s*=\s*\{[^}]*\}', gguf_source)
assert arch_match, '無法定位 IMG_ARCH_LIST；請檢查 ComfyUI-GGUF 版本。'

'''
Record whether the running ComfyUI process would require a reload.
記錄是否需要讓已啟動的 ComfyUI 重新載入節點。
'''
H3_RESTART_REQUIRED = H3_CLIPPROJ_UPDATED or H3_CORE_UPDATED
if '"minimax_h3"' not in arch_match.group(0):
    previous_arch_list = arch_match.group(0)[:-1].rstrip()
    separator = '' if previous_arch_list.endswith(('{', ',')) else ','
    updated_arch_list = f'{previous_arch_list}{separator} "minimax_h3"}}'
    gguf_source = gguf_source[:arch_match.start()] + updated_arch_list + gguf_source[arch_match.end():]
    gguf_loader_path.write_text(gguf_source, encoding='utf-8')
    H3_RESTART_REQUIRED = True
    print('已加入 MiniMax-H3 GGUF 架構相容設定。')

print('✓ 第 2 節完成：ComfyUI、GGUF、ClipProj 與 WebSocket 工具已安裝／檢查。')


3. 下載量化模型（Q4／Q5／Q6／Q8，權重保留於 `/tmp`）


In [ ]:
'''
Import the libraries needed for this section.
匯入本段程式所需的函式庫。
'''
import os
import shutil
os.environ['HF_HOME'] = str(HF_CACHE_DIR)
os.environ['HF_HUB_CACHE'] = str(HF_CACHE_DIR / 'hub')

'''
Disable extra Xet chunk caching to conserve temporary disk space.
停用額外的 Xet 分塊快取，節省暫存空間。
'''
os.environ['HF_XET_CHUNK_CACHE_SIZE_BYTES'] = '0'
from huggingface_hub import hf_hub_download

'''
Choose the generation mode from the four supported task strings. flf2v = first-and-last-frame-to-video; r2v = multi-reference-to-video (Ref2VA model). Available choices: t2v, i2v, flf2v, r2v.
從四種合法字串中選擇生成模式。 t2v = text-to-video (文字生成影片); i2v = image-to-video (首張圖片生成影片). t2v 為文生影片；i2v 為首張圖片生成影片。 flf2v 為首尾影格生成影片；r2v 為多張參考圖生成影片（需使用 Ref2VA 權重）。 可選值：t2v、i2v、flf2v、r2v。
'''
TASK = globals().get('TASK', 't2v')  # 't2v', 'i2v', 'l2v', 'flf2v', 'r2v'
assert TASK in ('t2v', 'i2v', 'l2v', 'flf2v', 'r2v'), f'不支援的 TASK：{TASK}'
MODEL_VARIANT = 'ref2va_pruned' if TASK == 'r2v' else 'fl2va_pruned'

'''
Use edited task parameters when the corresponding settings cell has run.
若已執行對應的參數 Cell，則優先採用使用者修改後的量化選項。
'''
task_settings = globals().get('TASK_CONFIGS', {}).get(TASK, {})
FL2VA_QUANT_SIZES_GIB = {'Q4_K': 10.64, 'Q5_0': 12.97, 'Q6_K': 15.45, 'Q8_0': 19.97}
REF2VA_QUANT_SIZES_GIB = {'Q4_K': 10.60, 'Q5_0': 12.94, 'Q6_K': 15.42, 'Q8_0': 19.94}

'''
Select the size table matching the actual diffusion model family.
依照真正要使用的模型種類選擇大小對照表。
'''
QUANT_SIZES_GIB = REF2VA_QUANT_SIZES_GIB if TASK == 'r2v' else FL2VA_QUANT_SIZES_GIB
assert QUANT in QUANT_SIZES_GIB, f'不支援 QUANT={QUANT!r}；請選擇 {list(QUANT_SIZES_GIB)}。'

'''
Build the actual published filename, never reusing FL2VA weights for R2V.
根據實際發布檔名選擇模型，R2V 絕不沿用 FL2VA 權重。
'''
DIT_NAME = f'minimax_h3_{MODEL_VARIANT}-{QUANT}.gguf'
models = [
    (GGUF_REPO, DIT_NAME, 'diffusion_models'),
    ('Comfy-Org/Krea-2', 'text_encoders/qwen3vl_4b_fp8_scaled.safetensors', 'text_encoders'),
    ('NicoLab28/ClipProj-MiniMax-H3', 'mmh3-4b-ClipProj-v3.1-mlp.safetensors', 'clip_projections'),
    ('Comfy-Org/MiniMax-H3', 'vae/minimax_h3_video_vae_fp16.safetensors', 'vae'),
    ('Comfy-Org/MiniMax-H3', 'vae/minimax_h3_audio_vae_fp32.safetensors', 'vae'),
]
AUX_SIZES_GIB = {
    'qwen3vl_4b_fp8_scaled.safetensors': 4.88,
    'mmh3-4b-ClipProj-v3.1-mlp.safetensors': 0.47,
    'minimax_h3_video_vae_fp16.safetensors': 4.85,
    'minimax_h3_audio_vae_fp32.safetensors': 0.56,
}

'''
Keep the valid downloaded GGUF from either the current or historical folder.
可沿用目前或舊版資料夾中已完整下載的 GGUF 權重。
'''
GGUF_CACHE_PATHS = (STORE / 'diffusion_models' / DIT_NAME, STORE / 'unet_gguf' / DIT_NAME)
missing_gib = 0.0
for repo, filename, folder in models:
    size_gib = QUANT_SIZES_GIB[QUANT] if folder == 'diffusion_models' else AUX_SIZES_GIB[Path(filename).name]
    candidates = GGUF_CACHE_PATHS if folder == 'diffusion_models' else (STORE / folder / Path(filename).name,)
    complete = any(path.is_file() and 0.97 * size_gib <= path.stat().st_size / 1024**3 <= 1.03 * size_gib for path in candidates)

    '''
    Count only components that still need to be downloaded.
    只累計尚未完整下載的模型大小。
    '''
    if not complete:
        missing_gib += size_gib

'''
Read the free space of the scratch filesystem used for all model files.
讀取實際存放所有模型權重的暫存磁碟可用容量。
'''
tmp_free_gib = shutil.disk_usage('/tmp').free / 1024**3
assert tmp_free_gib >= missing_gib + 3, (
    f'/tmp 空間不足：仍需下載約 {missing_gib:.2f} GiB，'
    f'目前剩餘 {tmp_free_gib:.2f} GiB（另需預留 3 GiB）。'
)
print(f'選擇 {MODEL_VARIANT}/{QUANT}｜去噪權重約 {QUANT_SIZES_GIB[QUANT]:.2f} GiB｜'
      f'尚需下載約 {missing_gib:.2f} GiB｜/tmp 可用 {tmp_free_gib:.1f} GiB')

'''
Warn when high-bit quantizations will require greater GPU/CPU offload.
高精度量化需要較多 GPU／CPU 卸載時，事先提醒使用者。
'''
if QUANT in ('Q6_K', 'Q8_0'):
    print('注意：Q6／Q8 可能需要大量 CPU Offload，速度較慢且有記憶體不足風險。')

for repo, filename, folder in models:
    print('模型：', Path(filename).name, flush=True)
    source = next((path for path in GGUF_CACHE_PATHS
                   if folder == 'diffusion_models' and path.is_file()
                   and 0.97 * QUANT_SIZES_GIB[QUANT] <= path.stat().st_size / 1024**3 <= 1.03 * QUANT_SIZES_GIB[QUANT]), None)

    '''
    Download files with Hub-managed validation when no complete cache exists.
    沒有完整快取時，交由 Hugging Face Hub 管理下載及檔案驗證。
    '''
    if source is None:
        local_file = STORE / folder / Path(filename).name
        expected_gib = QUANT_SIZES_GIB[QUANT] if folder == 'diffusion_models' else AUX_SIZES_GIB[Path(filename).name]
        invalid_cache = local_file.is_file() and not (0.97 * expected_gib <= local_file.stat().st_size / 1024**3 <= 1.03 * expected_gib)

        '''
        Keep models under /tmp without copying gigabytes into Kaggle working.
        模型實體仍放在 /tmp，避免複製大型檔案到 Kaggle working。
        '''
        source = Path(hf_hub_download(repo_id=repo, filename=filename,
                                      local_dir=STORE / folder, force_download=invalid_cache))
    expected_gib = QUANT_SIZES_GIB[QUANT] if folder == 'diffusion_models' else AUX_SIZES_GIB[Path(filename).name]
    assert 0.97 * expected_gib <= source.stat().st_size / 1024**3 <= 1.03 * expected_gib, f'模型檔案大小異常：{source}'
    target = COMFY / 'models' / folder / source.name
    target.parent.mkdir(parents=True, exist_ok=True)

    '''
    Keep an already-correct symbolic link to avoid unnecessary service restarts.
    已指向正確權重的符號連結直接沿用，避免不必要的服務重新啟動。
    '''
    if target.is_symlink() and (not target.exists() or not target.samefile(source)):
        target.unlink()
    if not target.exists():
        target.symlink_to(source.resolve())
        H3_RESTART_REQUIRED = True
    assert target.samefile(source), f'模型檔案衝突：{target} 與預期來源 {source} 不同。請先檢查舊檔。'
    assert target.is_file(), f'模型連結失敗：{target}'
    print('ComfyUI 路徑：', target)

obsolete_link = COMFY / 'models' / 'unet_gguf' / DIT_NAME
if obsolete_link.is_symlink():
    obsolete_link.unlink()

'''
Confirm all requested model paths can be opened.
確認所有所需模型檔案都可正常存取。
'''
assert all((COMFY / 'models' / folder / Path(filename).name).is_file() for _, filename, folder in models)
print(f'H3 {MODEL_VARIANT}/{QUANT}、4B ClipProj、Video VAE、Audio VAE 已就緒。')
H3_MODEL_CHOICE = (MODEL_VARIANT, QUANT)

'''
Remind users to revalidate the running loader after changing quantizations.
提醒切換量化格式後，先更新目前 ComfyUI 的模型選單。
'''
print('切換量化格式後：請重新執行第 4 節，再執行第 11 節生成。')
print(f'✓ 第 3 節完成：{MODEL_VARIANT}/{QUANT} 模型與相關權重已備妥（/tmp）。')


4. GPU 記憶體與 CPU Offload 設定


In [ ]:
'''
Require the teacher settings cell before applying ComfyUI GPU configuration.
套用 ComfyUI GPU 設定前，確認已執行教師共用設定 Cell。
'''
assert all(name in globals() for name in ('GPU0_MODEL_GIB', 'GPU1_MODEL_GIB', 'TEXT_MODE', 'TEXT_GPU', 'NODE_CACHE_MODE')), '請先執行「教師共用設定」Cell。'
gpu0_capacity = torch.cuda.get_device_properties(0).total_memory / 1024**3
gpu1_capacity = torch.cuda.get_device_properties(1).total_memory / 1024**3

'''
Reject values that exceed the available GPU memory.
避免設定超過 GPU 實際容量的數值。
'''
assert 0 < GPU0_MODEL_GIB <= gpu0_capacity, f'GPU0_MODEL_GIB 必須介於 0 和 {gpu0_capacity:.2f} GiB 之間。'
assert 0 < GPU1_MODEL_GIB <= gpu1_capacity, f'GPU1_MODEL_GIB 必須介於 0 和 {gpu1_capacity:.2f} GiB 之間。'
assert TEXT_MODE in ('dynamic', 'streaming', 'resident'), 'TEXT_MODE 設定無效。'

'''
Reserve the difference for GPU 0 so ComfyUI targets the chosen overall budget.
將 GPU 0 的容量差額保留，讓 ComfyUI 以指定的整體預算運作。
'''
GPU0_RESERVE_GIB = round(gpu0_capacity - GPU0_MODEL_GIB, 3)

assert NODE_CACHE_MODE in ('classic', 'none')
H3_DESIRED_CONFIG = (GPU0_RESERVE_GIB, NODE_CACHE_MODE)

'''
Clarify that per-model residency is not a hard limit.
明確提醒：這不是單一模型的精確權重常駐限制。
'''
print('GPU 0：ComfyUI Dynamic VRAM / CPU Offload（權重、運算暫存及其它元件共用預算）。')
print(f'GPU 1：ClipProj {TEXT_MODE}，{GPU1_MODEL_GIB:.2f} GiB 為監控警示值，並非硬上限。')
print(f'✓ 第 4 節完成：GPU 0 管理預算約 {GPU0_MODEL_GIB:.2f} GiB；保留 {GPU0_RESERVE_GIB:.2f} GB。')


5. 啟動本機 ComfyUI 與相依性檢查


In [ ]:
'''
Import the libraries needed for this section.
匯入本段程式所需的函式庫。
'''
import requests
import time
LOG = ROOT / 'minimax_h3_comfyui.log'
BASE = f'http://127.0.0.1:{PORT}'

'''
Read server readiness without requiring a public tunnel.
不開放公開連線即可檢查伺服器狀態。
'''
def ready():
    '''
    Handle an unavailable local service safely.
    安全處理尚未啟動的服務。
    '''
    try:
        return requests.get(f'{BASE}/system_stats', timeout=3).ok

    except requests.RequestException:
        return False

'''
Read the ComfyUI loader's actual model dropdown values.
讀取 ComfyUI GGUF Loader 真正提供的模型選項。
'''
def listed_ggufs():
    '''
    Retrieve the registered GGUF model names from the local service.
    從本機服務取得已註冊的 GGUF 模型檔名。
    '''
    payload = requests.get(f'{BASE}/object_info/UnetLoaderGGUF', timeout=30).json()
    info = payload.get('UnetLoaderGGUF', {})
    spec = info.get('input', {}).get('required', {}).get('unet_name')
    return spec[0] if isinstance(spec, list) and spec and isinstance(spec[0], list) else []

need_restart = False
if ready():
    need_restart = (
        globals().get('H3_RESTART_REQUIRED', False)
        or DIT_NAME not in listed_ggufs()
        or globals().get('H3_ACTIVE_MODEL') != (MODEL_VARIANT, QUANT)
        or globals().get('H3_ACTIVE_CONFIG') != H3_DESIRED_CONFIG
    )

'''
Never terminate services launched by another notebook or process.
不任意終止其它 Notebook 或行程啟動的 ComfyUI。
'''
if need_restart:
    '''
    Only restart a service for which this kernel holds the process handle.
    只有目前 Kernel 持有行程控制權時才能自動重新啟動。
    '''
    assert 'H3_PROC' in globals() and H3_PROC.poll() is None, (
        '服務需要更新記憶體設定，但不是本 Notebook 可管理的行程。'
        f'請先停止連接埠 {PORT} 的舊服務，再執行本 Cell。'
    )
    H3_PROC.terminate()
    try:
        H3_PROC.wait(timeout=20)

    except subprocess.TimeoutExpired:
        '''
        Handle a service that does not respond to graceful shutdown.
        處理無法正常關閉的服務。
        '''
        H3_PROC.kill()
        H3_PROC.wait(timeout=10)
    print('重新啟動 ComfyUI：模型索引、節點版本或 GPU 0 記憶體預算有變更。')

if not ready():
    command = [
        sys.executable, 'main.py', '--listen', '127.0.0.1', '--port', str(PORT),
        '--enable-dynamic-vram', '--reserve-vram', str(GPU0_RESERVE_GIB),
        '--cache-classic' if NODE_CACHE_MODE == 'classic' else '--cache-none',
        '--disable-pinned-memory',
    ]

    '''
    Keep the local service log available for debugging without blocking Kaggle.
    保留服務 Log 方便除錯，且不讓 Notebook 卡住。
    '''
    with LOG.open('w', encoding='utf-8') as stream:
        H3_PROC = subprocess.Popen(command, cwd=COMFY, stdout=stream, stderr=subprocess.STDOUT)
    for attempt in range(180):
        '''
        Report the original service error rather than waiting indefinitely.
        發生啟動錯誤時直接輸出 Log，避免無限等待。
        '''
        if H3_PROC.poll() is not None:
            raise RuntimeError(LOG.read_text(errors='replace')[-4000:])
        if ready():
            break
        time.sleep(2)
    assert ready(), LOG.read_text(errors='replace')[-4000:]

H3_ACTIVE_CONFIG = H3_DESIRED_CONFIG

'''
Get the exact input schemas registered by the running ComfyUI process.
從目前執行中的 ComfyUI 讀取真正註冊的節點輸入規格。
'''
NODE_INFO = requests.get(f'{BASE}/object_info', timeout=90).json()
REQUIRED_NODES = (
    'ClipProjLoader', 'UnetLoaderGGUF', 'MiniMaxH3SigmaShift',
    'VAELoader', 'MiniMaxH3ImageToVideo', 'BasicGuider', 'KSamplerSelect',
    'BasicScheduler', 'RandomNoise', 'SamplerCustomAdvanced', 'VAEDecode',
    'VAEDecodeAudio', 'CreateVideo', 'SaveVideo', 'LoadImage',
    'MiniMaxH3ReferenceToVideo',
)
missing = [name for name in REQUIRED_NODES if name not in NODE_INFO]

'''
Show a clear explanation for a failed or incomplete ComfyUI installation.
若 ComfyUI 安裝不完整，直接指出缺少的節點。
'''
assert not missing, f'ComfyUI 缺少節點：{missing}。請檢查 {LOG} 並重新執行安裝與啟動。'

'''
Read a dropdown's allowed values from ComfyUI's live schema.
從 ComfyUI 正在使用的規格讀取下拉選單允許值。
'''
def choices(node_name, input_name):
    '''
    Support both required and optional node inputs.
    同時支援必填與選填的節點輸入。
    '''
    groups = NODE_INFO[node_name].get('input', {})
    for group in ('required', 'optional'):
        spec = groups.get(group, {}).get(input_name)

        '''
        Return None for numeric/string inputs that are not dropdown lists.
        如果不是下拉式選單，就不視為有限選項。
        '''
        if spec is not None:
            if isinstance(spec, (list, tuple)) and spec:
                if isinstance(spec[0], list):
                    return spec[0]
                if len(spec) > 1 and isinstance(spec[1], dict):
                    for key in ('choices', 'options'):
                        if isinstance(spec[1].get(key), list):
                            return spec[1][key]
            return None
    raise RuntimeError(f'節點 {node_name} 沒有預期的輸入 {input_name}；可能是節點版本不相容.')

'''
Validate dropdown values without crashing if a node exposes only a scalar input.
驗證下拉選項；如果節點只提供一般輸入型別，就不要因此發生例外。
'''
def check_choice(node_name, input_name, value, fallback=None):
    '''
    Read the node's current explicit choice list if present.
    有明確選項清單時優先讀取目前節點的清單。
    '''
    allowed = choices(node_name, input_name)
    if allowed is None and fallback and fallback[0] in NODE_INFO:
        allowed = choices(*fallback)
    if allowed is None:
        print(f'預檢提示：{node_name}.{input_name} 沒有選項清單，將交由 ComfyUI 驗證。')
    else:
        assert value in allowed, f'{node_name}.{input_name} 不支援 {value!r}；可用選項：{allowed}'

assert DIT_NAME in (choices('UnetLoaderGGUF', 'unet_name') or []), f'GGUF Loader 沒有辨識 {DIT_NAME}。'
assert 'qwen3vl_4b_fp8_scaled.safetensors' in (choices('ClipProjLoader', 'clip_name') or []), 'ClipProj Loader 沒有辨識 4B Encoder。'

'''
Verify that the correct projection matrix is registered, not merely downloaded.
確認正確的投影矩陣已註冊，而不只是檔案下載成功。
'''
assert 'mmh3-4b-ClipProj-v3.1-mlp.safetensors' in (choices('ClipProjLoader', 'projection') or []), 'ClipProj Loader 沒有辨識 v3.1 投影矩陣。'
assert 'auto' in (choices('ClipProjLoader', 'type') or []), 'ClipProj Loader 不支援 type=auto，請更新節點。'
assert 'krea2' in (choices('ClipProjLoader', 'type') or []), '目前 ComfyUI 尚未提供 krea2／Qwen3-VL 4B 編碼器載入支援，請更新 ComfyUI。'

'''
Verify the supported video and audio VAE filenames.
確認影片與音訊 VAE 都出現在實際模型選單中。
'''
for vae_name in ('minimax_h3_video_vae_fp16.safetensors', 'minimax_h3_audio_vae_fp32.safetensors'):
    assert vae_name in (choices('VAELoader', 'vae_name') or []), f'ComfyUI 沒有辨識 VAE：{vae_name}'

print('依賴檢查通過：FL2VA、Ref2VA、GGUF、4B Encoder、ClipProj v3.1、Video/Audio VAE。')
H3_ACTIVE_QUANT = QUANT

'''
Remember the model family as well as its quantization.
同時記錄模型種類及量化格式，避免混用 FL2VA／Ref2VA。
'''
H3_ACTIVE_MODEL = (MODEL_VARIANT, QUANT)
H3_RESTART_REQUIRED = False
print(f'本機 MiniMax H3 {MODEL_VARIANT}/{QUANT} 已就緒（GGUF 模型選單驗證通過）。')

'''
Confirm that all service and model checks finished without an exception.
所有服務及模型檢查通過後，明確顯示本格完成。
'''
print(f'✓ 第 5 節完成：ComfyUI 服務與模型預檢完成；GPU 0 保留 {GPU0_RESERVE_GIB:.2f} GB。')


6. T2V：Prompt 與參數設定


In [ ]:
'''
Describe the text-to-video scene and synchronized sound.
設定文生影片的場景與同步音效描述。
'''
T2V_PROMPT = '''
A cinematic close-up of a red panda in a misty forest. The red panda slowly turns its head.
Natural fur details, gentle camera movement, soft morning light. Sound: birds chirping and light wind.
'''.strip()
T2V_WIDTH, T2V_HEIGHT = 384, 224
T2V_SECONDS = 10

'''
Choose sampling iterations for the original model.
設定原始模型的採樣步數。
'''
T2V_STEPS = 20
T2V_AUDIO_SHIFT = 6.0
T2V_SEED = 42

'''
Set the T2V output filename without overwriting earlier runs.
設定文生影片輸出檔名；若同名會自動編號。
'''
T2V_OUTPUT = 'minimax_h3_t2v.mp4'
TASK_CONFIGS = globals().get('TASK_CONFIGS', {})
TASK_CONFIGS['t2v'] = dict(prompt=T2V_PROMPT, width=T2V_WIDTH, height=T2V_HEIGHT,
                           seconds=T2V_SECONDS, steps=T2V_STEPS, seed=T2V_SEED,
                           audio_shift=T2V_AUDIO_SHIFT, quant=QUANT,
                           output_name=T2V_OUTPUT, first_image='', last_image='')

'''
Provide explicit feedback after configuring the task.
完成任務設定後顯示明確提示。
'''
TASK = 't2v'

print(f'✓ 第 6 節完成：T2V | {T2V_WIDTH}×{T2V_HEIGHT} | {T2V_SECONDS} 秒 | {T2V_STEPS} steps | {QUANT} | Audio Shift {T2V_AUDIO_SHIFT} | {T2V_OUTPUT}')


7. I2V：Prompt、起始圖片與參數設定


In [ ]:
'''
Describe how the uploaded first frame should animate.
描述上傳的第一張圖片要如何轉化為動態影片。
'''
I2V_PROMPT = '''
Animate the subject in the reference image with subtle natural motion and smooth cinematic camera movement.
Keep the person's or object's identity, clothing and environment consistent. Include realistic ambient sound.
'''.strip()
I2V_FIRST_IMAGE = str(ROOT / 'first_frame.png')
I2V_WIDTH, I2V_HEIGHT = 384, 224

'''
Configure output duration for image-to-video.
設定圖生影片的目標秒數。
'''
I2V_SECONDS = 10
I2V_STEPS = 20
I2V_AUDIO_SHIFT = 6.0

'''
Set this task’s seed; use -1 to randomize each generation.
設定本次任務的種子；-1 表示每次重新抽取。量化設定沿用第 1 節的 QUANT。
'''
I2V_SEED = 42
I2V_OUTPUT = 'minimax_h3_i2v.mp4'
TASK_CONFIGS = globals().get('TASK_CONFIGS', {})

'''
Register the first-frame conditioning input and task parameters.
登記起始影格及圖生影片參數。
'''
TASK_CONFIGS['i2v'] = dict(prompt=I2V_PROMPT, width=I2V_WIDTH, height=I2V_HEIGHT,
                           seconds=I2V_SECONDS, steps=I2V_STEPS, seed=I2V_SEED,
                           audio_shift=I2V_AUDIO_SHIFT, quant=QUANT,
                           output_name=I2V_OUTPUT, first_image=I2V_FIRST_IMAGE, last_image='')

'''
Select the current generation task and its model settings.
選擇目前生成模式及此模式的模型設定；尚未執行生成。
'''
TASK = 'i2v'

print(f'✓ 第 7 節完成：I2V | {I2V_WIDTH}×{I2V_HEIGHT} | {I2V_SECONDS} 秒 | {QUANT} | Audio Shift {I2V_AUDIO_SHIFT} | 圖片：{I2V_FIRST_IMAGE}')


8. L2VA：最後影格生成影片


In [ ]:
'''
Describe how the uploaded last frame should animate.
描述上傳的最後一張圖片要如何轉化為動態影片。
'''
L2V_PROMPT = '''
Animate the subject in the reference image with subtle natural motion and smooth cinematic camera movement.
Keep the person's or object's identity, clothing and environment consistent. Include realistic ambient sound.
'''.strip()
L2V_LAST_IMAGE = str(ROOT / 'last_frame.png')
L2V_WIDTH, L2V_HEIGHT = 384, 224

'''
Configure output duration for image-to-video.
設定最後影格生成影片的目標秒數。
'''
L2V_SECONDS = 10
L2V_STEPS = 20
L2V_AUDIO_SHIFT = 6.0

'''
Set this task’s seed; use -1 to randomize each generation.
設定本次任務的種子；-1 表示每次重新抽取。量化設定沿用第 1 節的 QUANT。
'''
L2V_SEED = 42
L2V_OUTPUT = 'minimax_h3_l2v.mp4'
TASK_CONFIGS = globals().get('TASK_CONFIGS', {})

'''
Register the last-frame conditioning input and task parameters.
登記最後影格及圖生影片參數。
'''
TASK_CONFIGS['l2v'] = dict(prompt=L2V_PROMPT, width=L2V_WIDTH, height=L2V_HEIGHT,
                           seconds=L2V_SECONDS, steps=L2V_STEPS, seed=L2V_SEED,
                           audio_shift=L2V_AUDIO_SHIFT, quant=QUANT,
                           output_name=L2V_OUTPUT, first_image='', last_image=L2V_LAST_IMAGE)

'''
Select the current generation task and its model settings.
選擇目前生成模式及此模式的模型設定；尚未執行生成。
'''
TASK = 'l2v'

print(f'✓ 第 8 節完成：L2V | {L2V_WIDTH}×{L2V_HEIGHT} | {L2V_SECONDS} 秒 | {QUANT} | Audio Shift {L2V_AUDIO_SHIFT} | 圖片：{L2V_LAST_IMAGE}')


9. FLF2V：Prompt、首尾圖片與參數設定


In [ ]:
'''
Describe a video that connects its first and last reference frames.
描述從第一張圖片過渡到最後一張圖片的影片內容。
'''
FLF2V_PROMPT = '''
Create a smooth cinematic transition from the first reference frame to the last reference frame.
Maintain consistent subject details and lighting, with natural intermediate motion and synchronized ambient sound.
'''.strip()
FLF2V_FIRST_IMAGE = str(ROOT / 'first_frame.png')
FLF2V_LAST_IMAGE = str(ROOT / 'last_frame.png')

'''
Set output dimensions for first-and-last-frame generation.
設定首尾影格模式的輸出解析度。
'''
FLF2V_WIDTH, FLF2V_HEIGHT = 384, 224
FLF2V_SECONDS = 10
FLF2V_STEPS = 20

'''
Set the audio sigma shift for this generation mode (ComfyUI default: 3.0).
設定這個生成模式的音訊 Sigma Shift（ComfyUI 預設：3.0）。
'''
FLF2V_AUDIO_SHIFT = 6.0
FLF2V_SEED = 42
FLF2V_OUTPUT = 'minimax_h3_flf2v.mp4'

'''
Keep previously configured modes unchanged.
保留先前其它生成模式的設定。
'''
TASK_CONFIGS = globals().get('TASK_CONFIGS', {})
TASK_CONFIGS['flf2v'] = dict(prompt=FLF2V_PROMPT, width=FLF2V_WIDTH, height=FLF2V_HEIGHT,
                             seconds=FLF2V_SECONDS, steps=FLF2V_STEPS, seed=FLF2V_SEED,
                             audio_shift=FLF2V_AUDIO_SHIFT, quant=QUANT,
                             output_name=FLF2V_OUTPUT, first_image=FLF2V_FIRST_IMAGE,
                             last_image=FLF2V_LAST_IMAGE)

'''
Select the current generation task and its model settings.
選擇目前生成模式及此模式的模型設定；尚未執行生成。
'''
TASK = 'flf2v'

print(f'✓ 第 9 節完成：FLF2V | {FLF2V_WIDTH}×{FLF2V_HEIGHT} | {FLF2V_SECONDS} 秒 | {QUANT} | Audio Shift {FLF2V_AUDIO_SHIFT} | 參考圖 2 張')


10. R2V：Prompt、圖片／影片／音訊參考


In [ ]:
'''
Describe how multiple reference images define the subject and environment.
使用多張參考圖片指定人物外觀、服裝及環境。
'''
R2V_PROMPT = '''
<Picture 1> defines the subject's face, hairstyle, and identity.
Additional pictures, when provided, help specify the outfit and environment.
Create a cinematic video of the subject walking naturally through the scene.
Preserve identity, clothing, and scene details from the reference pictures.
Natural motion, soft daylight, realistic ambient sounds.
'''.strip()
R2V_REF_IMAGES = []
R2V_REF_VIDEOS = []  # Example: ['/kaggle/working/clip.mp4']

'''
Optional standalone reference audio files (up to three).
Ref2VA 可選的獨立音訊路徑（最多三段，依 <Audio N> 編號排序）。
'''
R2V_REF_AUDIOS = []  # Example: ['/kaggle/working/voice.wav']
R2V_VIDEO_AUDIO = False  # True, False
R2V_REF_IMAGE_SIZE = 'match'

'''
Select the R2V-specific ClipProj mode: resident, dynamic, or streaming. resident = prioritize vision-encoder stability; dynamic = lower residency; streaming = page weights. Available choices: resident, dynamic, streaming.
R2V 專用 ClipProj 模式可選 resident、dynamic、streaming。 resident 優先保持影像編碼器常駐；dynamic 動態卸載；streaming 使用分段載入。 可選值：resident、dynamic、streaming。
'''
R2V_TEXT_MODE = TEXT_MODE
R2V_WIDTH, R2V_HEIGHT = 384, 224
R2V_SECONDS = 10

'''
Set the diffusion sampling steps for the normal Ref2VA model.
設定正常版 Ref2VA 模型的採樣步數。
'''
R2V_STEPS = 20
R2V_AUDIO_SHIFT = 3.0

'''
Set this task’s seed; use -1 to randomize each generation.
設定此任務的種子；-1 代表每次生成都重新抽取。
'''
R2V_SEED = 42
R2V_OUTPUT = 'minimax_h3_r2v.mp4'
TASK_CONFIGS = globals().get('TASK_CONFIGS', {})

'''
Save the reference media and settings without downloading extra models yet.
儲存參考圖與參數，不在此 Cell 下載額外權重。
'''
TASK_CONFIGS['r2v'] = dict(prompt=R2V_PROMPT, width=R2V_WIDTH, height=R2V_HEIGHT,
                            seconds=R2V_SECONDS, steps=R2V_STEPS, seed=R2V_SEED,
                            audio_shift=R2V_AUDIO_SHIFT, quant=QUANT,
                            output_name=R2V_OUTPUT, first_image='', last_image='',
                            ref_images=R2V_REF_IMAGES, ref_image_size=R2V_REF_IMAGE_SIZE,
                            ref_videos=R2V_REF_VIDEOS, ref_audios=R2V_REF_AUDIOS,
                            video_audio=R2V_VIDEO_AUDIO, text_mode=R2V_TEXT_MODE)

'''
Select the current generation task and its model settings.
選擇目前生成模式及此模式的模型設定；尚未執行生成。
'''
TASK = 'r2v'

print(f'✓ 第 10 節完成：R2V | {R2V_WIDTH}×{R2V_HEIGHT} | {R2V_SECONDS} 秒 | '
      f'{QUANT} | {len(R2V_REF_IMAGES)} 張圖片；若尚未下載 R2V 權重，請重新執行第 2、4 節。')

import ipywidgets as widgets
from pathlib import Path
from PIL import Image, ImageOps
from io import BytesIO
from uuid import uuid4
from html import escape
from IPython.display import display, clear_output
import shutil
_REF_ROOT = R2V_REF_DIR
_REF_ROOT.mkdir(parents=True, exist_ok=True)
_REF_MAX = 9
_REF_FILES = sorted(p for p in _REF_ROOT.iterdir() if p.is_file() and p.suffix.lower() in ('.png','.jpg','.jpeg','.webp'))
if len(_REF_FILES) > _REF_MAX or any(p.stem != f'{i:03d}' for i,p in enumerate(_REF_FILES,1)):
    raise RuntimeError('參考圖編號不連續；請清理舊資料夾再執行上傳 Cell。')

'''
Synchronize the numbered paths with the generation configuration.
將最新圖片清單同步到生成工作所使用的設定。
'''
def _ref_sync(paths):
    global R2V_REF_IMAGES
    R2V_REF_IMAGES = [str(path) for path in paths]
    TASK_CONFIGS['r2v']['ref_images'] = list(R2V_REF_IMAGES)

'''
Replace all reference images only after the new upload is fully validated.
新圖片全部通過驗證後才取代上一批；選得更少時也會清除多餘舊檔。
'''
def _ref_update(change):
    items = change['new']
    if not items:
        return
    if isinstance(items, dict):
        items = [dict(name=name, **item) for name, item in items.items()]
    stage = _REF_ROOT.with_name(_REF_ROOT.name + '_stage_' + uuid4().hex)
    stage.mkdir(parents=True)
    backup = _REF_ROOT.with_name(_REF_ROOT.name + '_backup_' + uuid4().hex)
    try:
        if not 1 <= len(items) <= _REF_MAX:
            raise ValueError(f'一次請選擇 1～{_REF_MAX} 張圖片。')
        for index, item in enumerate(items, 1):
            filename = Path(item['name']).name
            suffix = Path(filename).suffix.lower()
            if suffix not in ('.png','.jpg','.jpeg','.webp'):
                raise ValueError(f'不支援的圖片格式：{filename}')
            blob = bytes(item['content'])
            if len(blob) > 20 * 1024 * 1024:
                raise ValueError(f'單張圖片超過 20 MiB：{filename}')
            with Image.open(BytesIO(blob)) as photo:
                photo.verify()
            (stage / f'{index:03d}{suffix}').write_bytes(blob)
        _REF_ROOT.rename(backup)
        try:
            stage.rename(_REF_ROOT)
        except Exception:
            backup.rename(_REF_ROOT)
            raise
        shutil.rmtree(backup)
        current = sorted(_REF_ROOT.iterdir())
        _ref_sync(current)
        print('最新參考圖：', ', '.join(f'{p.name} = <Picture {i}>' for i,p in enumerate(current,1)))
    except Exception as error:
        shutil.rmtree(stage, ignore_errors=True)
        print('上傳未生效，保留上一批參考圖：', error)
    finally:
        _REF_PANEL.children = (_ref_new(),)

'''
Reset the browser picker so selecting the same filenames works again.
每次上傳後重建選檔元件，確保可以重新選相同檔名。
'''
def _ref_new():
    picker = widgets.FileUpload(accept='image/*', multiple=True, description='多選參考圖片')
    picker.observe(_ref_update, names='value')
    return picker

_REF_PANEL = widgets.VBox()
_REF_PANEL.children = (_ref_new(),)
_ref_sync(_REF_FILES)
display(_REF_PANEL)
print('目前圖片：', ', '.join(p.name for p in _REF_FILES) or '尚未上傳')
print('✓ 第 10 節完成：R2V 設定與圖片上傳介面已就緒（尚未生成）。')


11. 生成影片（沿用最後執行的模式 Cell，顯示即時步數進度）


In [ ]:
'''
Serialize notebook generation with optional remote FIFO GPU jobs.
讓 Notebook 手動生成與選用的遠端 FIFO 共用 GPU 鎖，避免同時推論。
不必先啟動 ngrok 或任何 HTTP API。
'''
import threading
GENERATION_LOCK = globals().setdefault('GENERATION_LOCK', threading.Lock())
with GENERATION_LOCK:
    import json
    import shutil
    import uuid
    import re
    import math
    import websocket
    from tqdm.auto import tqdm
    TASK = globals().get('TASK', 't2v')
    assert TASK in globals().get('TASK_CONFIGS', {}), f'請先執行 {TASK.upper()} 的參數設定 Cell。'
    config = TASK_CONFIGS[TASK]

    '''
    Keep one submission flow shared across all four tasks.
    四種生成模式共用同一份工作流程提交邏輯。
    '''
    MODE = TASK
    PROMPT = config['prompt']
    WIDTH, HEIGHT = config['width'], config['height']
    SECONDS, STEPS = config['seconds'], config['steps']
    SEED, OUTPUT_NAME = config['seed'], config['output_name']

    '''
    Resolve task-specific reference images without mixing modes.
    依據所選任務使用對應參考圖片，避免不同模式混用。
    '''
    FIRST_IMAGE, LAST_IMAGE = config['first_image'], config['last_image']
    AUDIO_SHIFT = config['audio_shift']
    QUANT = config['quant']
    assert QUANT in QUANT_SIZES_GIB, f'不支援的 QUANT：{QUANT}'
    REF_IMAGES = config.get('ref_images', [])

    '''
    Select optional reference videos and standalone audio.
    取得選用的參考影片與獨立音訊。
    '''
    REF_VIDEOS = list(config.get('ref_videos', []))
    REF_AUDIOS = list(config.get('ref_audios', []))
    VIDEO_AUDIO = bool(config.get('video_audio', False))
    REF_IMAGE_SIZE = config.get('ref_image_size', 'match')
    ACTIVE_TEXT_MODE = config.get('text_mode', TEXT_MODE)
    assert 0.01 <= AUDIO_SHIFT <= 100.0, 'AUDIO_SHIFT 必須介於 0.01 與 100.0 之間。'

    '''
    Ensure edited GPU 0 memory budgets have been applied to the running service.
    確認調整過的 GPU 0 記憶體預算已套用到目前服務。
    '''
    assert globals().get('H3_ACTIVE_CONFIG') == H3_DESIRED_CONFIG, 'GPU 0 預算已變更，請重新執行第 4 節啟動 ComfyUI。'
    EXPECTED_VARIANT = 'ref2va_pruned' if MODE == 'r2v' else 'fl2va_pruned'
    assert globals().get('H3_MODEL_CHOICE') == (EXPECTED_VARIANT, QUANT) and globals().get('H3_ACTIVE_MODEL') == (EXPECTED_VARIANT, QUANT), (
        f'目前選擇 {QUANT}，但尚未完成下載或服務預檢。請先執行第 2 節與第 4 節。'
    )
    assert DIT_NAME == f'minimax_h3_{EXPECTED_VARIANT}-{QUANT}.gguf', '模型檔名與任務種類／QUANT 不一致，請重跑第 2 節。'
    LAST_OUTPUT = None

    '''
    Confirm the existing local service is still running.
    確認先前啟動的本機推論服務仍正常運作。
    '''
    assert ready(), 'ComfyUI 已中斷，請先重新執行第 4 節。'
    assert MODE in ('t2v', 'i2v', 'l2v', 'flf2v', 'r2v') and PROMPT.strip() and STEPS > 0 and math.isfinite(SECONDS) and SECONDS > 0, (
        '請確認生成模式、Prompt、STEPS 與 SECONDS 設定正確；SECONDS 必須為正數。'
    )
    assert WIDTH > 0 and HEIGHT > 0 and WIDTH % 32 == 0 and HEIGHT % 32 == 0, (
        'WIDTH 與 HEIGHT 必須為正數且是 32 的倍數。'
    )
    FRAMES = max(5, 17 * math.ceil((SECONDS * 24 - 5) / 17) + 5)

    '''
    Respect the underlying ComfyUI node's frame limit, not an arbitrary notebook duration.
    遵守 ComfyUI 節點真正的影格數上限，不另外設定 Notebook 的秒數上限。
    '''
    assert FRAMES <= 3600, f'要求 {FRAMES} frames，超出 MiniMax H3 節點上限 3600；請縮短 SECONDS。'
    import secrets
    RUN_SEED = secrets.randbelow(2**32) if SEED == -1 else int(SEED)
    assert 0 <= RUN_SEED < 2**32, 'SEED 必須介於 0 至 4294967295，或設為 -1。'
    print(f'本次生成 seed：{RUN_SEED}')

    '''
    Build a unique ComfyUI output prefix from the user filename.
    根據指定檔名建立不重複的 ComfyUI 輸出前綴。
    '''
    output_stem = Path(OUTPUT_NAME).stem
    run_name = f'{output_stem}_{EXPECTED_VARIANT}_{QUANT}_{uuid.uuid4().hex[:8]}'
    graph = {
        '1': {'class_type': 'ClipProjLoader', 'inputs': {

            'clip_name': 'qwen3vl_4b_fp8_scaled.safetensors', 'type': 'auto',
            'projection': 'mmh3-4b-ClipProj-v3.1-mlp.safetensors',
            'device': TEXT_GPU, 'mode': ACTIVE_TEXT_MODE}},
        '2': {'class_type': 'UnetLoaderGGUF', 'inputs': {'unet_name': DIT_NAME}},
        '3': {'class_type': 'MiniMaxH3SigmaShift', 'inputs': {
            'model': ['2', 0], 'shift_video': 12.0, 'shift_audio': AUDIO_SHIFT}},
        '4': {'class_type': 'VAELoader', 'inputs': {'vae_name': 'minimax_h3_video_vae_fp16.safetensors'}},
        '5': {'class_type': 'VAELoader', 'inputs': {'vae_name': 'minimax_h3_audio_vae_fp32.safetensors'}},
        '6': {'class_type': 'MiniMaxH3ImageToVideo', 'inputs': {
            'clip': ['1', 0], 'vae': ['4', 0], 'prompt': PROMPT,
            'width': WIDTH, 'height': HEIGHT, 'length': FRAMES}},
        '7': {'class_type': 'BasicGuider', 'inputs': {
            'model': ['3', 0], 'conditioning': ['6', 0]}},
        '8': {'class_type': 'KSamplerSelect', 'inputs': {'sampler_name': 'res_multistep'}},
        '9': {'class_type': 'BasicScheduler', 'inputs': {
            'model': ['3', 0], 'scheduler': 'simple', 'steps': STEPS, 'denoise': 1.0}},
        '10': {'class_type': 'RandomNoise', 'inputs': {'noise_seed': RUN_SEED}},
        '11': {'class_type': 'SamplerCustomAdvanced', 'inputs': {
            'noise': ['10', 0], 'guider': ['7', 0],
            'sampler': ['8', 0], 'sigmas': ['9', 0], 'latent_image': ['6', 1]}},
        '12': {'class_type': 'VAEDecode', 'inputs': {'samples': ['11', 0], 'vae': ['4', 0]}},
        '13': {'class_type': 'VAEDecodeAudio', 'inputs': {'samples': ['11', 0], 'vae': ['5', 0]}},
        '14': {'class_type': 'CreateVideo', 'inputs': {
            'images': ['12', 0], 'audio': ['13', 0], 'fps': 24.0}},
        '15': {'class_type': 'SaveVideo', 'inputs': {
            'video': ['14', 0], 'filename_prefix': run_name,
            'format': 'auto'}},
    }
    if MODE == 'r2v':
        assert isinstance(REF_IMAGES, (tuple, list)) and 0 <= len(REF_IMAGES) <= 9, 'R2V 圖片最多 9 張，也可以只提供影片或音訊。'
        assert REF_IMAGE_SIZE in ('match', 'max'), 'R2V_REF_IMAGE_SIZE 只能是 match 或 max。'
        picture_numbers = {int(number) for number in re.findall(r'<Picture\s+(\d+)>', PROMPT, flags=re.IGNORECASE)}

        '''
        Refuse to submit a prompt referencing an absent image.
        若 Prompt 指向不存在的圖片，避免送出無效的參考條件。
        '''
        assert all(1 <= number <= len(REF_IMAGES) for number in picture_numbers), (
            f'R2V_PROMPT 引用了不存在的 <Picture N>：{sorted(picture_numbers)}；目前只有 {len(REF_IMAGES)} 張。'
        )
        check_choice('MiniMaxH3ReferenceToVideo', 'ref_image_size', REF_IMAGE_SIZE)
        graph['6'] = {'class_type': 'MiniMaxH3ReferenceToVideo', 'inputs': {
            'clip': ['1', 0], 'vae': ['4', 0], 'audio_vae': ['5', 0],
            'prompt': PROMPT, 'width': WIDTH, 'height': HEIGHT, 'length': FRAMES,
            'ref_image_size': REF_IMAGE_SIZE}}
        for image_index, image_path in enumerate(REF_IMAGES):
            source = Path(image_path)
            assert source.is_file(), f'R2V 找不到第 {image_index + 1} 張參考圖片：{source}'
            assert source.suffix.lower() in ('.png', '.jpg', '.jpeg', '.webp', '.bmp'), f'不支援的參考圖片格式：{source.suffix}'
            node_id = str(16 + image_index)

            '''
            Copy files into ComfyUI's private input directory with unique names.
            將參考圖片複製到 ComfyUI 私有輸入資料夾，並使用不重複檔名。
            '''
            destination = COMFY / 'input' / f'r2v_{image_index + 1}_{uuid.uuid4().hex[:8]}{source.suffix.lower()}'
            destination.parent.mkdir(parents=True, exist_ok=True)
            shutil.copy2(source, destination)
            graph[node_id] = {'class_type': 'LoadImage', 'inputs': {'image': destination.name}}
            graph['6']['inputs'][f'ref_images.ref_image_{image_index}'] = [node_id, 0]

    '''
    Validate reference media counts and numbered prompt placeholders.
    驗證圖片、影片、音訊張數與 Prompt 的參考編號。
    '''
    if MODE == 'r2v':
        assert len(REF_IMAGES) <= 9 and len(REF_VIDEOS) <= 3 and len(REF_AUDIOS) <= 3, 'R2V 最多 9 張圖片、3 段影片、3 段獨立音訊。'
        assert REF_IMAGES or REF_VIDEOS, 'R2V 至少需要一張參考圖片或一段參考影片；音訊可作為附加條件。'
        assert len(REF_IMAGES) + len(REF_VIDEOS) + len(REF_AUDIOS) <= 12, 'R2V 參考素材合計最多 12 個檔案。'
        tags = {'Picture': len(REF_IMAGES), 'Video': len(REF_VIDEOS),
                'Audio': len(REF_AUDIOS) + (len(REF_VIDEOS) if VIDEO_AUDIO else 0)}
        for label, count in tags.items():
            numbers = [int(n) for n in re.findall(r'<%s\s+(\d+)>' % label, PROMPT, re.I)]
            assert all(1 <= n <= count for n in numbers), f'<{label} N> 編號超出實際上傳數量：{numbers}。'

        '''
        Read reference videos using native ComfyUI video nodes, without extra pip packages.
        使用 ComfyUI 原生影片節點載入參考影格與選用的影片音軌。
        '''
        for index, filename in enumerate(REF_VIDEOS):
            original = Path(filename)
            assert original.is_file() and original.suffix.lower() in ('.mp4', '.mov', '.webm'), f'參考影片不存在或格式不正確：{original}'
            destination = COMFY / 'input' / f'h3_ref_video_{uuid.uuid4().hex[:12]}{original.suffix.lower()}'
            destination.parent.mkdir(parents=True, exist_ok=True)
            shutil.copy2(original, destination)
            node_id = str(120 + index * 2)
            parts_id = str(121 + index * 2)
            graph[node_id] = {'class_type': 'LoadVideo', 'inputs': {'file': destination.name}}
            graph[parts_id] = {'class_type': 'GetVideoComponents', 'inputs': {'video': [node_id, 0]}}
            graph['6']['inputs'][f'ref_videos.ref_video_{index}'] = [parts_id, 0]
            if VIDEO_AUDIO:
                graph['6']['inputs'][f'ref_video_audios.ref_video_audio_{index}'] = [parts_id, 1]
        for index, filename in enumerate(REF_AUDIOS):
            original = Path(filename)
            assert original.is_file() and original.suffix.lower() in ('.wav', '.mp3', '.flac', '.ogg', '.m4a'), f'參考音訊不存在或格式不正確：{original}'
            destination = COMFY / 'input' / f'h3_ref_audio_{uuid.uuid4().hex[:12]}{original.suffix.lower()}'
            destination.parent.mkdir(parents=True, exist_ok=True)
            shutil.copy2(original, destination)
            node_id = str(140 + index)
            graph[node_id] = {'class_type': 'LoadAudio', 'inputs': {'audio': destination.name}}
            graph['6']['inputs'][f'ref_audios.ref_audio_{index}'] = [node_id, 0]
    for mode, image_path, node_id, input_name in [
        ('i2v', FIRST_IMAGE, '16', 'first_frame'),
        ('flf2v', FIRST_IMAGE, '16', 'first_frame'),
        ('flf2v', LAST_IMAGE, '17', 'last_frame'),
        ('l2v', LAST_IMAGE, '18', 'last_frame'),
    ]:
        if MODE == mode:
            '''
            Require an existing image file for the selected mode.
            確認所選模式的圖片檔案確實存在。
            '''
            source = Path(image_path)
            assert source.is_file(), f'找不到參考圖片：{source}'
            destination = COMFY / 'input' / f'{node_id}_{uuid.uuid4().hex[:8]}{source.suffix.lower()}'
            destination.parent.mkdir(parents=True, exist_ok=True)
            shutil.copy2(source, destination)

            '''
            Register the image loader in the API graph.
            將圖片載入節點加入 API Workflow。
            '''
            graph[node_id] = {'class_type': 'LoadImage', 'inputs': {'image': destination.name}}
            graph['6']['inputs'][input_name] = [node_id, 0]
    NODE_INFO = requests.get(f'{BASE}/object_info', timeout=90).json()
    check_choice('ClipProjLoader', 'device', TEXT_GPU)
    check_choice('ClipProjLoader', 'mode', ACTIVE_TEXT_MODE)

    '''
    Verify the H3 sampler against the specialized node or the core KSampler options.
    優先用目前採樣器選單驗證；必要時參照核心 KSampler 選項。
    '''
    check_choice('KSamplerSelect', 'sampler_name', 'res_multistep', ('KSampler', 'sampler_name'))
    check_choice('BasicScheduler', 'scheduler', 'simple', ('KSampler', 'scheduler'))
    if MODE in ('i2v', 'l2v', 'flf2v'):
        frame_inputs = NODE_INFO['MiniMaxH3ImageToVideo'].get('input', {}).get('optional', {})
        expected = ('first_frame', 'last_frame') if MODE == 'flf2v' else (('last_frame',) if MODE == 'l2v' else ('first_frame',))

        '''
        Reject an incompatible core node implementation before generation.
        如果核心節點不支援這些欄位，在生成之前先停止。
        '''
        assert all(field in frame_inputs for field in expected), f'MiniMaxH3ImageToVideo 缺少 {expected} 輸入。'
    if MODE == 'r2v':
        ref_inputs = NODE_INFO['MiniMaxH3ReferenceToVideo'].get('input', {})
        ref_fields = set(ref_inputs.get('required', {})) | set(ref_inputs.get('optional', {}))
        assert 'ref_images' in ref_fields or any(name.startswith('ref_images.ref_image_') for name in ref_fields), (
            'MiniMaxH3ReferenceToVideo 缺少 ref_images 輸入，請更新 ComfyUI。'
        )

    '''
    Confirm the installed ComfyUI version supports all reference-media loader nodes.
    確認目前 ComfyUI 具備參考影片與音訊所需的原生節點。
    '''
    if MODE == 'r2v':
        required_nodes = {'LoadVideo', 'GetVideoComponents'} if REF_VIDEOS else set()
        if REF_AUDIOS:
            required_nodes.add('LoadAudio')
        missing_nodes = required_nodes - set(NODE_INFO)
        assert not missing_nodes, f'ComfyUI 缺少參考媒體節點：{sorted(missing_nodes)}，請更新 ComfyUI。'
    for node_id, node in graph.items():
        '''
        Examine each declared input for possible node-reference links.
        檢查節點輸入中是否包含連到其它節點的參照。
        '''
        for value in node['inputs'].values():
            if isinstance(value, list) and len(value) == 2:
                assert str(value[0]) in graph, f'節點 {node_id} 連結到不存在的節點 {value[0]}。'
    for node_id, node in graph.items():
        schema = NODE_INFO[node['class_type']].get('input', {})
        accepted = set().union(*(set(schema.get(group, {})) for group in ('required', 'optional', 'hidden')))
        unknown = set(node['inputs']) - accepted

        '''
        Allow dynamic Ref2VA image slots declared by the live autogrow parent.
        當節點宣告 ref_images 動態欄位時，允許自動展開的圖片插槽。
        '''
        if node['class_type'] == 'MiniMaxH3ReferenceToVideo' and 'ref_images' in accepted:
            unknown -= {f'ref_images.ref_image_{i}' for i in range(9)}
            if 'ref_videos' in accepted:
                unknown -= {f'ref_videos.ref_video_{i}' for i in range(3)}
            if 'ref_video_audios' in accepted:
                unknown -= {f'ref_video_audios.ref_video_audio_{i}' for i in range(3)}
            if 'ref_audios' in accepted:
                unknown -= {f'ref_audios.ref_audio_{i}' for i in range(3)}
        assert not unknown, f"節點 {node_id} ({node['class_type']}) 不支援的輸入：{sorted(unknown)}"
    print(f'{MODE.upper()} | {WIDTH}×{HEIGHT} | {FRAMES} frames | {STEPS} steps | {EXPECTED_VARIANT}/{QUANT} | Audio Shift {AUDIO_SHIFT}', flush=True)
    client_id = uuid.uuid4().hex

    '''
    Track WebSocket availability separately from workflow success.
    個別追蹤 WebSocket 可用性，不影響實際生成流程。
    '''
    ws = None
    try:
        ws = websocket.create_connection(f'ws://127.0.0.1:{PORT}/ws?clientId={client_id}', timeout=5)
        ws.settimeout(2)

    except (OSError, websocket.WebSocketException):
        '''
        Mark event updates as unavailable and retain an elapsed-time display.
        停用事件更新，改成單一進度列顯示經過時間。
        '''
        ws = None
        print('即時步數事件不可用；將顯示階段與經過時間。', flush=True)
    response = requests.post(f'{BASE}/prompt', json={'prompt': graph, 'client_id': client_id}, timeout=120)
    if not response.ok:
        if ws is not None:
            ws.close()
        raise RuntimeError(response.text[:5000])
    prompt_id = response.json()['prompt_id']

    '''
    Start timing the actual queued job.
    開始計算已提交生成工作的耗時。
    '''
    start = time.monotonic()
    pbar = tqdm(total=STEPS, desc='排隊／載入模型', unit='step', leave=True)
    item = None
    last_history_check = 0.0
    phases = {'1': '文字編碼器', '2': '載入 GGUF', '6': '建立影片條件',
              '11': '採樣中', '12': '影片解碼', '13': '音訊解碼',
              '14': '合成影片', '15': '儲存影片'}

    '''
    Wait until ComfyUI reports a final history record for this task.
    持續等待 ComfyUI 回傳此任務的最終執行紀錄。
    '''
    try:
        '''
        Keep processing until success, an explicit error, or timeout.
        持續處理直到成功、明確錯誤或逾時。
        '''
        while True:
            '''
            Measure real wall-clock time instead of estimating model progress.
            計算真實經過時間，不將時間誤當模型百分比。
            '''
            elapsed = time.monotonic() - start
            if elapsed > 7200:
                raise TimeoutError(f'生成逾時；prompt_id={prompt_id}，可查詢 {BASE}/history/{prompt_id}')
            message = None
            if ws is not None:
                try:
                    message = ws.recv()

                except websocket.WebSocketTimeoutException:
                    pass

                except (OSError, websocket.WebSocketException):
                    '''
                    Release the unusable client connection.
                    關閉失效的 WebSocket 連線。
                    '''
                    ws.close()
                    ws = None

            '''
            Only JSON text frames contain progress data; preview images are binary.
            只有 JSON 文字事件含進度資料；預覽圖片為二進位資料。
            '''
            if isinstance(message, str):
                event = json.loads(message)

                '''
                Read the event payload without assuming optional keys exist.
                讀取事件資料，並避免假設選填欄位一定存在。
                '''
                data = event.get('data') or {}
                if data.get('prompt_id') == prompt_id:
                    if event.get('type') == 'executing' and data.get('node'):
                        pbar.set_description(phases.get(str(data['node']), '執行中'), refresh=False)
                    if event.get('type') in ('progress', 'progress_state'):
                        progress = data
                        if event.get('type') == 'progress_state':
                            progress = data.get('nodes', {}).get('11', {})

                        '''
                        Ignore progress emitted by loading or decoding nodes.
                        忽略非採樣節點的進度，以免誤顯示百分比。
                        '''
                        if str(progress.get('node', progress.get('node_id'))) == '11':
                            value, maximum = progress.get('value'), progress.get('max')
                            if isinstance(value, (int, float)) and isinstance(maximum, (int, float)) and maximum > 0:
                                pbar.total = int(maximum)
                                pbar.n = max(0, min(int(value), pbar.total))
                                pbar.set_description('採樣中', refresh=False)
            if elapsed - last_history_check >= 4:
                last_history_check = elapsed
                if not ready():
                    '''
                    Preserve recent server logs for actionable diagnostics.
                    保留最近服務紀錄供後續除錯。
                    '''
                    raise RuntimeError('ComfyUI 服務中斷：\n' + LOG.read_text(errors='replace')[-4500:])
                item = requests.get(f'{BASE}/history/{prompt_id}', timeout=60).json().get(prompt_id)

                '''
                Stop polling once the job has a final record.
                任務產生最終紀錄後停止輪詢。
                '''
                if item:
                    errors = [m for m in item.get('status', {}).get('messages', []) if m[0] == 'execution_error']
                    if errors:
                        raise RuntimeError(str(errors[-1][1])[:5000])
                    assert item.get('status', {}).get('status_str') != 'error', (
                        f'ComfyUI 工作失敗：{item.get("status", {})}'
                    )
                    break
            pbar.set_postfix_str(f'{elapsed / 60:.1f} 分鐘', refresh=False)
            pbar.refresh()

            '''
            Avoid tight polling if the WebSocket connection is unavailable.
            WebSocket 不可用時稍作等待，避免過密輪詢。
            '''
            if ws is None:
                time.sleep(1)
        pbar.n = pbar.total
        pbar.set_description('完成', refresh=False)
        pbar.set_postfix_str(f'{(time.monotonic() - start) / 60:.1f} 分鐘', refresh=False)
        pbar.refresh()

    finally:
        '''
        Always free the live progress widgets and connection after this job.
        無論任務成功或失敗，都釋放進度列與 WebSocket 連線。
        '''
        pbar.close()
        if ws is not None:
            ws.close()
    files = [p for p in (COMFY / 'output').rglob('*') if p.is_file() and p.name.startswith(run_name)]
    files = [p for p in files if p.suffix.lower() in ('.mp4', '.webm', '.mov')]
    assert files, f'工作流程完成但未找到影片：{item.get("outputs", {})}'
    source_video = max(files, key=lambda path: path.stat().st_mtime)

    '''
    Save the user-visible output in Kaggle working storage.
    將成果儲存至 Kaggle working 目錄。
    '''
    OUTPUT_DIR = ROOT / 'minimax_h3_outputs'
    OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
    output_base = OUTPUT_DIR / f'{Path(OUTPUT_NAME).stem}_{EXPECTED_VARIANT}_{QUANT}{source_video.suffix}'
    output = output_base

    '''
    Number duplicate filenames deterministically.
    依序為重複檔名編號。
    '''
    for index in range(2, 1000):
        if not output.exists():
            break
        output = output_base.with_name(f'{output_base.stem}_{index:03d}{output_base.suffix}')
    shutil.copy2(source_video, output)
    print(f'完成：{output}｜耗時 {(time.monotonic() - start) / 60:.1f} 分鐘')
    LAST_OUTPUT = output
    print('✓ 第 11 節完成：影片已生成；請執行第 12 節預覽與下載。')


12. 影片預覽、下載與 GPU 使用量


In [ ]:
'''
Display the most recently generated video and its download link.
顯示最近一次成功生成影片的預覽與下載連結。
'''
from IPython.display import Video, FileLink, display
assert globals().get('LAST_OUTPUT') is not None and LAST_OUTPUT.is_file(), '請先執行第 9 節生成影片。'
display(Video(str(LAST_OUTPUT), embed=True, width=480))

'''
Provide a clickable Kaggle file download link.
顯示 Kaggle 影片檔案下載連結。
'''
display(FileLink(str(LAST_OUTPUT)))
status = subprocess.run(
    ['nvidia-smi', '--query-gpu=index,name,memory.used,memory.total',
     '--format=csv,noheader,nounits'], capture_output=True, text=True, check=True
)
print('GPU 記憶體使用量（MiB）：')

'''
Display both devices in a readable form.
逐行顯示兩張 GPU 的使用狀態。
'''
print(status.stdout.strip())
for line in status.stdout.splitlines():
    fields = [part.strip() for part in line.split(',')]
    if len(fields) == 4 and fields[0] in ('0', '1') and fields[2].isdigit():
        used_gib = int(fields[2]) / 1024
        limit_gib = GPU0_MODEL_GIB if fields[0] == '0' else GPU1_MODEL_GIB
        if used_gib > limit_gib:
            print(f'提醒：GPU {fields[0]} 使用 {used_gib:.2f} GiB，超過目標／監控值 {limit_gib:.2f} GiB；非硬上限。')

print(f'✓ 第 12 節完成：影片預覽與下載連結已顯示；{LAST_OUTPUT.name}')


13. 擷取最後影格（選用）


In [ ]:
'''
Count decoded video frames to identify the final frame.
計算影片影格數，找出最後一個影格。
'''
import json
from IPython.display import FileLink, display
from PIL import Image
assert globals().get('LAST_OUTPUT') is not None and LAST_OUTPUT.is_file(), '請先執行第 9 節生成影片，並確認第 10 節可正常預覽。'
assert shutil.which('ffprobe') and shutil.which('ffmpeg'), '找不到 ffprobe 或 ffmpeg；請先安裝 FFmpeg。'

'''
Build a PNG path next to the generated video.
在生成影片旁建立最後影格 PNG 檔案路徑。
'''
LAST_FRAME_PATH = LAST_OUTPUT.with_name(LAST_OUTPUT.stem + '_last_frame.png')
probe = subprocess.run(
    ['ffprobe', '-v', 'error', '-count_frames', '-select_streams', 'v:0',
     '-show_entries', 'stream=nb_read_frames', '-of', 'json', str(LAST_OUTPUT)],
    capture_output=True, text=True, check=True,
)
last_index = int(json.loads(probe.stdout)['streams'][0]['nb_read_frames']) - 1

'''
Save the exact last frame as a PNG for the next I2V clip.
將最後一個影格儲存成 PNG，供下一段 I2V 使用。
'''
subprocess.run(
    ['ffmpeg', '-y', '-loglevel', 'error', '-i', str(LAST_OUTPUT),
     '-vf', rf'select=eq(n\,{last_index})', '-frames:v', '1', str(LAST_FRAME_PATH)],
    check=True,
)
display(Image.open(LAST_FRAME_PATH))
display(FileLink(str(LAST_FRAME_PATH)))
print(f"下一段：TASK = 'i2v'，FIRST_IMAGE = '{LAST_FRAME_PATH}'")
print(f'✓ 第 13 節完成：已擷取最後影格；{LAST_FRAME_PATH.name}')


14. （選用）啟動 ngrok 與 FIFO 遠端 API


In [ ]:
'''
Require the first configuration cell before enabling the optional remote API.
選用遠端 API 啟動前，必須先執行第一格共用參數設定。
'''
assert all(name in globals() for name in ('REMOTE_PORT', 'REMOTE_SERVICE_ID', 'REMOTE_ROOT')), '請先執行第 1 節共用參數設定。'
import zlib
import math
import re

import queue
import copy
import secrets
import io
import json
import os
import shutil
import subprocess
import tarfile
import tempfile
import threading
import time
import urllib.request
import uuid
import zipfile
from email import policy
from email.parser import BytesParser
from getpass import getpass
from http.server import BaseHTTPRequestHandler, ThreadingHTTPServer
from pathlib import Path
from urllib.parse import urlsplit, parse_qs

REMOTE_KIND = 'video'
REMOTE_FIELDS = ['task', 'prompt', 'seed', 'width', 'height', 'seconds', 'steps', 'ref_image_size', 'audio_shift', 'quant', 'video_audio']
REMOTE_IS_LTX = False
REMOTE_SCRIPT = zlib.decompress(b'x\xda\xd5}ks\x13W\xb6\xe8w~E\xdf\xa9\x9a\x924\xb1\x15\xdb\x90d\xc2\x1d\xe7\x16\x03\x86\xf8N\x02\x1c\x0c\x99s.C\xa9\xdaR\xcb\xeeA\xafQK6\x1e\x1fW\xd9\x04c\x831v\x12pb0\xcf\x00\xf1@\xb0!\xbc\x8c1\xe6\xc3\xf9\'\xf7\xa8\xf5\xf8\x94\xbfp\xd7c\xef\xdd\xbb[-c\x92\xb9\xe7\xd6\x9d\xa9"V\xab\xd5\xbd\xf6\xda\xeb\xfd\xda\x91HdGo\xb6\x90/\x96\x8c\xd2\xa0ed\xec\xfe\xa2Y\xb4-\xc7\xc8YV\xcaJ\x19\xe9|\x11\xbe\xb0\x1d\xc3\xb1\x92%;\x9f\x8b\xefp/\xac\xba\x93w\xabK?VW\x9e\xd5\x96g\xdc\x8d\xb9\xea\xb9\xf1\xc6\xd2x\xed\xca\x19w\xea5|t\xd7\x1f\xfc\xe7\xf8\xe9\x1d\x11x\xb2\xcdO\xfe\xab\x93\xcf\xc9\xbf\x9d\xc1r\xc9\xce\xc8O\xe5\xb2\x9d\x92\x7f\x17-\xf9W\xd6,\r\xca\xbf\x87\xad~\'\x9f<i\x95v\xa4\x8b\xf9\xacQ\xfa[*\x1b7\xcb\xa5\xbc!\xbe\xc7\x0b;\xe8e}V\x06`\xa4e\xa4,\xc7.\x02\xf4%\xd39i\x0c\xdb\xa5\xc1|\xb9d\xe4\x87\xac\xe2p\xd1.\xd9\xb9\x01\xc3\xce\xa5\xac\x82\x05\xff\xe4JF\x01\x9e[(9qc\xef`>\xefX\xf4\x80\x01+g\x15M\\0?\xc3\xce\x19}\x8c\x00\xa3+\xbe\xa31\xb1V\xfdf\nQ\xf0\xe3\xad\xda\xa5\x1b\xd5\xe9yX|\xe5\xd5+wf\xf1\xe7\x8d\x0b\x95\xb5\xd9\xfa\xbd\xb3\xee\xea\x03w\xf2\xb9\xbb\xf2%_\x87\x1b\x8c\xc3\xf4"\xc0\x8d\xe1.-\xd7~\xfc\xd1\xe82j\xab\xe3\xfc\xb0\x9f7\xaeDJ]C\x916#b\xf3\x7f\xd2\x994\xffQ\x84\xff\xd0\x02\x8f\xee\xe9\xfb\x93\xd1m\x0cd\xf2\xfdf\xc6\x89\xc6\xe2\x03V)\x1a\xc1\xabx\x1b\xfe:\xc6\x988b\x99)Z\x86C(\x11\x88\x888\xb0Tk\xc8\xce\x97\x9d\xcc\x88a\x9d\xb2\x92e\xfc*\x99\xcf\xa5\xed\x81\xb2Xm\xd2\xcad\xe2;\xea+\xe3\xee\xdc\x02\xee\xeb\xc4\x1a\xc3\xef\xbe\xf8\xc9\xbd\xf1\xa2~\xebBc\xe2"\xac\xa5\xbe\xfc\xd0]\xb9b\xec\x85\xbb\xe5^\x9b\x8ec\xc1~\x10\x90\x80\xae\x10(\x13{\x0f\x1d\xdc\xdf{\xa0\x0f\xa0\x1d\x1d\x8b\xb5\x19\xe9H\xfd\xc1\x8c;9\xcd\x0f6F\xf1\x9ex\xb9P\xb0\x8a\xd1\xd8\x98\x81\xf44\xf7e\xf5\xf2\x9a\xff]\x11\xb9D\'\x9f\x19\xb2BW\xa9o\xaec\x95p\xc3\x1d\xa0\xdb\xb9\x05w\xf3[}M\xf0\x86\xda\xecr\r`\xa0\xf7\xc8\x850B\x00\xd1:\xcc\xc7\xf1\xc3\t~\xf7\x9f,\xab`\xe4s\xf0\xe2r\x7f\xd6v\x1c\xc4[:\x93\x1f\x06\xda6\x91\xea\xccd1\xef8\x86\x99\xc9\x00\xfb\x94\x8b\x04\x16\x02p\xf5jmy\x85\xe9\xa5\xba|\x0b9e\xf2q\xed\xd2\xb2;\x0f$3^y\xf5\xda}q\xb7\xf2z\xa9\xfal\x02\xd8\xaa:7_Y\xbf\xd38=W\xdfX\x95\x80}~h_\x8f\x00\x8b\x019&\xa8\x95)\xd8pJy|= \xbf\t+b_+\xaf\xdf\xc0\x0bkWW\xdcs\xb3\x8c\x03\r\xb9F\xfd\xfb[>2\xa5w\x1c>r\xe8\xf3\xc3G\xe1\xb5\x8c\x96\xe3\x11~Y\xe4\x84\x1f\x04\xf5\xb6\xa4\x99\x1b2\x1d\x1d\xefw\xbf\x85w\x06\xf0^\xdfXs\xa7\xd6\xddG\xeb\xee\xaa\xc2\xfb\x9f{\xf7\x1d\xfd\xb4\xcd\xf8\xb4\xa7\xf7\xc0\xa7\xfa\x0b\x87\xedTi0r\xa2M]\x18\xb4\xec\x81\xc1&\x08p\x99\xedN\xc1J\xdai;i\xa4$A\x9b\xb9\x94\xe1\x98\xd9B\x06\xd9>\x99/\xe7J-@\xaa\xfd\xf05\x12\xdb\xf4T\xf5\xe2\xad\xea\xf2\xf7\xd5\x87w5\x9a\xe8\xeb\x01:\xd8\xd7\xd7f\xf4\x1d\xed9\xdc\xa7\x01\x07b1\x9fK9:xN\xc9*8\x12\xba=\x85\x020[3|\x8e\x85\x84\x02\xb0\x81d*\x80pJ\xdb\x19+gf\xadV\xc0-\xaf\xb8\x0f\xe7\x018F\\\xf5\xfe%w~\xd6\x03\xaeg_\x9bq\xe8\xd8\xd1\xc3\xc7\x8e&\x0e\xee\xf9\xbc\xc7\x07\x9f\x95\xd2\x81\xe3\xd7%\xf0U\x12D\xc5J>\x00\x8bV\xda*Z\xb9\xa4\x05\x82\xd6\x1c\x00\x85 \xc5h\xd6>\x85\xa8\xcc\xe6A\xc8\x02Em^\xab~sE\x87\x96i\xcc}4[\x9d:\x07\x8cU\x1f\xff\xd2]Z\xa8\x9d\x03\t\x07\xb2\xe3\x8d;9\x0b\xf2\x11H\x9e9\xa0\xfa\xe2\x05\xdc,\x17\xb2\xbf\xf7H\xdf\xd1D\xef\xe7{\x0e\xf4\xb4\x19\x9f\xed\x91\x7fk\xcbI\xdbE\xa7\x94 \x88\xf4UeL\xefj\x10\xefA\xe9`\x96Sv\xdep\xec\x81\xac\t\x0ck\xa7KjeV\x8a\x95C@\x01\x0c\x14\xcd\xc2\xa0\xe4\x9d\xea\xc3;jS\x1a7\x9e\xd4\x97\xcf\x1b}\xf4\xa8>|\x14K\x7f\xf7\xcdd\xe5\xcdJ\xf5\xd2K\xe6u\xe3\xcf\xf9\xe2I\x94\x0fr\x99{\x8e\xed\xeb=\x94\xe8\xfb\xb4w\xbfN\xe6\x04W\x82 z\xfb"\x0e\x1c8\xb6\xdf\xf8[\xd9\xcc\x95\xec\xbf3\x94\xa8\xa8qO2\xf8@\xc7\x06"\xcc%G\x8c\xe4\xa0\x95D\xd9\x13\x00\xbeza\n\xb8\x1e\x99\x9d\x1e\xd4\x98\x9as/,Ton\xc0\x8e\xe0\x12^\xdd\xad\xdd^\xaf\xbc\xb9\x86{t}\xa6z\xffv\xf5\xc6]\x10Q\xf5\xa9\xa7r\t\xffrl\xcfA\x1dx\x02\xc5#\xa8\xbf\xa2".\xe7\x1c\x10\xe6\xa0\xa3\x01l\x1f\xa8H|\x8e\xd1o\x01\xc8\x96a\x96J\x16\xc8\x13\xc4\xbb\x87\xf3\xf8\x0eF\x1d\x88)\xd0\x0f\xd5K\x97\xaa\xd3\xf7\x01\xb1\xd5K\xab\xd5\xb9\xa7\x88y\r``\x84\xda\xf2\xe3\x80\x16b\xf8@\x12\xd2\x1f\x89\xbe\xde\xff\xd5\xd3\x978\xd0\xfbGT:\xfa\x83\xf8{P\xbe\xa3\xf4\xc7\x98\xd0.\x9f\xe5A\x81*\x0eh\xcf\xe7`\x1b\nf\x11\xe0.YE\x8f\x15\xcct\x1a\x8d\x02\x00=\x8d\xdf\xb5\xf7\x9b\x8e\xd8"G\xeaP\xe3H\xd7\x17\x86\xfbh\x1aY\x82\xb4\x8c \x91\xd7\x8f\x1b7\xde\xb8\x17o\x80\x11\x81\xeb\xb9\xb7\xe0>\xda\x84\x8b\xb0$f\x0c\xb9\x9e#=\xfb\x99\t<\xa1\xc3\n\x15\xa0c\x8aw@\x9d\x1e?\x11\xf3\xd9@\xf9\x02b\xd1\xcch\\<d\xa7\xac\xbc\xc3\xe2\xb0\x04\xff\x9a\x19\xd4`DtR7\x02\x0b\xa3v \xbd\x8bl\xfb\xfa1\xb0-\xc8\x1cV\x92L\xed:\\_\xf4\xee\xeb9\x84pe\x80\xdc\xa2A\xe0\xf8\x85\x0c\\\x8c\xee\'\xbaoy?\x81\xa2\xee\xa7g\xf3/\xe0\x07\xfd\xf9|\xc6\xf7\x03z8\xff\x04~\xb1\x1f\x0c\r+\x164~\xbc\xb5;\xf6\xdf\xe1\x9f\x12\xd0\x9650b\xd0n\xb2]k\x81\x99;L[T\x00\x93\x13\xf1p\xbf\xbat\x8e.\x081\xa6\x04\xd8\x1a0\xf4f\xed\xe1B\xed\xf2\xdd\xa6\xbd!\xfaj\xb5?\t|;\x1ah`\xd5&\x07#\xb1&c\x15@\x04\xbe-\x02\xbc\x0e\xac\n\xb9V\x81*\xa1L\x96\x8bE\xb4f\x90\xb4\xa4\xf9\xc9Z\x9c\xa9\x05)\xb9G<\xa6\xbeq\x0e\xecs?\xa4{\xf6\x1e\xed\xfd\xa2\'q\xb4\xe7_\x8f&\x84\x15\xa1\xc3Z\xb2N\x95\x12(<\x00Lu\x8f\x00\xf4s\x84\x9aQU\xceZE\xd4\x0cfn\x00M\xecd\x86\xac\x9d~\x96Pf\xb2T\x06\x82\xfb\xdc\xce\xd9\x9f\x9b\xa7>\xddIR\x91\x84\xa2\x91\x83\'\xc7wTW/\xd5&\x7f\x08\xbd\x01\xad\xe0W\xd7\xdd\x95\xef\xdd\xaf\xce\xc3R\x80K\xdc\xf1\x8d\xda\xea\x19wi\xb61\xfe\x13\xd8\x86,\x82\x02<\xde\x11\xef\xe84\xfe\xd0m\xe8\xe2\x14>vv\xc0\x17\x80n\xfd2H\xe4\xc6\xcd\xe9\xca\xab\x99\xea\xc2\x06\xff\x0e\xc8\x9a\xef4*/g\x1a\x0b\xdfx\x96e\x0f\x88-\x90K\xa8\x0c`u\x07\x0e\x1f3:\x8c\xac\x95\xcd\x17G\x8c\xfer\n\xf0\xe5\x18\x83&\xa8\xcb~\xcb\x02\xeb\x02$\xb4\x8d<\x9fg\x92+\xe7r(\x0e\x00\xbe!;\t\x8b\x069Z\xbf?[\xbf\xff\xa6z\xf9)[\xce\xe2\x89\xf5\xe5\xef\xaag\x9f7\x1e\\j\xdc\x9c\xaf\xad|\x8b\xe65\xe9}w\xfa\x91\xd8\xd8\xa5Y\x10\xd4\x81%\x07\x8c\xeaOw&\xc4\xce\xb2\x95\x1a\x89\x19\xdd\xdd\x06\\\xdd\xd7\xd3\xd7{\xa4g\x9f\xb8\x0c\xc8\xe0\x97\xaaw\xd5W\xceW\xaf>\x05Q\x04\x06xcj\xb6\xba\xf0\x88mptJv\xe1v\xb8\x97o\xb83\x97\x8d\xbd\xf9lz\xe4X\xaf\x87\x9c\xc3\xe0@ !j\xea\x11,*;\xc3T:\x08\xeeS\xce/\xeb\x87\xc1\x16L\xe5\x87s\x19\x90\xa8\xc2\xe2)Z\x03\xa8\x9e\x80v\xe2F\xcf\xa9\x02\xb2\x81i\x1c\xb1\xd2]_\xec\x01\xaa\xca\xe5mX*3\xe9\xf0\xa0\x95\xf3\xb3r;\xb1\x94\xf0\xc9\x1c\xa5\x16\x81s\xc1n\x9b\xbb\xea\xd3\x0b\x9b\x17`\x81\xbc\xf3\xe8d\xac\\\x00\x85RY\x9b\x01\xfe\xc0\xcd\x17K3X\xc7\xd5\x97_\xb9g\xcf\x93kF"\x00XL\n\xef\xea\xe2\xe9*X\xcb$\x0e$\x98\xee\xc5W\xee\xe2}\xfe\xa9\xdc\xa2\x9e\x7f=\xdc\xb3\xf7( \xfd\x8b=GzY;\xa2\x10\xe8\x1a2\x13\x05\xa0\x0b0\xc3\x0c;m0\xf7u\xb3WgX \xb7\xd0\xd3\xd3n\x92B\xecoe\xf0`A\xf0\x95\x98\xf9X\xb5\xa7\xcd\xac\rxA,\xe2E\x1f\xa6\x81\x04\xb3\x8aWYP\xa0\x99\xb5x\x9ai\x90\x81\x05;\xb2q\xeb\xba;w>\xa0\xf2\xd92k\\X\xad\xdf\xbfS\xfd\x89\x96\xba\xff3\\i\xf5G\xb8u\xf6\xadd\x88\xcb\xfa,\xb1\xf7\xd3C\xbd{{\x98\n\xa3A|\xb4\xb1\xae\x8d\x11\xf4-\t\x99\x1e\xf4\x96\'\xb4\x19\xd1\x1d\x06\xfc/\x1daVayh\x08\x15\x8e\n\xf6\xf5Y\xf7\xd1\x95\xea\xd2}}\xd3\xab\xd3\x0b\xccS\xc0\x05 O`I\xba\xf7\xa9\xdcq\xd4w\x92\x0b\x88\xeec>\xb1\x80\xc8\x1d\x16\xf6\x9c2\xda\xc1\xd5\x1b6G\x1c\xc6?\x986\xba\xc8\xd6\xf7H\xca\x03e\x11\x1al\xc8\xe3;\xb5\x95\x04\x0c\x1c\xbf\xe5%\xb6`_\xaf4\xf5\xbb\x01\x0fY\x10\xaaY\xf3Tbpgb4\x88\xb5\xb1v\x81\x98\xf8\xc0@9\x8d\x8aH\x9au\xe2\xd5\xc2\x9e%\xca\xf8yc\x8eM\'\xb0Q\xf8\xbdJH\xd4_|\xa5\x90\xe4\t\x84=Cy\x1b,\x8a\xc1\xfc0\xca=\x13m\x8b\x0c\xf2\xabS\xce\x80S\x83\x11\x1b3gXf\x11\xa4d\x11\xfc\xe5d\xd2r\x9ct9\x83\x92\x12\xcc\'\x90\x02\xf0\x9b\xb4ig\x00\xb7\xa8\xd8\x90\n\xf5\xb8\x8a{\xe7q\xf5\xf2\xb7\xcc\xca\xd5\xa5q\xf8\x03\xa0\x02\xb4T^\x81P\xbd\xd3\xb8\xb5Z\xbb\xb3^\x9f>\xcf\xa6\x8aD\x11y\x0e\xec\x0e\x01\x13\x1e\x04;\x87a\xdd\x8b*\xaf\x98e\xa5{\nd\x10\x82\x9c\xc9\'Am\t\x81M2\x05\xc4YFJr\xb9cH\'`\x8b\x92TD\r\x05 \xfe\xe3F\xf5\xe2r\xfd\x81 *\x00\xa8\xfa\xf0{wm\xad11\x03N|`\xb7\x8a`\x93\x8cD\x81p#R\xec\x80\x10\xae\xac=\xac.\xbc`\xfc\xc2\xf3C\xe5\xb0\x87\xe8/\xcc\x8c\x9d\x02\xab \xe8\x9d\xa0`h\x13\x11\x806\xcf\xd3%\'\xb4\x8dx\xad\x90w\xc0\xaf\x01\x8d%}b\xd0\xc5\xa4O\xf5P\xc4\x7f\x8eOH\xbf\x7fBw\x80AT\xb0c\\\xfdn\xd5\x9d\xbf\x07\xa2\xb1zy:\xb0:\x92i`kG\x03\x11\xacLH \x8b\xd9\x9f\x83\nq\xb0s\xecB\x94/\xb1c\xfd\th)\xfc\x84\x11\xc0\xb8\xed\xa4\x81\xaeKVTx\xdf\xe2F\xfe\x80\xb7JA\x80q$\xde\xa7VK\xe2\xa7\xa3\xdc\x97?\xe7\xc0\x07\xec\x19\xfc\xf0\xe7\x8d\xab\xf22\xeb\x8b\xda\xe9u\xf8\x86\xfd\x7f%\x01\xa4L\xf6\xf0ig\xad\x1cF\x7f0\xd6c\x0f\xe4\xd8\x0e\xd8\xd9\xd5\xe69\x08@\xe4\xc5\x12\xfa\xd36\x19\xe3\xe0h\x97i\xd72v\xd6\xc6@\xc4\xea\x8f\x8d\x07\xdfa\xf0T\xbe\xb0\xb2v\xa9\xf6\xe3=w~\x1a\x9eC!\xb0\xf1Y\xcfi\x98\xbb\xe7\xdeYh,\xce\xbb\xd3\xcf\xeb?|_\xbd>\xef\xae\xdf\x03\xb1\xd6\xb8\xfc\x02\x1e\x04\xf6U`_(\xa0\xa2P*\xe2*\xf2#\x7f\xf9[|\r\x88\x10\xdf-\xea\xa2\xc2/\xdf\x8c\xe8\x13\xb7\x04\xd0\x04P\x03}\xf8@\xf6a._\xceI\x9b\xfcoe\xcbA\xc3J\x05h\xca\x05\xc4\x1a`\xea\xd3\x9d\xedI\xd8.\xb8\xda\x0f"\xa4\xf3\xa3\xdc{\x1f\xb0_%\xc36\x95\xcdkhA~\xba\x13\xdf"n\xa8\xdf\x9bs)\xb6\xe0>\x9a\x12^-\x91\xab;\xffUe\xed<\x061/?\x050\x01\xa1\xd5\'\x97\x85\x83\xe5\x85u\xf6\x1f\x01\x11\x8a\x1e\t\x08\xcf\xe8\x07m\xf0L\xe3wL{I\xcb\xceD%\xe1\xc1\xc5\xae]F\xbb\xf1A\xccx\x1f\xee\x89\x19\xef\xc1\x9f*xR\x90\x86<,\xd2*fF\x90\xfd$\x9b\xa3\xe9\x0b\xee:/\x836\xbd\r\xae\t\xc2\xe8\xb7\xc1\xce\x07\xb3\x12.X\xe0\xe1\x9c\xd4\x18\xb41\xf1\xcc]\x99V\x8fa\x03\xb9\xb6t\x03\xe9\x150,\xd7\x01K\x04r\xd0\xa9C\xc4\xf3\x0e\xcaG\xaa\xc0\x16\xdf\x1a\xa0\x10\xb1~\xb0\x9aw~\xd8\xd1A\x11\xd9{\x13\xd5\xc7\xa7\x8dQ\xfef\x8c!wPH=\x9ft\xa7\xd6\xa5\xf9N\x9b@@\xf1s\xe9\xf7\xc0F\xc8\x88k+\xb5\x1b\x0f%\x9fy\x02\xec\x00\xcb,\x0b\x05>p\xc2 PK;`\x80#a\x9e\xb9\xd7\xde\x89BX\xd1\x89\xf4\xc9\xc4\xba\xe0[\xb6\xc7\xdcW\xeb\xe0\x93\x82\xc0DR\x10Oj\\Y\x06\xa9\xcc\xa1\xb2@\x06\xc2\xb1\x92E\xb0\xdc\xfd\x01/\n\xe1\xe2\xdb\x0b\xe8}\x81B\n\xcb\x14\x90\xe7#\xb1)\xa2\x9a\xe8\xcc\xcc)\x1dU}\xbc\x0e\xa0\x81^r\xc7g\xf8\xe5l\x82\xa8<\x80o7\xc0\xe7w\x7f\x98\xe1\xe5(G\xf2\xd8\xc1\x04\x06\xf1\x80\x10\x05\xa0qp\xb2R\xfd\x16\x98\x08\xd1\xae\xdf\xfdngW\x0cmG\xbe\xa5\x1b\x91@\x86\xa3\x9d+E\xf1Z@>\x81\x12Vq\x17Z\x9e\x88\xb4\xf4\x97\xedLJ.\xc9\xceI\xe7XZ3\x80\xe9\xa5e\x81VRlz8\x1a\x94\x1f\xf2\xd8\xe4\xb4\xd0\x1c\xb4L\xf6\xd0\x82^\x19R\x93Z\xd0\x1f\x0c\x02\x1f\x84?}\xf4\xfbaF}\xea\x89\xb1\xab\xeb\xe3]\x1f\x7f\xf8Q\xd7\xc7\x1f\xa0\x8a\x9fF\x02\x06~\x855z\x94\xd3\x07\xd6\x85\xeec\xd2\xa2\x1c\x90\x19\xe8\x8a\xa6\xc0\xc4\xc0\xbdK\x02K\xf5\xa3\x80\x01U\x98*\'\xe1\x8e\x0c\x10[\x11X\x89\x8c\x04w\xf5^\xe3\xcaU\x11\xff\x96\xf1T|\xe3\xc2\xcb\xca\xe6\x1b\xf0\x01\xc1\xb4\x00\r\\\x9b\xdb\x14T\xc4i\x9dg\xf3\xd5\xebJ\x99\x17\x8a\x88\xf1tD7P\x08\x18\x8c\x1f\xc9%\x8fI\x0f\xff\x8f\x88n\x80\xb1\x9c\xb3\x81\x96\x15+\x8bxo\x01\\\x03\xfb\x14[H$;\xd0\xe9\xf1\x82\xc0\xd5\x9b/1\xacJ\x02\x8d\r5\xde\x19\xa0)4\xc5\xee\xa0\x8d\xa8\x1e)\x82\xe9\xe7fk\xcf\x95\x99(\xe2\xbc\xc0BY\xa0\xaa\xc3 \xd2\xa2Z\x808\x16\xc7/\xa4\x83\x7f\x12\xcc!\x13\x1c\x07D\xa3\x9e;AW\x8f\x99RF\xf3Ah\xa1\x98K\x0eb\x08\x00ys\xf6\t\xa0\x94\x89\xb9v\x15\x84.\xdai\x95\x977P1h\x1cRys\xadvy\x91\xc3H\xb0\x9a\xea-eA\xc0+)\x16m\xa0\x15;\xaa\x01=\x16b\xc8&\x84!\x9b\x18\xc5\xb4a\x1c\xff\xd9\x05\x1e\xc4\xa0u\xea\xf8\xee\xdf\x9f\x90\xb1\xbb\xbd`l\xa1\x94A\x85\xe2\x99\xea\xa8\xdeRVI\x8a\xea\x7f\x19\xb6r;\xdb\xbf\xf8L\x05_h#l\xf0\xed)bZ\xc8\xdb\xa4p\xd8\xe9S\xb2\x8e\xf7@Y\xef$w\xef\xb9\x9b\xd7\xd9I\x82\x95\xd5\xa7\xee\x83\x81X\xdf\\\xae?\xfc\xce{G\xed\xc5r\xed\xf6\x86\xbb\xa8\xe2\xdc\x14Q\x86%\x8f\xb2v\xed\x8c\xec6F#\xc9\x0cpP\xa24R\xb0\xe0cdo\xc6.\x80\xf5\xf2W\x8cDZE2\xaar\x80\x1b\x07o\xddA?\xa3\x9f&\xe16\x0e\xe6\xc3o\xfe\x86/\x1c\xca$v\xf5\'\xd2\x85\xdf\'\x1c0la\x93\x1c3\r\xeb\xce9\xf9\xa2CiB\xf1\x02\xcc\xa1F\xda\xbc\'\x01\xdb\xfc\x953\x9c\xf8m6;\xb8\xb3}W\x7f\xbb\x04\xa3}hg\xbc\xb3=\x9b)\xf8\x1f\xe7\xfd<e\xa1\xfd\x0c?\xa5\xf8\xd1\x81\xc3\xc70\xe0\x85\xcao\xb7\x11\x0c>\x8d\x8d\xf1\xef"]!\xeb>\x96\xb3J\xbcf\x8cL\xfb\xd7\x1d\x01\x07\xb9$W+\x1d \xf5\xb4\x9d!O\x0b\t6\x05P\xa9\x16@\x0e6\\:\x0e`\xb5\x19\x1d\'\xe06\x8a\xc5s\x1c\x13\xbe\xe8\xec\xa2\xb0\x12_\xe4\xc8\xe3n=\xf6\xa4\x00\xd9\x15\x02\xc8\x17{z\xc2v22dZj\xfb4\'\x8e\xc3\x9b\xf8e\xba\xd0\xf9\xa1\x0f\xe7\xea5\x1f\xfc\xfa\xd7p\xca\x81_\xb3\xb3+\xfc5\x1fn\x85\xd6^\x8c\xc6\x1c\xcd\x7fA(j\x81X$Q\xc2k\xa7\xc4+\xbc\x90.\xec\x92\x17DJq\xb7\xf0\x0b4\xb2\xe2\xdc\xdfnC\xe4\x06e\xeao\xb7\xb0C\xd1\xc9\xb0r\x03t\x8b\xb0X$\xdc\x1f\x85\xc0\xfdG\xd3\xb1\x93\x07@l4qT\x08\x19\xec\x94\xd0a\x82\xcfF\xc6\x00\tH\xdf|H\xdf\xa87\xfd>\xe4M\x7f\xeaCG\xcc*r\x987\xb0\x1b\x0e\x7f\xa7v\x04\xac\xa1D\x16\xfce\x8c\x8a\x15<\xcc\x7f\xdcj\x05} \xa3R\xe5\xcc;-\xc2Q\xbf\x81\xa786\x02\x80\xbf\xe6\\\xe5n\xf6\xbe\xda\x90\x8d)\xfe\x86\xe4\x1e\xefP\x90tv\x84\x80r\x84t\xefA\xba\xdd\xbf>zD\x822\x8f\xbb\x95)\xe0=,L\xdc\tt\xed-;\xa5|vOj\xc8\x04\xcb$\xd5jy\x12\xc4\xe3\x08\x98X\xdf\x00\xef*^\xfc\x88\xafy\xf7\x0b|\xd3\x97\xbfW\x08A\x89\xe0\xd0\xb5\x8f\xe55\xb4\x18r2\x91(w\xba\xd3\xdb\xe9\xce\xaep\x9e\xdbg%9x\xde\xbc\xcb\xfc\x86\xce\x16\xb4\xef=y\xe7VO\xde#\xf2\x1c\xdb\x7f\xfc\x07\x81\xc7\x87\xc9$V\x94[\xf2\xaeH0\xd1;\x94X\x94\xb2\xef8B-\xae\xa5\x89\x8a\xbav\xe9D\xf3A\xe8>\x0fm\xfdF)m\x8f#\xcc\xf2\xe1\xc2,J\xb0\xd1\x04\xdfJ\xb3A\xdbd0q\xc1qS\x9a\r\xc1\x18\xf3\n\x048n\x1b\xc1\x881\xe6\xf1J\xe8Sz\\Mn\x1a\x99\x08f\xa9\x84\x96\x10A\xf1>g\x8aa\x17\x9c\xf8\x0e\x8a\x04cFwY\x981\xb7\xafU^=g\xb3\x80\xed\x80\xc6\xf8\xf7\xd5\x8bw9\x12\x85\x01VN\x0e\xc3\xaf\x95+\x12\x88\xfe\xeef=\x8e_\xe1\x7f\xf7\xa2\xe1\x01\xc6\x89Y\x92\xd9\x96~\x8c\x83\xa7\xb5<V\xc1\x06\x03\x18\xa4\x85\x91\xb50\rA\xb7\x99\x14\x8a\x10~\xa6\x08+\xe0\xe3\xd8P\xd7\x13\xf0\xee\xc63\xf0\x039\xba\x00^\x08\x98f\x0c>\x07\x14\x10J\x1d\x1ca\xce\xdb\x8e\x9d\xc3|a\xd2\x8az\xa9\xc86#Z*\x03\xf5\xb5Q&/\xc6q\x192\xfca\xa3\xb4\xfbbx\xe9c\xd8AJ\x82\x12\x10\x18\xc4\xbbs\xc5\xf8\xd8\x00h\xd8Vt\xe7V+\xaf\xee\xa2\x9777_\xd9\xbc\xca\x08\xc4\xa8\x86\xcc7F\xfcx\n\xc9.k\x99>\xb0ypC9\x01\xeaP\xe4qD d\x0e\xd3\xc9\x9cK\xae\x9d\xb9U\xbd<\xede:\x97\x16\xb8\x14\xa51\xb1\xd6\xb89\xd9\x02\x15\x81t\x1f\x05\xbc8\xa7G\xc9\xbdS\x91\x18\xaf4\x11\xb8\x11\x96V\xff\xf25\xc6G8>\x0fk\xc3xC\xf3\xcazN\x95\x8a\xa60P;E2\xf90o\xb9 \x07X\xd0\xa9B\xc6N\xda%p\x97\xcb\x8eW\xec\xc3\x1a\x94\x17*\x12\xcf\x1c\xfb2*k\x0f\x03\xbe\x8fz&\xd8\xa5\xf5\xc5\x1b?oL\x83\tkt\x1a\x8d\x85\x19\xf0L\x7f\xde8\x17\\\xbf \xbb\x84\x84\x01\x0cWt\x84\xf8c\x8c\x92\x94\x82Z\x01\x9a\xa2\x15O\xdb\x98^\xceD\x8b\x91?\x88W\xfd\xc5y/\xfa\x97\xd4{\xb1O\x00UB\xc7\x1b\xe9\x8c9\xe0t\xc3\xed\xbd\x07\x0e\x1e:\xd2\xb3wO_Ol,\xb8\xd5\xe92V\xf7\xe4\xb9\xd2\n\xf3D\xa2\xd4I\xee8\x05\x9ds\x86\xd9\xef\xa0\xc3B\xe2J a\xe6\xae\xc4\x00zQ\x18\r\x9au\x1f~\x87\x85w\xb0\xeb\xfe\x82\x94\xc6\xf8\x04\xf8O\x01\x9a`\x06oA\n\xb8:\xca=\x8au\x87P\xbe\x1f+\x01\x14\xaa\x14\x06\xa71\x90hD\x9d\x95\xbbq\x19v\xa9\xb2~V\x87\xd7\x90x4\x0e~\x82\xbe\xa6Cd\x1f\r>t\xec\xe7\x8d\xab\x9cG\x10y\xec\xd1\x00Pc\xc8uDw\xf8\xde\x98\x1f\xd9Z\x80\xd9\x06:\x83\x87\x1a\xe6\x80\x89\xdc/*C\x87,_X\x8b=\xc3B\xdeq0^\xc7X\xaf]\xbd\x05\xec\x0c\xd4.\xebM\xbe\xc7\x98\x02\x87\xb45\x87U\xc8\x9d\x7f|[\x7fx\xae\xfa\xf0N8\xd7\x91#\x96H\x0e\xe6\xc1\xaf\x88z&\xe7\x11\xc9\xeb\x9a\xd9\xd9\x94k\xf7s`,HUeQ2\x96C\x9d\x91\x91.\xb9\x1e\xce\xd5\x8b\xc6\xbcR6\x12#\x94\x1d\xabOOq\xf1\x86\xc8\xbf\x93\xef\x8bum2,\xab\xea\xc8\xbcH\x8f\x0e\x03y\x80h_\x9c@nje^\x87\xad\xf5\x17\x9a\xd8\xca\xd4\xd7\xcc\x03\xbf\xf7\xe7\xb3\xc0\x7f\x81\xe5\xad=-\xb0\x1f\xbb\x03\xfb1\xd6\xc4\xe6\x9c\x10\xe6(\x84\xa0j)\xd9\xf2Et\xcfI\xd6\x89J\x03\x01\xab\xd3$#K\xa6\xdc\xa4\xca\xe65\xc9\xfd\xba\xc0\xeb\xc4\x94\x80\xf8\xbb\xcbh\xdc<\xeb\xae\xcf\xd5\x16_a.~u\x0eXC\xd7\x96\xc1\x1dC~\xe65a\x88\xe4T\x9b\xf8\x80u#\x08\xaaE\xd5\x11\xc0>:\xbf\xed\xd6<u\xf1\x18\xfc__)_\x90A9-\x93\x03\xf6\x01\x95zq\x05\x9eO\xad\xa5\xf2X\x91\x9d/q\xda*\xae\x9e\xa4\xea\xb8t\xc8\x95\xe4\xa8.\x9e\xe6\xa0\x9d;\xb1T}x\x9bc1r]A\xa0\x9c|\xb9\x98\xb4d\xac\xc8[\\,|\r\x9a\xb5b\x1aC(:D&\r\x8d5\xaa\xa6\xa1\xd4?\xec\x15\xdaR)\x0fd\x91O\xd3\xe0\xad\xbfXu7\xbf\xe2\xcc\x12&\x12(@\xc4+h\x05\xac\x10\xc3\x0cs\xdcv\x12\xf8\xd6(\x95\x14\x93\xa1vn\x13\x910\xfd\x08\xf3h\xa3\xda\xae\x19\xef\x19\x9d$\x04\xfd\x85\x89(Q\xf1Ic\x91\xf0\xc5\xeeI&\xadBI\x8a\x0b.A`{\xd3\xe1\xa0?x-y\xa2O!\xe1\x808\xd1\x0f\'\xf7X\x94\xbf\xc8g\xa1\xb5\x85\x05\xb7B\x14z\xb7\xa1\xe0\xa4\xc0\x11n\xe7\xdaZ\xfd\xde\x8c\xb0\x98(\t\xbc=L8\xe54\x18\xc8\xf1L~\x18k\xa8\xd9D\x89\x17\xc0m\x05v\x8d\xff\xb5 \xffk\xf1\x1f\xc3V\x7f\x81\xfe\xe8\xcf\x16"\xb1`i\x9c\x8e"Y\' \x11%^\xd4\x12_\x19\xa4\x85\x12f\xc80\xc9\x9a\x94\xf0\xb1\xde\xe8\xdd\xe7\x1075\xdb\xb5\x1a\x95P\x001\xb0Q\xee\xf4\xd9\xc6\xe4,\xdb\xdfTCL\x85N\xa4K\xe0\xa1\xad0\x84\xefL\x00yvcYU\xb4\xf3C\xa0\x01\x8d"Z\x91w\xbe0B\xa4\x8c\xd5\xe4`\x7fx\xfbZ(\xdaC\xb84\x92\xc3\xb0\xc0"\x18\xa3X\x1b\x84\xb9\x03\x19\xeb\xa5\x02Go1\xee\xa3)\x1f\xbd\xdf\x99\xaa\x7f\xff\x1a\xa8\xd3\xd3\x87?L`\x8acc\r\xd6S\x7f2U]Xt\xefl\xb2W\xc1\x8aT\x85}\xbd\xc2\xdf0\xa8S\x16\xa2\x9b\xa5I\xb7\xb1\xf7\xd0\xe7\xfb\xff\xcdx_\xa8\x8c\x08\xfc\x95F\xbf#\xd1\xc4\x10-\xe2\xa9\xa3\xa1D\xd5j\xcfE\xd8\x95r\x0bD\xcfA\x04\x81x\xe3\x9e\x12\x0f1`\xa4\xa9\xe5*\xa9\xe5\xc2FS\x9cu\x1b\xab\x8c\x833\x07\x86_<{\x12\xde\x13\xe5\x0fN\xf7\xd1b\x19\\\x13\x12\x95\x89\xfcI\xfa\xd8b\x9b\x0f\x03\xd1YE\xd1F`\x9dB\xe3\x9b\x81\xef\x1f)Y\x8e\x97\xc7\x05\x1a\xa0\xac\x02\x9a\x03\xb0\xc7\xb9\x12F\xc8S\x06\x98\xeb9\x07\x05\x81\xb6\xd9\x1c\xeb\x16\x04K\xb9\x1ew\xf2\x07w\xe5%\'\xe9\x1a\xb7f\xdd;\x0b\\\x1d\xec\xd73M\xf2\x98\xfad\xe2I \xc4\xae(\xefD\x9b\xbe\xf4\x96\x84\x9b\xcbqE\x15Wy\x16S\x01I\xe4u\xf4(\xfe\xf3\xb1]c|\x11\xd9\x8e\xe1\xe7l\xd0\xf2bu}A{\x0c\xb3\\+\xc0\xd9\xb4\x11\\\x17f\xde\xa8\x07\x05"\x1a2\xec\xa2\xef/r\x924\x17\x9a\xa4\x0c\xf8\xf9\x03El\xbb\xa0\xfe L]\x80\xa9\xc89\x8a\xbf[\xc5<[\x07m\xb0a\x19\xfb$o\xb12\x04\xa4\x1f\x01&C\xbc\xf9\x81\xcc\x89\xb5\xa7\xb7\xc1\x1aG\xdf\xa8C\xf9F\x170\x07\x1efY\x18\\\xd5\xbe5R\xd0\xde;.\x97|\xe2xZ+\xe2\x8d{6\x93\xce\x9fcd Jl\xa2\xd1\x16\xa8\x06\xf1\x84h\xd6J\xd9&\xa7\xca\xd9\x80e\xb7\x00\xd3\x9b\x0co!c&\xad\xc1|\x06\xf5U|\x07[\xdf\x92\x06\'d\x05\xcd\x04\xbb\xdd\x1c+\xf0/\x96e\x18;\x8d\xadc\x1a\x9aR\n\x0b\x06\x10d\xf2\x0b. \xa6/v\xfa\xbe\xe0Ja\xfeB\x84\x0f\xf4\xc0\x81\x82z\xa7Q]y\xa6@\xa7O\x81b\xe5p\x0f\xbe\x0f\xccJ\xc3\x83@\xbc\xa2>\xf5\xc4}\xf4Uci\xbc~o\x02\xd3\xb9~\xed\x03n;\xd6\x1e\xad<\xd3\xab\xa3\xc1\xe3\x12\x08\x9b[\xad\xbc^B\xfeY<\xe3\x9e\xbf\xa9\xbc\xc7\xc8V\x08y\xaf\t\x13\xef\x85\xa1\xa0\xb3K\x86Px\x07\x9e\xde\xac^\x9bw\xe7\xa7\xeb\xcb\xd3\x02+\x9d]\x86;\xee\xa5\xd6\xf8\xa5h\x0e\x13\xfb\t"\x05\xd6\n\x00\x00\x8f\xfdB\xc4\xdb\x02\x80x\xc6<Q\xf2\x1e\x11\xf2\x0b\x02\xf7\x9e\x11\r\xae\x00(B\xaf\xdd\xa6Du\x078\xf6\xd2\x86\xce\x98\xfdV\xa6\x8d\xe9\x94l|4\xda\xed\x92\x95u\xa2\xb1\xdd\x9e\xd2V\xa1\x86\xe3\x14j\x10\xfets\x80\xe1\xb7\x8e\x17[0~+\x9f.\xbd\x18\x8c.\xc4N\x04\x8d%\xcf{\xc7\x7f\x18\x12\xf5x\xcfCOG\xfe0J\xcf\x1b\x03\xb7[\x04K\xb8\nB\x04T\xd6\xce\xbb\xa7\x9f\x00\x9b4\xa6\xc84\x12\xbf\x1c\x0b\x0bY\xe9\xed\x04\xb2\x14\xbf\xecP\xe8\x91\x03x\xd2\x14\xa0\xefHPk\xea\xc7\xc2\xc0\x10XI\x050\xac\x93\'I\\\xb0\x9b\xc3\xf9IUSFA=&L\x96\xd1l")\x82\x053\x0eX\xda+\xf3\xa7;\x91~_]\x08uw\xd8\xd1QU\x8eMn\x8e\xd8so\xdb\xf2E{\xc0\xc6\xbe\x03\xe1F\xc8\x9f\xc6\x82[ o\xf4lwn\x86\x92\x97C\r\xd9la\x17\x99\xab\xd9\xfc\x90\xb4_\xb3l\xb7\xea,\xe9\xb9?\xd3\xaa\x80rV\xd6y]\x19\x95\xaf\x18{\x17\xb3ipgB\xb54\x84YK\x9d]`.\xb5\x00~\xec\xd7Z.\xa1\x16\x81|Y\xc0&ha\xf4vu\xa0\xd5K\x06\xdf\xef\x8c.\xef6xo\xc9\xd1\xef\xeb\x0c\xbfo[:=$DA\xa1\xfbp\x95\xee\x7f\xb4\x04$\xec\xd9\x07\xac\x12=\x1a\xc8\xbc\x90\xcf!\x9e\x82\xd9M\x993\xd0\xd4d\xd3\x1bBu/sb\\\xdb[]\xebJ\xa0H\xed\xca\xa7\xf9\xe5\xdbn\x9f\x9c|\xdb\x9bD\x8fK<p\xa1\xd5[;O\xf8\xc5\x08u%\x05\x9bw\x0c\x0c\xfe`.\xa0\x98/\x0fp\xf1w\x92\xfa\xaa\xe8K\xcf\xf7\x0c\x95\x15\xac\xbbtY\xe1yW\x13?\xc1\xb7"\x16\xaaj6\xdeYD\x08M\xf1_%"\x86M\x16\r\xd9\xc2N\xfao\x1al\x1e\xfa#?\xc0>ov\x97\xa9\x8b\x0cF\xc0\xffE\x91!6\xf8\xff?\x91\xb1K\x89\x8cw\x14\x03a\xc9J\x992|\xab\x1c\x08\xe5\x1d\x8dkB\xf8\xa5\xc96\xde\xc39<\xd3\x18\xb2\x8a\x1cH\xe3\xc8\x94h\x1d\x11\xb5\xa6\xfc\xb1\xdd\x97\x06dG"\xbe\x03\x1cv\x11\xa6\x92\x11x.\xf8\x16\x86\xe0\xc5\xbb\xe8\xbc\xc3#\xc4g\xb2\xf5\xd8g\x90v\xb1\xec\xc6\xd4\xc3\x84m\x86\x02\x95\x16\x98\x90\x1cs\x9c\x90\x10\x15e\xd3\xbe&\xd8H\xe7\x87TBMm\xafT\x8b\x19\x11\xb6YT\xd5U\xbf\xf3\x0f\xbc\xceZ\xbc\xff#*\xd56\x9bo\xcf\x84\xdc\xfc\xfb\xe6\x9bO\x04\xf2\x98\x87\xb0\x94S\xe4QE\xebp\x91+\x13\xf5^1\xb4y\xb2J8a\xbe\x02\xb0\n\xe2GkjcC\\\xa5k\x82\xe2Gs=\xf0A-b\xad\xaa(2\xe7\x95\xfc\x8b\x00\x1e\x06*e\x83\x9d\xea\xaf\xcd\xfa\x82u\xa2u\x86\xba\x9bU\xa7\x9d\xd8u\x92\x89p\x03\x18\x83[\x07+\xdf1\xb2\xda::lj6$A\x0fN<\xcd\x1f\xc8\r\xf8b*\x8100\xc6\x80\x81\x88eC-\xce;\x98}\xc2!\xe1_\x10_U\xb1\xd5w\x0b\xa1R@\x8d\x0bO1,$\xfa\xe5\xa46\xe2\x1dQ\xb1"\x7f\xe8\xacE\xd0L8\xe9WW\x1a\x17\xce\xfc\xf2\x80\xd8\xa8\xe0\xc9\x7fz\x10\xacd\x16\x07((A\xdd\xfd\xe4us\xc6\xac)\x08vo\x81)\x9d\xa3-b\xb9\xfeX\xe0\x7fu(,\xd0\xfc\xaf\x1c\x11*\xfc$\x16*5E\xbc8\xd6\xc5q#=\xe2E\r\x9c\x1c\xf1\xfa\'\xc7\xbaT\xde\xc8\x0b:R\x9baQ\xe6\x8e\xf6\x1c\xee\x95\xdd\xfb\xcd\x04\xa5E\x8f\xc1a\x87\xbf\xe9v\xbdW\xff\xffitK\x06\xf2(\xf7\xca\xcd\x93J\x81\x81\xea\xd1\xf5\x96\x7fu/~\x12^\x1f-\xb31\xfe\xbdPVA5\xb5\xed0\x95\xa7\xac\xc2\xf5\xed\x11\xab\xbd\xc8\x0en\xc6\x1a2\xc1\x97N\x15\xf3\x05\xec\xf9\xc4\x1e\xff\xa4\x89\xc9V\xee^\xc4b\x11\x07#/T\xec\xac\xf5\xd3g\xcd\x11&-K\xf4\x84\x81\x91[,\x95\x0bT\x92\xce\x93P\xb09\x8e\x1a\xa4X$W\xd6f\xaa3\xd4\xa5\xb9\xb0\xa2\xf2\xf8\xdc\x9e\xe5n^\xe0\xf6V\xd1\xe66-z"\x1a\xe37\xa9\x17p\x16\x93\xb5+/\xa5\xa6>\x08:$\xd1{p?\xb6\x99\x8b\xae\x03\x87\x9a\x12A6\xfcqO_\xcf\xd8\xfb\xf9~,8I\xd8\xb94\x9a5%;k\x01+t\x7f\xdc\x11\x8b\xe3\\\xa3h,\xa4?\x0b\x17\x88S\x1a\x0e\x1f3:\rNa\xfb\x92\xe9\xa2\x90]\x16\x1f\xb3\x95\xce\xed-\xb5\xcb\xcfQH\xcb\x92a\xd1\xf9q\xe5\x0c\'\xc9\xb9\xaaG<\xd8\xdfD\xe0\xcf\x977\xd7\x0f\x8b\n\xdd6U\xa1\x1b\x06\xb8\x84\x88#\xdb\x16\x1b1^;\x93\xe3\x94\xb3\xc8\xfer\x87\r\xd8@\x13~l\xa2*\x12M\x90\x05\x14$X=-\xf2\xfb\x0bS\xee\xc3o\xd5z\x986Y\x97\xaa6\twb\n\xcb\xfeE?\xc98\xd6\xbc/\x9ds\xafb[\x85X\xf6\xda$l\xf4vW*:\xd9\x83u\xc7r\xbdh\x15\xb2.\x02\xae\x10\xb5\x81\xbe\xdd\xa1\x0c,`\xe5\xef\xc0s\x142\x97]\xf8\xa8\x92ee\xa7,1\x02\x1a=s\xdf\x9d\x9cVsu8\xed\x0fKe\xead,\xfc\xbcq\xd5}3\x89\xcd5\x8b\xa7Aq\xc2>Wo\xae\xb9o\xbe\xf4\x1e\xe7\x15A4/\xb1\xb9\x9a\xd4WB\xda\x16,!m3\xbc\xdf4\xdd\x1dk\xc6\x83*\x0e\x05\x05of\xa8\xe8\xa1\xdf\xc4\xf4n\xde[\xb7,\xa2t\xe4\x94#\xf5#\xf2\xf8\xd4v_\xfc\x1a\xdbE\x16\x9712Jj\rV,\xaaAh\xc5\n=\\\xee\xe5Q\xf6\x16[\xdc\\\xf5\xea\xf8>\xc8r\xd6\xc0\xaa\xd5=r\xc9\xb2\x01T\x8d\xc7 3\x99#\xcf`\xd3\x8a6-\xf6\x02\x1c\xd6\x9a\xcd\x16"Ka1\xfe\x06\xe4\x93\x18\x14A\xd2V\x18Y\\\xc2F\xd9w\xd1\x96\xc8\\L\xc6"\'X\xf5\x89\x1f~\xd7A\xda\xb5\xe4\xd0\x86uR\xc6v\x87D\x17I\xaa\x0c\x99v\xc6\xc4\xe69\xb5@\x9f{\xc3\x12\xdd\xeb9\x91\xd3\n\x88\xc6S\x165[R\xf7\x99WE\xc6\xb5;XC\xc6-gZ\xd5#\xef\x9c{\xeb\x81\xb0U\xee\xcd\xc1\rM1\x02\xc4hB\xbc\xb8\xdbP\xc2\xf6x\x8b\x1a\xf2\x13\xdc\x0f\xce\x06\x1a\x8d\t\xe3\x0brA|-\xb8~6\xf1qI\xbc\xa1\xf4\xd66\xa3\x90)s\x91\xa4\xb6\xb9\xd4h\x06\xa8\xdd\xff\xd9\xfe\xae/t\x0f\x04\xf6O\xc6N\xb9KI$\xc2\x7f\xfc\x91;\x8e\x91\x9e\x9b\xa6\xb44N\x9f\x11D\xae\xf5&\x07q`\xd1\x8c\x03\x0b=\xeb\xa8\xcf1\x0b\xb8R\xbe\xf1\x00\xbc\xd5\x1c>\x8fF\xf5\xfb\xda\xfc7f\xbc\xdb\xfc\x0f\x8f\xc5B\xeb&M\x8crk}\x96\xc4\xdcD\x01\xc4EY\x10\xddl/7\xb9 \x02Q\xf7NW\xaf/1/K\x89-\x8a\x06\x1a\xe3\x8b\x95\xf5\xab\xd5\x1f\xcfT^\xcf\xeaN\x07vW\xd1 \x1f\xbf\xd3\x11RS\x07\xdez\x86j\x1a}t\x83,\xa8\xbe\x91\xd8$g$\x94\x88\x8c\xda\xc6\xba\xfb\xe8+cT\xde:f(\x0e\x8b\x04;\xc1E\xb5\xadd\x051\xdf\x81\xf0!\xeaK\x1dj\x0b2enP8\x8f4\x88Av\x87\xb3\xe0\xa7i\x11\x8c\x12\xc6\x87;\xb7Z\xfd\xe6\xa9;9\x19\xa8\xa6\x08\xe7\xf7\x16e\xc1\x8a\xbb\xe5@\x01Q\xea\xab|j\n\x8aI~g\x8c\xe9\x1c\xcc\xb0)\x96\x05\xdaF\x96\xa5,\x80\xe2\xdd\xe0\x8eP.r+\x9em*Lk\xe6[\xff*\x0e\x80\xbf+\x0cu\x9e\xfe\x94O{(eg\xc5\x07;\x0f$R\x08\x14\xbe\xde\x953\xd5\xd5Y\xa6/o\xeaS\x10p\xa2\x14\x87Z)KQo%r*\x0fc\x8da\x8c\x19\xff\x1ez\x97_\xda\x84\xf3\xd1\x10\x18\xaeT+H4\x94\xca\x8b\xf2\xac\x02N\x90\x0c\x8e+\xf1\x88G_"1\x92\xd8\x98s\xd38\x08\x8a\xe8\xb6\x99R\xa8\xbb\xf91\xd7/o\xc9B\xfaP(N\x95)t\x00\x0b\x99\xb9\x91(u\x17\x92u\xed\xa0Q\x17\r\xcf@GD\xbaM\xc4\xa7\xbc\xc7\xf8\xeaV[\x93\x83\xe4@\xef\xe1\x86ZI\xfd\xc1\x0c\xda\xe6\x0b\x8f|\xc3eD=jpJ\x03\xd5\x9eg2@\xe5*?\xc6h\xf7\x98\x13\xa7,z\xf8\xe6$\xb8\xf0\x02s\xac\xa3}<\xaa\xe2\xdf\x93/\xdc\xd3\x8b\x81AW\x1c\r\xf6\xc6\x99jE\xf2[\'\xbc%Y%\xe8\x95\xe4\x15\xfa2"!)\x8c1\x94\xe2\xda\x14-\x12\xe1H\x8c1\x19\xdd\xf2\xc2\xe7^\xf4\xdc\xff\xa2\xb8\x99JE\xb5\xa0+\xffT\x84\x84\x14,\x01\xe0\xda\xe9-\x8a\xabc:\x01!\r\xfb~\x8eBVU)\xe9\xe4y\xff\xeb\xc6\x83K\xb2\xfdAU$\xfb~\x8b\xf5\xc8-\xb6;\xe8x\x98$\xfe\xd9\xfd4d\x03jr\x04<\xc6\xdcIG\x86RT\xfcN\xec\xab08\xcf\x8dc\xa9\xb3\x9a\xa8\x02\x0eo\xed\x05\x96DV6\xcfV\xd7\xe7u\xcb\xcb\x0b\xd3\xc9\x10\xad\xf2eY\x01\xcaY\x7f*\x1d\x1d(\xcf\x07\xa1\x9f\x13\xbd\xb2j\xfc\x95\x0c\xf0\x14U)4=\xac\xdd\x0b\xa1\xd0"\xf4\xa6\x0c\x99oA\x8e\xc0! 4Y\xc3\xbd0\xe9\xce?`w\x9d\x07\xc7z2\x9b\x1c\x86\xb0\xf4\xcb\x90\x99)\x13\xdc\xf8J\xcfe\x8f\xd3uo\x05A7\x9fB\xb4\xa5\xe1|\xbb\xc5\n_x\xe6X8#7\x9b\xb7"\xc9A\x08\xf6sT\x90\x81\'$]]t\'\xa9\x16\x81\x0cwl\xc9\xe7\xca\xefW\xdf\xd7o-\x07K\xbdiSZE\x1d\x80\xd6\xb56\x13\x82]t\x96\xa8\xa2\x10\xbaHS\x81\xba\xb45\x05\x1f\xa4\xc2\xa7v\x96\x04A\xc9\xca\x8cp\xe0\x14\xf7\x80\x9a\xc6\xbd2\xc4\xb0\xe0\xa9\x08\xa0"\xb8\xcf\xe6a#\xf4"|\x9d\x9e\xd0\xb0\xa70\xaah.\x9f|\x81\x93\xbchz\x92\xbe\xc80\xf8dh\xb5T\xe4E\x1d\xef8\x11S\x94\x87\xec&\n\x1aU`\xd2\x08\x85F\xde%\x9f1\xa6\x1b7Y\xd4\xa8L\xa7\xaa\x01\x9a\xe94$\x08!H\x9f\x8a\xf5\x94\x0f\xe0\xc8p\x84r\x00\xbc\x1d\x95/g\x8b\x9fI\x1a\x8b5\xc7\xbd\xceh\xc3?Q\xf7\x9dYM\x18=\xb6#\xad\x1di\xfcH \xd0\xb7\xcb\x9a\xdc\xb2E\xc5\xb9\x8a\x13\xb5\xda\xcbf\x03\xc8\x9d\x9c\xa8/?\x16\xf3g\'\xefneJ\x887\xe8\xf6\x0f3\x99\x16\xee;\xf16\xb3\xa77\x97\xcc\x94S\x96\x12\xc1m\x9e\xa5F-\x06#\xd8\xea\xde.\x8b\xa4\x8dA;\x05r\xcfo%<\x9ab\x93\rK\xb3\xd8d\x9b\x9e\xaa=yU{\xf6\xb8q\xe5q\xfd\xdb9\xb1\x86\xc9\xe5\xc6\x97\xcb\xb5\xa7\xb3\xb8*5,\xcfg"H4\xb1a\x14\x8b\x97s\x00H\xf4wQ\xfc\xc4\x8b\xa5\xc5\x80\x9dR.\x08\xeb\x08\xf7\x8d>\xb33\xaa\x19O\xba\x89\x14a\xb0\xc1\xcb\x0e\xac~\xbf\x8d\xd3\x87\xbc0_\xd1\xca\xe6\x87p\xd8\x07\xd6\xfe\xe2>QB\xc8\xe4A\rjkqD\xb5\xa3\xdc\x8d\xea\xb9M,\xb8!\xd1R\xbf\xfd\x00\x03\x80\xe7\xb4\xb9(?\xbcj,\xde\xc1\xac\xed\xa5\x97\xb8\x8drdtp\xf1\xe5\xdc\xc9\x1c\x86\xabx\xed~a\x19\x03\x8d(\x91\xe3\x87\x1f\xcb\x98\x87\x8d\xd4\x08\x80j\'\xa5k\xc0\x16\x9d\x93\xc9\x83\xc0\x0c\xce@\xa4\x1e\x99\x80Y+\x1ac.?\xd7\x87\x1c\xea\x96\x91;s\xb9:9\xc3\xdb(+\xf6\x91Jy\x96\x80\xfb\xf8rca\xc6\xcb:\xcd}]\xfd\xe1uH*\xac\x996\xc9Dim\xb1\x13\x01\x06\rF\x89\x87\x96\xc94\xdc@\xf6\xa1\xb9\xe6\xdf\xab\xb3\xd4\x9b\xf0\x089\xdeh\x0b\x81|\xc7*\xf9\x94H\xed\xf2\xdd\xc6\xc4E\x9e\xcc\x83\xb5\x94\x1d?\xbf\xbe\xfe{\xc3WrG\xab\x95Cu_\xc3\x0fX3\xb2\xa4\xdd:\x94-\xdf\xda\x0e\xc6X\xcbB\xcb\xb1\x08W1\x90\x95\x8b\xf1\xe7\xe8\xc7\xb11]/\xe9\x93L\xfd\x08\xd2ez\xf3\xbbB\nK\x9a\xdf\xb5\xb3\xd5\xbb\xe4,\xd4w~c\xeb\x02\x93\xed\xbd\xfd\x17\xbc7$9\x1f\xfe.\xbfh0\xed\x0c7g\xb2~\x0e\xa8)\x12\xfc`\xc6\xc3o\x8b\xde4\x1b\xc5JA=\x83\xe4qv\x16\xc4\x83\xee}\x8b\xf8\x9b\x96\xed\xac\xac\xdd\xc3\x86\x19\x10\'\xa1\x12_\xb3\x81\xc52A\x1d\xff\xa6I\x1dGG\x9bym,f\xe8\xad\x13\xd2\xd9Qv\xb1x^l\xec7r\x8c\xa6\x9d+\x85LiG\x9eDT\xa0\x03#\x030\x18\xc5\xc89(YdG\x1a\x85\x1d\xb9"\xb9>\xfe-Nx\x94\xb9]6Gx>\x8d\x98\xef>=\xc5s|\x1aSs\xc196\xa3\xe8\xc9x\x03\xf7\xff\xdd\x18\xa5f\xaf\xb1\xff\xf8v\x94\x9b\xbc\xe8\x9a\x7f\x12\x14^\xa1a\x01c<b\x0e?7\rjy_\x0cj\x81/\xc91\xe1\xe1\xd8\xc6\xa8>-#\x82\x1d\xa0egP\xe6$\xa9r\x03\xcc\xb1\x81\x9c73\xe7\xcfV\x7f\x1f\xd7v\'36\x85\x08p.\x8d]\x1a\xe1\xc84P\x89od4N#\xbb\xe3\x8d\x0e\xa4t\xaa\xaag\xd2\x1e\xe6\xae\xdc\xaeN?\xaf=X\xad\x8355}\xb7v{C\x85\x9d\xe95\\\x05\x13L\x073\x88G\x8b\x18\x18\xf7\x9e%b\xaev\x06\xa1\x92\xcd\xe58_\x18\xa5\x9f\xa2j1\xf8\x106n|\x06\xdeX\x7f\xf3\xba\xfe\xf2\x8e\x0e\xd1\xdc*\x06\xc8\xc7\x7f\xf0M\x87\xa6*S1c\x8a\xa60I0\x87\x9d\xe0\\CJ\x15\n\x82\x01\xcc\x95-\x9a\xf3\x94\x07\xf0p<\xa3\xb0s\x05\xc3\x81\xb91P\x04h\x0c\xd8N\x87\x8bI\x91\x9e\xa8\tV\x1c\xf6@\xd1:6\xdaUv\xad\xba1W\xbdx\xce\x9d\xbeV]\xba\xd1\x18\xff\xc9]\xbfWY\x9f\x11-\xb2\x08E\xa98\x12\xech\x97)LN#\xa4\xf2\xc92z\x1b\x9a7\xaf@\xe1\xa9\xb2`\x14[f\x96\xd9\x9c\xdb\xe9=\x7f}\xe5\xbb\xea\xc2K\re:\x08\x95\xb5\x9f\x00AA~&,\xa9\xe3C\xe2I\xca\xcd\'<\x97\x068`\xd8\xd9\xfd\xfe\xfb\x9d]\x1f\xc5;\xe0\xff\x9d\xbbG\x0f\x1f:\x02\xd4;\xec\xfc\x0f&\x84\xdeT\xf7\xa8"\x891-\xf9\xf7A\xc0\xd2\xf9\x0c\xfb\xee\xa9\x01\x01\xa4n\x99F\xfd\x99)\x07gT\x89\xd9\x96\xc4\xd9\x18u\xcdX|*A\xd1\xe2\x06WR\x9c\x03\xe5\x8c\xd7\xa7\xce\xed\xf8\xe8R\x01%?<\xe7nN\x02\x877\x16\xbeA\xc3\x00\x13c7\xd8\xbe\x13:P\x8e\xef\xe4I\xa7\xb5\x99q\xb0%\x9a\x11\x11\xa7\x1eV\x82=\xda\x05\xb0[\xa7\xa8\xa1-z\xa8\xaf\xa7X\xcc\x17\xdb44)\x0c\xf7\xd0=\x98}o\xdeVp\xc3\xcb\xfa\x0c1oR\x14\x99@j[mG\xef\xd3\x17\x16-\xcd\x8c\x82\x7fqY\xb4\x89Lh<\xea\x93g\x02(\x1e\x109\\\x9aG\xa6/\xcb\x07\xd0\xe7f\xf1\xa4\xa0\xa0r\x01#\n\xe0\xca\xe2\x8b\xbd|\x08\x8f\x1d.\x99\xa8\xd7r\x86\x951\x0b`\xb1\xb4#F\xb0E\xad\x901\x05\xeeAOP\xf75\x12\x15\x87-pD\xd8\xa5\x97(N\x16V*k\xe3\x0c\xb0;\xfd-K\xda\xda\xf3o\xc0\x84\xe1\xfd\t%?fR\x7f\x14\x01^g#\xb6p\xf0\x1f\tQ\x9c?\x97\xc4\xb8;UZd28\xc7\x0c\'\xaa\xcaL\xcc\xfd\xfb\xd5\xef.\x82t\xab\xbc\x1ao\xaco\x88\xd3;@\xdd,}\x89m\xebs\x9b<\xa9\xb3\xb6\xb8\xe9N\x9f\x15S }\xed\xfb$\xee#\x8cq\xbeW\xb2\xcd,\x0b\x1e\xccN>\x9a\xe2%5\xae\\\xac\xae<C\'\xc3\xbf\xb60y\xddg\x89\xb1\x8eJ\xd0\tV\x97Mr\\x\xa7\x9c69\x15\x1b^\xc6T\x0bB\xa7\xb2~\x87\x8b\xeb\xbc\x89\xcc\xbe\xc9\xd7@G\x05\xf0G-==_\xc8;Z~^\xf4\x03\xb7\xd11C\xdd\xa3^/\xb3p\xa9#\x8a\x85\xe1\x9a\xc7\xce\x1e7wvu\xa8\x11z4-\xd0\xce\xb1u+\x97\xa5\x0e+ \x19\xc2\x83nA\xa6f\x80\xc9\xf1\x83"\xf7~\xb3\x08\x8b\xd3\xc6\xe6\x89&\x1b\xa2l\xcc\xfcNN\xbbW\xaf\xbb7\x1f\xcbf0\x1a_{\xff\x8e\xfb\xe8"\xc8~L\xa8\x11yUo_\xd3\xc2\x8dh\x8fH$\xc4\xf3\'\x83\xcc\x98\xc1\x80\xb3\x89c\xb8\xa9G\xda\x13\x90\x02\xe4\xa2i;r\xe8\x1f\x15\xf8\x14\xdbq\xcc\xbca!\xe3\x0b\xaa\'\x98X\xa0\x08\xd1\xff\xedRc\xe1\x1c\x9bU\xde\xdc\x08\xf5\xec\x10\xffc\xd8a\xa3\xadD$\xdf\xd2\x7f\x00\x94\x89\xde{\xa1\xd1=q\xac\xb5\xc8N\xcdT/m\xfad\xfdV\xe1#\x10oI\xc4B4 \x94UE\x93\xd7\xbc9\xc4!G\x94\xc0)\x14\x07\x19\xd9HA%L\x1e\x05\xca\xd44\x9b\x97\xbc\x8bB\xe0>\x99\x12\xa7 \xf8\xf2\x0e\x80d\xcb8\x82\xa3\xd0\xb3\x16\x89\xd4\xa8\xda2\x1c\xae\x7f|\xf7\x07\x1d\x1d\x1d\'\x14\x95e-\x1a\x0c\xa1\x9d\xdd\xd3\xbb\x0f\xb5E\x93j6p\x06$\xabr\x1e\xcb@=\x8a<\xdd\x9b\x81fF\x82\xdf{e0\x8f\xa6D\x00\x91\xed \xa2G\x8f\xb8^\xbc\xd0\x92O\xfcv6x\x14\xc4\\\xd6"O\xfaA\xbe\x11\x95>}\x988@\xb6\x91\x04%BHdq\xa4\x8c\xbf\xe6\xfb\xe3;\x84]\xba<\xcd\x93\xe6\xb9z\'\x00\t[\xad\x12\x06JG\xc0\xfb\x11w\xf1l\x1eh(\x9f\xb3\x93\xb2\xaeF\x8e\xf33\x90\x8a3\xda,N\x9d\xeb\xb8\xfb\x16\xee\xcc\xb0@%e\xe0\x90\xa3\n\x0c\xc9uu$\xbf\xbdQ\x9c\x8a\xd5P\xaf~9)\xa8|\xe9\x06f\xe8\xb5\xf9\xc7\xac\x05\x14\xba\xf0]\xddt\xe6X\xb4\x94/\x99\x99n18+e9\xc9\xeeH\xf5\xe2\xd7\x8d+\xe7\x7f\xde\x98\xf3\xaa\\\xae\xcfD\xb0\x03\xd1.uGDi\x06\xb0\xc0\x90\xa5KQE\xa7\xa6\x91\xa6\xcax\xd8h\xea\x98-Z\xc9|\x91\x8f`\xd3&gkV\x04Y\x10\xe4\x100-T\x97\xc6k\xcf\x847P{:\xde\xb8p\x06= 0,\x176D\xd9\x94f+(\x11\xc3#"=s\x92Gz\x17\xad\x02">\x85\x06c\x9ekB$\\j,\xbf\xc7\xa1E\x8b\xb31i\x92\xd1bv$\xc6\xb0\xbc[X\xe74n\xbe\xaa\xbd\xc4\x9a\x101ze\xe2"\xee\xc2\x8d\xbb\xf5\x7f\xdc\xd6\xe1\x96\xc0Q\xde[\xbc7A\xcb\x05P\xc1XcH\xf7a\xbc\x12\xc4I\xb1\\\x1aD\xd4P\xe0\xdd!=\xcaYF*\xc3\x96\xac$\xab\x98X\xe5\n\xea\xe1bo\xd9Cn\x88c\x8b\xa6\x1f1%\x88#\xcfH%\xa2%\xf0\xd3\xd78\xa6\x80\tc\xf1t\xe5\xd5\xac\xd7\xb4\xaf,/\xa2\x93A\x10s\x9c\x07\xc2ig\x91@\x11\x14\x06\xcdp\x94X\x84\xe9\xc4\x90\xa3\x0eq\xd6^D\x90+\xd7\x90P\xbdED\xef\xc6\xe3\xf9i\x11\xa6\xd1\xca\xdaC\xfc\x1d\xcd%\x8b\x884\xd6\x0f\xdf\xc3K\xe8*\xce\x14\x8bpNK^\xf5=h\x17\xfdj~Z\x95\xac\xd0\xaf>\xa0\xabg~\xc2P3_\x15s\xb4\xfel\xda%q\xda\x83\x17\xab\x13I\xb8\xd6\x94Kb\x8b\x05V\xf5\xc2Dm\xfd\x02\xdb\xb4\x9ea\x0f\xca\xe7\xf4\x13\xfdX%\xa6b15\xc6G\r\xcd.\x06\x1d\xc5\x06{\x89\xcc\xc1\xcd\xd9\x08\x9e\xe0\x966\xae\xfa\xe6\x81I\xac\xf0\xda\xa8\x96\x8b\xf5\xbe\x080\x12P\xf5\xc5\xcb\xb5\xf9\xb3\xb5\xabOa\xeb\x11\x1d\xe7o\xe0\x1ct\xb0\xban\xaf\x8bC\x11\xa6\x17\x1a\xe3\x9bBj\xf9\xcc\xbcA\xac\xc4Fvn\xa1\xf4>\x07\x95\x87\x1e\x00I\xa7a3\x93i\x07m\x85\xc5Uv\x96\x13\x9c\x18\xeb\xce\xa7\r\xac\x0b\xcd\xb2u\xc1\xe5\x92\x1e\x89\xaa\xd4\x04IU&M\xdd<\x13^\xe3\xa3)\xfe\x08\xe0\xd6.?\x17\x07>\x04\xad\xc2 x\xc2\x18\x0e\x91\xbf\x98)D\xd1\x1c\xbe\xac?\xd2LoTN\xed\xb2x\x02\xe4\xbf\xa3j\xc6D\xc0\xfeO\xe6\x00Jm\xc7r\x1c\xbf\x8e\'/\xa7q\xf9\x05\x83\xcc\xaa\xc1Sa\xe4\xfd\x08\x8a8s\x0bnf\xaa\xd9"\x83$\xd7\xf1\x89\xf1QWG\xc7[\x92D2.\xefW\xc0\xdc\x9eZ\xa2\xa3x\x928\x03z\xc4\x9f\x19\n(\\\x03\xa7\x95m^\x00\xea\x11\xf2K\r\x0b\xde\xea\xd5l*\x1ce\x12dS!\x1da\xd1\xc7\x04\x06\xc6\xb8R\xbc\xdd\xa3\xeaOL\xa6b=\x04\xbd\xca\x10\xa6\xaf\xe0\xb7\xf7\xb5\xdb"-\x8b\xa5\x93\x16\x85\xa75\xa9-D#O\x9e\x016m\x99d\xd7\xcb\xbdE\xdd\x02gU\xa8n\x81\xe3^\xdes1\xed\xcb\xd5c\x9a\x93\xe7\x05\x0b\xc2\xa0\x93\x90\xe8\x1eS\xf3\n\x80Q4\xcd.X/\xd4\x98\xc4\\\x03f\xba\x8cA\xe0\xba\xd2\xe0H0{\xa9!A\x1c\xdb@&&\xcf\x16g\xe0\xb7\x06x+\x937l\xdbIP\t\xc1cX\xd2\xb5\x16Qr\xd8\xc4B!x\x02\x9a\xfek\x0fZ\x86\x8d)\x05}\xc0\xa9s|\x8cD\xf3p\x9f0(<\x01\xba\xd5]~r\t\xc4h\xf2T\x9d\x8du\xe8X"l7\xc3JB\xf5\xe2\xdd\xea\xa5\xe7\xac\xa3\x84\xa15\xbd\xa0+S\xae\xd0\x0e\x82\xdb\n\x18\x8f:\x861\xd2\x9c\x1c\x8a\xc6\xfc\xab\x10\xf1\x8c\x90(\x86d4\x89\xf1m.\x7f\x0f\xd8\xb46r\x08J\xe2v\x0c\xc7\x92\xa3\x87j\x9d7\x9d&\x01\xd1\x94\x05\x9c\xe6\xef\x1da\xd2\x84\t>^\x88\xed\xc1\xa5\x1b \xea\xd8\x88`\xear\x1f\xff\xc8\xe4W\xfdr\xb2\xfad\x82\xe58g\xd0\xd9il\x85\xa1m-B\x05j\xa4\x9d\x88\x12Z\xe9hy\xfa\n\xd5\xee\xfbT\xa2/\x1b\xae\xd1\x1b\n\xbb\x97\x1b(\xef\xf8D@\x16\xcfJCo\x07\xa4\x82\xe98\xa1[\xf7n\xa1\xa8mP/\xcao\x8c\x00$-\xb0\x0c\xf5\xb3\xa4\x98\x8c\xb1\xdd ga\xa8\x05\xeci0\xd5R\xb6#\xa4\x87\xde\x9b\xa3\xb4\x95\x88S]iL|\xc3\xfc\x86\xe8\xb8z\xbd1>\x8e\xb6"\xa1\xa0\xbeq\x1f$\xb3\x1e\xc0\r\xe7\xc9w\xdaB\xddcF\x0f\x9f\xa2Y[\xb8\xce\n`r\xe0\x89\x86\xa6\xfd\xc1\xeff_z\xab\xf7\x07}\xeb\xb7\xdd\xbf\xcff\x10\xd3\xe5"\x15\xfcq\x1cT\xa5\x02u,\xabn\xcb\xe038\x08\xa7\x0e\xc8Q\x9e\xeb\xbbB\xbe\xa52\xa1\x12\x98\xff\xd9w\xe8\xa0\x81\xfe\xb9Ln\x00p\x14$T2\x0f{\x1b\xfe;\x1f\x9bl\r\xcbvF\x0c\xde\xf6\x83\xbd[lV+\xf4D\xb6\xf6\x05\x87cM\x0fB\xcf\xbd\\XE|s\xbe~\xef\xb5\xa8\x17?\xbd^Y\xbf\x007T^\xcf\xf2\r\xdb\xab\x91\x11\x02\xb1\r#\xe6\xb1\xb7h\x9e\xc3f\x11#DJ\xb9{\xa4@\x9c\x10(~\xa1\xf9\x85!\xf76k\xc3\xb0w1ouS$.\x8eR\xd3\x91\xa0\xc6\xb6\x06RU\xb9\xf2\x13\n\xe6\x08\xa5\xc0\x9a\x1aOT\xf9\xc4Ik\xc4\tN\xc3\xf3\xe4\x15C+QN\xd3y\xd9\xae\xa4F\x13\xae\xa3\xe0\x8c;\xb7\x9b47p\x86\x01I\x8d.\xdd\x02mT\xf8\x81W"1\xd4\x89\xa3co[ U\x8d*\xc2\x12Gc\xe5\x89K\x04Cc\xb5E\x0b\x03\x80\xb3\xdf\x8dk\xd7\xf5\xdc\x9e\xe2\x0c/G\xc3\xae\xe9\x8b\x17\x95\xf5\xd3o[\rP\x14\x82\xcf\x0b\xf1B=Td\xa5>nS\xderV\x0c\xec1\x9c\xdf\x88\xd1\x9c|\xda\x1f\x17\xa6\xd0\x15\x1e8\xdeTnE\x9bF\xf901\xbaT\x16\x1c\xa1\xd8R\xbfo\x1e9\xb5\x158\xe8\x0ex\x9bD)ZZVD\xc1\xc0\xe5\x0f\xde\xfa\xa9\xcc(L\xbb\xb4z\x05\x95\x9a\xc9sID\xe0\tM\x94\xa2,t\xa6:{\x04\x82\xc3\xc5^\x88?\x1e\xfa,\xfd\x94\x12Q3\'\xa3\x0ex\x8e\xcc\xe2\xd5\xea\x0f3"\xc7\x13\xe6\xd1m\x07\\\x0c^a"(\x81\xc1\xaa\xa2M\x9c\x14\xe5H\x05!\x01K\xd2\x10#\xc7\x19\x1b\'p<\x90\xda\r\x1c\x9aP\xb4\xe8\xe0\x9en>nu{\x94\x81\xc3\xb7\xf3iq\x00\x16E^d\x8aF)d\xd9\x82%\x83\tp-\x1bfR\x8a\xbe!\x8eV\xab \x0c5\x91Q\x07\x0ee\x08u\x97\xe3W\x91\x0bU<Iv\x8d\xf0!\x05\xf4w\x026\xba\xf4\xee\xc4\xb2\x17\xcb\x05\xec\xa4t`\xe8\xacvF\x88\x18\xd9\xc5\xdd\xcf\xdc\x9dB~)\xcb\xc0prqO?\xc1\n0\xcd\xb3\xaa]}\x8a-\xf4K\xcb,\xf5PwR\x14\xdb;\xc7\xfc\x9dhE\xca\xa9nb\x92w[j\xcf)1\x1fO\xac\x957\x16Y\xa0\x9d\x8a\xff\xd4\xc3Q\xbccJ\x8e+z\rn\xb4\x0c_0\x17~\x89\xbc\x15\x8f2\xe1\x1f\x81\xa6u\'\xd7\xc4\xec\xf0yQ\x9a\xf8\xcb\x16\xddRl\x04\xb6\xbe\xc5\xceo\xf5h\xa5\xe3\xc8q%g[\xca6\xec\xa8\x93\xa3\x86\xfd\xa3B\x9b6}\xee>\xab\xb7\xea\xc3;\x9ah\xbcr\x86yC\x96\xda\xb6"\xfd\xed\xc0\x18\xd8wO8:z7\x16\x9d7\xa0\x8a\x1d\xb7\xfb\xfc\xde\x81\x1c\xd2\xbc\xe7Of\xedR\x89\x8b\xe7\xa4\x83\x05\x1a0\x85\'\x1ehU\xde\xa1\x88P\xdaP[\xb8\xae\rq\xa0\xfc\xa4wB\xe4/\x16\x99@\x11(\x15U$\xce\xd3\x15mF\xf3ER\x9f1&\x19\x8c\xd2\xfe\n2\xd1\x92+"\xde/:}(\xeb\x108\xe6q+\x8a\xc16\x1a_\xac\xf5&\x96\xbf\xe2q\xc3\xa4L0\xcf\xbb\xf6ZOx\xfcR\xca\x11e\xdbY\xf3\x94\x9d-cB\xc1\x8f\x1f\xfa\x1a\x95\x88\xff2\x8d\xcd\xffexR\xed\x03\xf2\\j\x92\xa3Xg*2\x9f\xe0\xbb\xdb\xe9\x11\x99\xaa\xf2\xe5\xf9Cq59\xcdI\x12\x11\xb0 \xf3\x10k\x8ah01\x9fI\x8d1\xb8\xb3\xb3\x9c\x13\xaa]~\xd0\xb8v\xfb\x97\xa2+\xb4\xe6=j\xe7J\x98f\xcf\x9b\xf2P\x05\xdd\xe6g\xcc\x86\xdd%\x91\xfe\x89\xd1\xb1\x05\xd1\xbd\r$\xaa\xcb\xcbS\xcd\xab\xa6\x96A<\tBd\xd2C\xc0q\xack:\x8d\xa8V\x9e\x9d:\x19o\xab\xa7\x8b\xdeA\xbf&\xe7\xc1\x99L|\x88_\xea\xa6\x16*\x9d\xcd! \xd0\xad\xe9r;+#\xeb\x87\x97\xd0M\'\xe7\t\xa4\xc5~\x1d\xc2\x8e\x15T\xef<\xe6\x06\xe59\xe0*\x13ID\xd9f\xe4\xac!\n\xb6`\x9cE\xd4\xcb\x8ax\xbf\xb55\xca*\x9b\xd7\xaa\xdf\\\x11yI-#\xe9\x89\xba5\x91\xa1\xe7\xa4j\xf5\xe2re\xe3\xd1?\x05Y9q.f\x07p\xb5\x9dC\xaa\x13m\x18m\x1a2c\xbf\x12\x7f\xbdT\xd1\'\xfa\xd09\xbef&\x93e\xae\xa2\xdb\x1a5<\xa0\xab\xba\xbc\x88\x12\x1e\xf8\x94\xa3\x8dd5\xffS\xd6\x1f4\x95}9\xb7V\xb6p\xd8\x8ct\x0bM\x1f\x07[\xf3\xc1\xc9\xf3Rd\x03f1\x95AE\x08\xceRxA\xa16c\x05|\x80\x07\x0bz\xd4\x9a\x8f\xe3\xa5\xe0!\xba\x7f_\xbe\xd6\x8b\xc3\x94\xc4o\x15\xa3k\x91<i7B\x12\xae\x9ft\x1b\xbbv\xbf\xcd\xbb\xa5|\x1f\x1fT\xee`:8\xa9\xaf\xf5oe\xab8\xe2\x95#\xa6\xca\x98\x92\xe3\xe9\xb5\xa2\xa2\xc7\xf74\xa6\xf0\xca\xday\xf0t9\xdf\xa1\xd2\\"kLId\x9e\x88\xcdu\x8b\xd8f\xf6\xf8\xad.oh6Y\xac\xfd-]E\xe8\xea\x89Ue\xf2\x03\x8e\x1a\xd6/\xce\x8dN\xe1\x981\x1a\x05od\xad,\xae\x19\xc7`8M!a!\x01\xa9\xc2\xc9\xbdz\x13O\x1e8\xfb\xbc\xf1\xe0\x92\xfb\xfd7\xee\xf9\xab\xeaDhY\xc7;^\x7f\xf3\x15\xda5\x94\x824>\xcb\x0fl\xc3\xa9\xe7z!:oz\x9b\xde\xbb*@\x10\x0b\xe42!^\'\xe6\xc4L\n2R\\/e\x9b`\xc5\x81\xe4J\x86X\x1d\xaa\x08\x01\xa0\xe6%2\xe1\xa9<Yc\xf1\x0e,e\xbbnYHQM\xc4_#&\xf1u\xe5/\xb9\x88\xf1\x9e\xf1\xd9\xa1\x03q\\y\x02CzQ\xca\xf7:\xdd\x11\xf0>p\xf0q$v\xbc}\xd7\x07\x1d\x1d\xbbOl/\x14e;\xdeA\x13a\xc9\xed\xb0\xb0\x93\xef\xe4rY\xec\x12\x9e\xce\xder\x13\xb9"#|\x1aMX\xca\xcf+e\xfbPM\xa5\xa1_\xa9{b\xdbh\x9a\x93e\x1ey\xac\xe8D\xe2\xa6l\x81\xe9e\xf7\xc3\x16.\x04\xcc%\xaa\xde\xe0\xf2\x13Z#l8\x17\xddsX|\x1bt\x8b\xab\xde&\xc1\xca\x83\x92\xb8\xde\x83\xcf\xcbH\xca\x00S>\xc7\xa9~\xafh/cr\xc5\x92\xaa\xc2n\xb2K\x96\x96\xb9/\x87S\xff\x98\xd8\xe4\xf1\x7f\xca\x9f\x94\x87\x9f\x8b\x1cVh\xa7_+h\x054\xdd\xc6\xf1,\x17\xa4\xa0\x87\x8f\xabe\x13\x18}\xca\xb2\xcf\xbd\x92\x15+p\xed\xf8\t\x9a\xf6\x90=\xdeqB\x0f`\xe5s\tzj\xe4\xc4v\xd3\x08\xe4\x81s\xba\x1b\xd0\xe5\xc8Y\xf0\xa2:\x99b18R\xd4\xd6\xe6\xf7\xb4\xb0\xden\xcc\xe39\xc7\xdc\xcc$Kv\xd9\x92\xa3\xaa\xb3\x0b\xd5\xcbO\xc3J\xe5\xb6\x13\x84!D\xbdk\x10\xae\\L\x9b\x82`\xe9\x01\x9eM\xcac?`\x89b\x8e\xcfV\x9175\xc0\x03+\xf4iY\x9eW\xd9r\xb7\xb7\x82+Dz\xa1+\xc9k<\xde\xdey\xe2x\xe7\x89\x98W\x17\xb8\xad|\x9e\xb2\'XB+\xc9\xc4$$\xe9\x9d;w8\xec\xe89\x8fRz\x11\xe1\xb5H\xf3U6\xd6Q\xb0..36\xb8\xa6\xc9\x0b.\x9f?\x8f\x02\xf7\xe1m\xcc\xe52\x8a\xae\x9c\x11\xfc\x7f\xf99\xd6\xbc\x8a\xc2\x99\xedm\xb8<`n\x0bF\xe0+\t\xc0[$f\xfc7$\x7f"z}\x08\x81\xfe?\xafU\x9d\x85\xaf$\xcf+\xa3\xea%\xbf\xe1G\xfe\x86^2\xd6\x0c\xd4vwB\xa0\x95\xc39\x9cI\x95\xef\x96\x88N\xf2\x1c\x03g\x0b\xb1\xc39#\xd5\xccH\x82SH\x1e\xf2\xce\xab \x88\x9e\xcdW\xaf\xddh\xae~\xd9\n\xba~P\x82\'\xc3MR\xe1\xae\xd0\xc9\xe7%\xd2\xe8*\x1c\x83-\x8bTF)\x8dA*P\xe2\x147\xda\x02\xe8qRdFKz\x81\xdc\x9c\xa7\xa97\xb7\xaf\xa9\xcax\xe1\x1d\xfb\xcb\x93\x04\x05Q\xadU\xf5\xe9\xb7\xb5s\xd3\xad\xacQezc\x91w\xda>\x85\xbb\x8f\xdaOB\xf5\xbe\xf1a\xc7\xeexgz\xcc\xc0\xe3`\xe6\xbf\xdb\xa6-~\x84\xef\x11\x07}\x88A\x02<\x92\xa3\x9d\xec\x03_*\xcf\x9d~\x01kp/\xdeP\xabbO\x0b/\xde\x06\xb1\xf7Ccj\x0e\'EL\xac\x826\xd8r%\x02\xb6h\x0b\xb0\xb8\xd6\xb2\x84Gk)\xfd+LK\xad\xf9\xc9+k\xb1}}\r\x1e\xccZ\xa2N\x16\xf5c_\xd8\xf2,\xda D<\x9e\xf1<q\xd1]=\xdb\xac\x96C\xab[\xb6Q\xd9\xf2o4\x99g\xef\xe1cL0\\\x94\xa3\r.-\x96s\x8e\x8ci\xb3\xdc\n\x9aN\xdfxr\x87\x1a=\xb8N\x02\xd1M\xf4Ry}\t\xc3\xbd\xaar\x17^\xf5\xd6:\x17,es2\x96U\x88v\xc6B\x9aE\xf4\xb4\x83\x08\x0c\xa1(\xb5\x9dA+\xa5\xf34\x8d\x94*g2d\n\x11\x1f{\xb3;\x99{A\xdb\xa1\xc6\xd0j\xfe\x85\r\xb1y\x01G"<\x9aRu\xc6\xcaQ\xf5\xcao}]\x1a\xd2\xe5\xf6|\xeb@\x8f\x11\x1eE\xc0\x90\xb3\xb87\x1dO\xbc\xabfq\x11\xf0$\xc9\x8d3\xf5\xb6|]\x93\x87\xcb\xf7\xb6`)\xedTF\xc5J\xac\x7f|\r5"\x0c\'\xaa+\xd9\x95\xd3\xca\x8e\x91,\xaf\x9c\xa9\xad\xbd\xf6\x8a\xc0C\xe1\n\xb0\x7f\xb4Uubl\xbb\x12A\xbeAe\xed\x18v%\xff(\xb6/Z\xb0\xa4I\x80\x86\xfb\x16\x8dUA\x0e\xa7\'f\x825\xaa{x."\xdcey]\xe2Z\x0e$5`\x89\x13[4F\x97\xf4\x07<He\xf5\xd4\xf9z\xe6\x16p\x07#\x93\xa9Lib\x0e\x01p\xe3\x84\x92Yx|K\xab\xc2\x8f\xa6&*r\x01(7\'#\xa6\xcd\x95\xf6\x0c\xabj\x95BS\x80g\xbb(\xde\xf4\x8a5\x08\x85\xa1\x08\x0bm\xd6\xe0>\x16\xbf\xdc#\xab\xd4\x16\x83_\x1c\xef\\OM\xd8\xbd\xf2f\x1c\x8b\x92\x97\x95/+\xaf\xee2"0\x13\xb6>\xff\xcfhV\xe1>&\xd1\xa4X\xeeW\x1c\x13\xecY\xe1N&=-\xb7\xfd\xd6\x15\xbc\xfe\x19\x1f\x9b\x86\xaf\x14\x12\x14\xe4\x11\x1f\t\x02\x12T\x1a\n\xea\xbcJ\x1e\x97lS\x07\xe7\xe6u\xdfd\xe4\xa9u5<\x98G(\xf8|Sqb\x8e\x98\xceA\xc7\x9c\xe1\xc9\x03<\xc6\x86R\x9fjV2\xbf,\x12\x8b\x17\x072\xf9\xfeh\xe4w<\xa8\xae\x10\x18\xcb_\x88\x07GK\xc93\x9ac\xa2y\x84* I\xb6\xa2\xb8%\xdb\x83W&N\xd2\x835\xcc\xdd\x17\xb3\x84\xe7V\xab_?\x04|\xaaCJ\xbcC\xf0ZA,\x0ekC\xc8\xb6<<\x84\x0e\r\x91\xa7\x88\xc4\xd4\x08[y\xd0,9\x8b\xd2d\x13\x03\x9b\xf3\x02P<\xd0\xd01\x87P\xdar\x11_\xf3\xb8G\x8c\xd3\xdd\xbf\xe3N\xdf\xf1\t^|\x850w\tL:\x7f@S\x19|g\xe5\xf5\xd9\xea\xd2}\xdc\xad\xe9G\xf2t!\xddt\xe5\x9d\xf0lWu\xb8u \xec\x06\x18\xa6\xd4\x07\x86\xa4\xe8\xc4adh\x81j\x1b\x8d\x06\xac\x7f\x99[\xe0\x83\xa0\x99,\x98b\xb8-\x9eC\x16h\xe9\xf8\xa8\x84G\xe7\xf0T\x01\x11\x18\x16k9i\x8dtg\xccl\x7f\xca\xa4s\x1fw\xd3\xbfH\t8e\xc4)%\xb2(\xbae\xc7\xa09\xe4M\xc8m\xc7A\x1f4\xae\x92\t\x1a6J\x94\x8bcr\x93\xd3[\xf9"\x95L`m\xfb4\x96WscB}\xeaI\xf0Vo\x108\xbe\xe7\xd0\xb1\xa3\x87\x8f\x1dM\xec\xeb=\x02\xb0\x1e9t\xe8(\x92r\x164<\xc0\x9d\x18\xdc\x99\x10\xb8\x14H\xec\xc9Q\x85>\x1d7\xea\xe7\xad\x11%|\xc0.\x07\xe2\x0c\xb0\x96\x7f\xa2\x94\xf7\xd2mM\xe1\xf6\xf5\xfeh\t?F\xb1(E\xc3\xf1\xaa\x98N\xe7\xeayf\x0f\x9d\xec\x02g;\xbb\xe7V\xbd#\x02\xf1\x05\xbc\x9e\x04\xa6\x1c\x00\x13\x1aZh\xf89M\xa4\x17\x17\x0f\xee\xf9\xbc\x07\xf7\xcb\xca\x8e%\x9a[\xfa\x13\xa2\xa5\x7fT\xa7\x03\xed\x18H|\xd7\x01!\xb2\xb4vDq<\xaajr\xc0\xaa\xcc\xe1\xa2]\xe2\xb6I\x1b\x98W\xec\x04 X\xc4\x90\xee\xab\xc5\xf0J\x94\xe1Z\xbfw\xb6\xfe\xcd\x0c\xf6P\x9e\x9b\xf5\xd3\xa6\xd8\xb4nC[-\x83t\x90O`\xf6"\xce\xf2\xb8\x11\x1c\xde\x02\x1a\x16I\x02\xa3\x99\xa8\xb6i\xe4\x11\x9e\x08{z]?l\xd1\x7f\xe0\x99:\xf4\xc4\x1bn\xd1\xd5ft\x82\x17\x1fl\x93&\x85J\xb14<\xa0Z\xb5"{G\xa5`p\xb5\x9cKysnp\xa6\xfej`\xf5*\x8e\x16:\xf3\xa5$V\x1cg*m9\xf0\xcbGf\xb25C\x0cY\x10\xc8#\xa6\r\xcct\x97\x87I\xaf2DL\xfc[\x1fA\xa9\xf9\x9e\xfa\x18\x1d\xa5\xd6rXm\xa9N\xa6\x93\xc8\x90\xad8x\x9amem\x86\xc30\x8cx\xff\xeb\xe43\xc3v<\x8ej\x87T\x0e\x1a\x8c\xfa\x17\x82\xa8i\xdfvw\xecL\x8d\xf9\xbf\x15D\xac\xe6\x0f\x8a\x93\n\xbc\xd8\ts\xa4\xc8\xed\x89\xd9\xdeAAA2J\xb4`\xc9\xb3\ndbnR!\xcf/\xa5B\xe6\xde3c\xb5\x89\xc7\xfb\xdb\x929\xb3\xe8\x943\xa56\x9f\x93\xde\xc6\xa9b\xd5\x8d\x9f4\x93\x98\x8e\xb4\x06\xcd!\x1b{|\x859+\x0e\xf1\x99`\xc3\x1b\x0f\xa8\xa6\xdat\xf7\xcd\x03\xd4\x06\xfe\xfe51!\x84\x15\x13\x9d\x81C\xf0\x8c\xfd\xfcz\x89\x7fn\xbc\x93E\x1e\x94\xfe\xd8e\xe3\x94\xf4\xaeF^\x97\x9a\xb6,ms\xaa\xbb}\x1fG\x8dS=h\xd2\xcadX\x19\x90\xdab#X\xf4\r*S\x81\x9bw\x9f\xdd\xd3g\xf2r\xd1-NKX\xc3b|\xb9N:\xdc\x84\xe5\x9f\xa2\xa5\x16\x13b\x85\x15\xe6pn\xc5;\x9a\r(9\x95\xe2#\x11\xa4\x97\xab9\xbf\x0c\xb0\x88\xe9\xb0\xc0\xc6\xf9\xfcH\n\xd4\x9a\x89\x90\x1b{\xe1&\x03VT_\x7f\xa0\xba\xc2\x19`\xff~D\xfe\xf7\xd27\x06\x1ei\xdc\xd9\x89\x05\x91js\xd4\x83\xf9y?o\\\xad?\x98\x11\t\x07\xbc\xbb\x8b\xca\'\x9b0\x00\xbb\xf2\x7f\x003\xb1m%').decode('utf-8')
REMOTE_ALLOWED_TASKS = ['t2v', 'i2v', 'l2v', 'flf2v', 'r2v']
REMOTE_MAX_IMAGES = 9

'''
Prepare remote-only defaults for tasks not selected manually in the Notebook.
遠端 API 為尚未手動執行的 Task 建立預設參數，不影響 Notebook 既有設定。
'''
TASK_CONFIGS = globals().setdefault('TASK_CONFIGS', {})
for task_name in REMOTE_ALLOWED_TASKS:
    if task_name not in TASK_CONFIGS:
        defaults = dict(prompt='A cinematic landscape with ambient sound.',
                        width=384, height=224, seconds=5, steps=6 if REMOTE_SERVICE_ID.startswith('06_HyperFlow_') else 20,
                        seed=42, quant=QUANT, output_name=f'{task_name}.mp4',
                        first_image='', last_image='', ref_images=[], ref_videos=[], ref_audios=[],
                        video_audio=False, ref_image_size='match')
        if REMOTE_SERVICE_ID.startswith('06_MiniMax_'):
            defaults['audio_shift'] = 6.0
        else:
            defaults['encoder_variant'] = ENCODER_VARIANT
        TASK_CONFIGS[task_name] = defaults

REMOTE_ROOT.mkdir(parents=True, exist_ok=True)
REMOTE_JOBS = globals().setdefault('REMOTE_JOBS', {})
REMOTE_JOB_LOCK = globals().setdefault('REMOTE_JOB_LOCK', threading.Lock())
REMOTE_ENGINE_LOCK = globals().setdefault('GENERATION_LOCK', threading.Lock())
REMOTE_QUEUE = globals().setdefault('REMOTE_QUEUE', queue.Queue(maxsize=REMOTE_QUEUE_CAPACITY))
assert REMOTE_QUEUE.maxsize == REMOTE_QUEUE_CAPACITY, 'FIFO 容量已變更，請先重新啟動 Kernel 及遠端服務。'
REMOTE_UPLOAD_SLOTS = globals().setdefault('REMOTE_UPLOAD_SLOTS', threading.BoundedSemaphore(REMOTE_UPLOAD_CONCURRENCY))
REMOTE_RESULT_TTL = REMOTE_RESULT_TTL_HOURS * 3600
REMOTE_RESULT_BYTES = REMOTE_RESULT_LIMIT_GIB * 1024**3
REMOTE_MIN_FREE_BYTES = REMOTE_MIN_FREE_GIB * 1024**3

'''
Do not start a second GPU job when a long-running backend appears stalled.
當底層生成疑似卡住時，停止接收新工作，絕不啟動第二筆 GPU 推論。
'''
REMOTE_STALL_SECONDS = REMOTE_STALL_HOURS * 3600

'''
Verify the CUDA backend and existing model without loading another model.
確認 CUDA 與已啟動的模型後端，不另外載入第二份模型。
'''
def remote_gpu_status():
    try:
        import torch
        if not torch.cuda.is_available():
            return False, 'Kaggle CUDA GPU 不可用；請啟用 GPU 並重新執行模型初始化 Cell。', []
        qwen = REMOTE_KIND == 'image_multi'
        yue = REMOTE_SERVICE_ID.startswith('06_YuE2_')
        if yue:
            selected_gpus = [globals().get('GPU_ID', 0)]
        elif qwen and globals().get('QWEN21_STATE', {}).get('mode') != 'bf16_dual':
            selected_gpus = [0]
        else:
            selected_gpus = [0, 1]
        available = torch.cuda.device_count()
        if any(not isinstance(index, int) or index < 0 or index >= available for index in selected_gpus):
            return False, f'需要 cuda:{selected_gpus}，但實際只有 {available} 張 CUDA GPU。', []
        devices = [f'cuda:{index} {torch.cuda.get_device_name(index)}' for index in selected_gpus]
        if qwen:
            state = globals().get('QWEN21_STATE', {})
            if state.get('pipeline') is None or state.get('manager') is None:
                return False, 'Qwen Pipeline／GPU 記憶體管理器尚未初始化，請先執行模型初始化 Cell。', devices
            if state.get('device') != 'cuda:0':
                return False, 'Qwen Transformer 沒有設定為 cuda:0。', devices
        elif REMOTE_SERVICE_ID.startswith('06_HeartMuLa_'):
            pipeline = globals().get('pipe')
            if pipeline is None or pipeline.lazy_load or pipeline._mula is None or pipeline._codec is None:
                return False, 'HeartMuLa 雙 GPU Pipeline 尚未完整載入，請先執行模型載入 Cell。', devices
            for name, module, index in (('HeartMuLa', pipeline.mula, 0), ('HeartCodec', pipeline.codec, 1)):
                parameter = next(module.parameters(), None)
                if parameter is None or parameter.device != torch.device(f'cuda:{index}'):
                    return False, f'{name} 未位於指定的 cuda:{index}，請重新載入模型。', devices
        elif yue:
            gpu_id = globals().get('GPU_ID', 0)
            process = globals().get('SERVER_PROCS', {}).get(gpu_id)
            if process is None or process.poll() is not None:
                return False, f'YuE2 CUDA{gpu_id} 常駐服務尚未啟動。', devices
            if f'CUDA{gpu_id}' not in [str(value) for value in process.args]:
                return False, 'YuE2 常駐服務沒有使用預期的 CUDA 啟動參數。', devices
        else:
            base = globals().get('BASE')
            if not base and REMOTE_IS_LTX:
                base = f"http://127.0.0.1:{globals().get('COMFY_PORT', 8188)}"
            if not isinstance(base, str) or not base:
                return False, 'ComfyUI 尚未啟動，請先執行模型服務 Cell。', devices
            with urllib.request.urlopen(base.rstrip('/') + '/system_stats', timeout=4) as response:
                stats = json.load(response)
            comfy_devices = stats.get('devices', [])
            if not any(str(item.get('type', '')).lower().startswith('cuda') or
                       str(item.get('name', '')).lower().startswith('cuda:')
                       for item in comfy_devices if isinstance(item, dict)):
                return False, 'ComfyUI 沒有回報 CUDA 裝置，已拒絕 CPU-only 推論。', devices
        return True, 'CUDA 後端已就緒（不代表所有權重常駐 GPU 記憶體）。', devices
    except Exception as error:
        return False, f'GPU 檢查失敗：{type(error).__name__}: {error}', []

'''
Verify every global required by the selected Notebook generation engine.
驗證所選模型的生成程式所需變數與設定已完成初始化。
'''
def remote_inference_ready(task):
    namespace = globals()
    common = {
        '06_HeartMuLa_': ('ROOT', 'pipe'),
        '06_YuE2_': ('ROOT', 'GPU_ID', 'PORTS', 'SERVER_PROCS'),
        '06_Z-Image-Turbo_': ('ROOT', 'BASE', 'LOG_PATH'),
        '06_Qwen_Image_': ('MODEL_MODE', 'GPU0_MODEL_GIB', 'GPU_ID', 'QWEN21_STATE'),
        '06_LTX-2.5_': ('ROOT', 'ENGINE_PATH', 'COMFY_PORT', 'COMFY_LOG', 'COMFY_PROC',
                          'ltx_available_ram_gib', 'LTX_MIN_AVAILABLE_RAM_GIB', 'importlib'),
        '06_MiniMax_H3_': ('ROOT', 'BASE', 'COMFY', 'LOG', 'PORT', 'TASK_CONFIGS',
                              'H3_DESIRED_CONFIG', 'H3_ACTIVE_CONFIG', 'H3_MODEL_CHOICE',
                              'H3_ACTIVE_MODEL', 'DIT_NAME', 'TEXT_MODE', 'TEXT_GPU',
                              'QUANT_SIZES_GIB', 'ready', 'choices', 'check_choice'),
        '06_HyperFlow_H3_': ('ROOT', 'BASE', 'COMFY', 'LOG', 'PORT', 'TASK_CONFIGS',
                                'H3_DESIRED_CONFIG', 'H3_ACTIVE_CONFIG', 'H3_MODEL_CHOICE',
                                'H3_ACTIVE_MODEL', 'DIT_NAME', 'TEXT_MODE', 'TEXT_GPU',
                                'QUANT_SIZES_GB', 'ready', 'choices', 'ENCODERS',
                                'ENCODER_FILE', 'MODEL_FAMILY', 'WEIGHTS_IN',
                                'TEXT_LOADER_NODE', 'PROJECTION_FILE'),
    }
    required = next((names for prefix, names in common.items()
                     if REMOTE_SERVICE_ID.startswith(prefix)), ())
    missing = [name for name in required if name not in namespace or namespace[name] is None]
    if missing:
        return False, 'Notebook 尚未完成初始化，請先執行對應的模型／服務 Cells；缺少：' + ', '.join(missing)
    if task not in REMOTE_ALLOWED_TASKS:
        return False, '此模型不支援生成模式：' + task
    if REMOTE_SERVICE_ID.startswith(('06_MiniMax_H3_', '06_HyperFlow_H3_')):
        config = namespace['TASK_CONFIGS'].get(task)
        if not isinstance(config, dict):
            return False, f'{task.upper()} 尚未設定；請先執行 Notebook 對應的模式設定 Cell。'
        if namespace['H3_ACTIVE_CONFIG'] != namespace['H3_DESIRED_CONFIG']:
            return False, 'GPU／Offload 設定與已啟動 ComfyUI 不一致，請重新啟動模型服務 Cell。'
        quant = config.get('quant')
        if REMOTE_SERVICE_ID.startswith('06_MiniMax_H3_'):
            variant = 'ref2va_pruned' if task == 'r2v' else 'fl2va_pruned'
            expected = (variant, quant)
            expected_file = f'minimax_h3_{variant}-{quant}.gguf'
        else:
            variant = 'ref2va' if task == 'r2v' else 'fl2va'
            encoder = config.get('encoder_variant')
            expected = (variant, quant, encoder)
            expected_file = f'minimax_h3_{variant}_hyperflow_flash_14b-{quant}.gguf'
            if encoder not in namespace['ENCODERS']:
                return False, f'{task.upper()} 的 Encoder 設定無效，請重新執行模式設定 Cell。'
            if Path(namespace['ENCODER_FILE']).name != namespace['ENCODERS'][encoder][1]:
                return False, 'GPU 1 Encoder 與此模式不一致，請重新執行模型下載／啟動 Cells。'
        if namespace['H3_MODEL_CHOICE'] != expected or namespace['H3_ACTIVE_MODEL'] != expected:
            return False, f'{task.upper()} 尚未載入相應模型權重；請設定模式並重新執行模型下載／啟動 Cells。'
        if namespace['DIT_NAME'] != expected_file:
            return False, 'ComfyUI 使用的模型檔名與任務不一致，請重新載入正確模型。'
    elif REMOTE_SERVICE_ID.startswith('06_Qwen_Image_'):
        state = namespace['QWEN21_STATE']
        if state.get('mode') != namespace['MODEL_MODE']:
            return False, 'Qwen MODEL_MODE 與已初始化的模型模式不同，請重新執行模型初始化 Cell。'
        if state.get('gpu0_model_gib') != namespace['GPU0_MODEL_GIB']:
            return False, 'Qwen GPU0_MODEL_GIB 與目前載入設定不同，請重新初始化 GPU 記憶體管理器。'
    elif REMOTE_IS_LTX:
        if not Path(namespace['ENGINE_PATH']).is_file():
            return False, 'LTX 模型引擎檔案不存在；請先執行模型與 Workflow Cell。'
        if namespace['COMFY_PROC'].poll() is not None:
            return False, 'LTX ComfyUI 行程已停止，請重新執行啟動 ComfyUI Cell。'
    return True, '模型初始化與所選模式已就緒。'

'''
Validate client parameters before they occupy a FIFO slot.
在進入 FIFO 前檢查 client 參數，避免排隊後才因設定錯誤而失敗。
'''
def remote_validate_fields(task, fields):
    required = {'task'}
    if REMOTE_SERVICE_ID.startswith('06_HeartMuLa_'):
        required.update(('lyrics', 'tags', 'duration', 'seed'))
    elif REMOTE_SERVICE_ID.startswith('06_YuE2_'):
        required.update(('style', 'lyrics', 'duration', 'steps', 'lm_seed', 'seed', 'cot'))
    elif REMOTE_KIND == 'image_text':
        required.update(('prompt', 'width', 'height', 'steps', 'seed', 'cfg'))
    elif REMOTE_KIND == 'image_multi':
        required.update(('prompt', 'negative_prompt', 'sampler', 'reference_size',
                         'width', 'height', 'steps', 'seed', 'cfg'))
    elif REMOTE_IS_LTX:
        required.update(('prompt', 'width', 'height', 'seconds', 'seed', 'image_strength'))
    else:
        required.update(('prompt', 'width', 'height', 'seconds', 'steps', 'seed', 'ref_image_size'))
        if REMOTE_SERVICE_ID.startswith('06_MiniMax_H3_'):
            required.add('audio_shift')
    missing = required - fields.keys()
    if missing:
        raise ValueError('請求缺少必要參數：' + ', '.join(sorted(missing)))
    integer_fields = ('seed', 'lm_seed', 'width', 'height', 'steps', 'duration', 'reference_size')
    float_fields = ('seconds', 'cfg', 'image_strength', 'audio_shift')
    numbers = {}
    for key in integer_fields:
        if key in fields:
            raw = fields[key]
            if isinstance(raw, bool) or str(raw).strip() != str(int(raw)):
                raise ValueError(f'{key} 必須是整數')
            numbers[key] = int(raw)
    for key in float_fields:
        if key in fields:
            numbers[key] = float(fields[key])
            if not math.isfinite(numbers[key]):
                raise ValueError(f'{key} 必須是有限數值')
    for key in ('seed', 'lm_seed'):
        if key in numbers and not -1 <= numbers[key] < 2**32:
            raise ValueError(f'{key} 必須介於 -1 至 4294967295')
    for key in ('width', 'height'):
        if key in numbers:
            size = numbers[key]
            divisor = 64 if REMOTE_KIND == 'image_text' else 32
            if size < 1 or size > 4096 or size % divisor:
                raise ValueError(f'{key} 必須介於 1～4096 且為 {divisor} 的倍數')
    for key in ('seconds', 'duration'):
        if key in numbers and not 0 < numbers[key] <= 3600:
            raise ValueError(f'{key} 必須大於 0 且不超過 3600')
    if 'steps' in numbers and not 1 <= numbers['steps'] <= 200:
        raise ValueError('steps 必須介於 1～200')
    if REMOTE_SERVICE_ID.startswith('06_HyperFlow_H3_') and numbers.get('steps') not in (4, 6, 8):
        raise ValueError('HyperFlow steps 只支援 4、6、8')
    if 'image_strength' in numbers and not 0 <= numbers['image_strength'] <= 1:
        raise ValueError('image_strength 必須介於 0～1')
    if 'reference_size' in numbers and not 64 <= numbers['reference_size'] <= 2048:
        raise ValueError('reference_size 必須介於 64～2048')
    if 'audio_shift' in numbers and not 0.01 <= numbers['audio_shift'] <= 100:
        raise ValueError('audio_shift 必須介於 0.01～100')
    if fields.get('ref_image_size', 'match') not in ('match', 'max'):
        raise ValueError('ref_image_size 只支援 match 或 max')
    if REMOTE_KIND in ('video', 'image_text', 'image_multi') and not str(fields.get('prompt', '')).strip():
        raise ValueError('PROMPT 不可為空白')
    if task == 'r2v':
        image_count = int(fields.get('_image_count', 0))
        video_count = int(fields.get('_video_count', 0))
        audio_count = int(fields.get('_audio_count', 0))
        if not (image_count or video_count):
            raise ValueError('R2V 至少需要參考圖片或影片，不能只提供音訊。')
        if image_count > 9 or video_count > 3 or audio_count > 3 or image_count + video_count + audio_count > 12:
            raise ValueError('R2V 最多 9 張圖片、3 段影片、3 段音訊，合計最多 12 個素材。')
        count_by_type = {'Picture': image_count, 'Video': video_count,
                         'Audio': audio_count + (video_count if str(fields.get('video_audio', 'false')).lower() == 'true' else 0)}
        for media, count in count_by_type.items():
            matches = [int(number) for number in re.findall(r'<%s\s+(\d+)>' % media, str(fields['prompt']), re.I)]
            if any(number < 1 or number > count for number in matches):
                raise ValueError(f'R2V Prompt 的 <{media} N> 編號超出已上傳的參考素材數量。')

'''
Report whether the model is ready for CUDA inference.
回報模型是否已具備 CUDA 推論條件。
'''
def remote_backend_ready():
    return remote_gpu_status()[0]

'''
A Python thread cannot safely terminate an already running CUDA inference.
Python thread 無法安全強制終止正在執行的 CUDA 推論。
'''
def remote_stalled():
    with REMOTE_JOB_LOCK:
        return any(record.get('status') == 'running' and
                   time.monotonic() - record.get('_start_clock', time.monotonic()) > REMOTE_STALL_SECONDS
                   for record in REMOTE_JOBS.values())

'''
Remove obsolete input files once the model no longer needs them.
模型不再需要輸入檔案時立即移除，避免參考圖片長期佔用空間。
'''
def remote_remove_inputs(job_id):
    folder = REMOTE_ROOT / job_id
    for file in folder.glob('*'):
        if file.name.startswith('input.') or (file.stem.isdecimal() and file.suffix.lower() in {'.png', '.jpg', '.jpeg', '.webp'}):
            file.unlink(missing_ok=True)

'''
Keep only finished results; never delete active jobs or in-progress downloads.
僅清理已結束工作，絕不刪除排隊、推論或正在下載的檔案。
'''
def remote_cleanup():
    now = time.time()
    with REMOTE_JOB_LOCK:
        finished = [(job_id, record) for job_id, record in REMOTE_JOBS.items()
                    if record.get('status') in ('done', 'error', 'expired') and
                    not record.get('_downloads')]
        finished.sort(key=lambda item: item[1].get('finished_at', now))
        sizes = {}
        for job_id, record in finished:
            folder = REMOTE_ROOT / job_id
            sizes[job_id] = sum(path.stat().st_size for path in folder.glob('**/*') if path.is_file()) if folder.exists() else 0
        total = sum(sizes.values())
        for job_id, record in finished:
            old = now - record.get('finished_at', now) > REMOTE_RESULT_TTL
            if old or total > REMOTE_RESULT_BYTES:
                shutil.rmtree(REMOTE_ROOT / job_id, ignore_errors=True)
                total -= sizes[job_id]
                record.update(status='expired', error='結果已過期或因儲存空間不足而清除，請重新提交。')
                record.pop('_path', None)
                record.pop('_last_frame', None)
            if record['status'] == 'expired' and now - record.get('finished_at', now) > 2 * REMOTE_RESULT_TTL:
                REMOTE_JOBS.pop(job_id, None)

'''
Sweep completed records periodically without interfering with inference.
定期整理已完成資料，不干擾單一 GPU worker 的推論。
'''
def remote_maintenance():
    while True:
        try:
            remote_cleanup()
        except Exception as error:
            print('API cleanup warning:', type(error).__name__, str(error)[:200])
        time.sleep(300)

'''
Serve only the generation API without exposing notebook controls.
只提供生成 API，不公開 Notebook 的管理功能。
'''
class RemoteHandler(BaseHTTPRequestHandler):
    def log_message(self, fmt, *args):
        return

    def send_json(self, code, payload, retry_after=None):
        data = json.dumps(payload, ensure_ascii=False).encode('utf-8')
        self.send_response(code)
        self.send_header('Content-Type', 'application/json; charset=utf-8')
        self.send_header('Content-Length', str(len(data)))
        if retry_after is not None:
            self.send_header('Retry-After', str(retry_after))
        self.end_headers()
        self.wfile.write(data)

    def do_GET(self):
        path = urlsplit(self.path).path
        if path == '/health':
            with REMOTE_JOB_LOCK:
                running = sum(record.get('status') == 'running' for record in REMOTE_JOBS.values())
            gpu_ready, gpu_message, gpu_devices = remote_gpu_status()
            requested_task = parse_qs(urlsplit(self.path).query).get('task', [REMOTE_ALLOWED_TASKS[0]])[-1]
            inference_ready, inference_message = remote_inference_ready(requested_task)
            if gpu_ready and not inference_ready:
                gpu_message = inference_message
            self.send_json(200, {'ok': True, 'api': 'kaggle-generation-bridge-v3',
                                 'service_id': REMOTE_SERVICE_ID, 'task': requested_task,
                                 'backend_ready': gpu_ready and inference_ready, 'gpu_ready': gpu_ready,
                                 'gpu_message': gpu_message, 'gpu_devices': gpu_devices,
                                 'stalled': remote_stalled(), 'running': running,
                                 'waiting': REMOTE_QUEUE.qsize(), 'queue_capacity': REMOTE_QUEUE.maxsize})
            return
        if path == '/schema':
            self.send_json(200, {'kind': REMOTE_KIND, 'tasks': REMOTE_ALLOWED_TASKS,
                                 'fields': REMOTE_FIELDS, 'image_limit': REMOTE_MAX_IMAGES, 'video_limit': 3, 'audio_limit': 3,
                                 'upload_field': 'images' if REMOTE_MAX_IMAGES else None,
                                 'queue_capacity': REMOTE_QUEUE.maxsize,
                                 'model_switching': 'configure in notebook first'})
            return
        pieces = path.strip('/').split('/')
        if pieces[0] != 'jobs' or len(pieces) not in (2, 3):
            self.send_json(404, {'error': 'Not found'})
            return
        job_id = pieces[1]
        with REMOTE_JOB_LOCK:
            record = REMOTE_JOBS.get(job_id)
            item = dict(record) if record else None
            if item and item['status'] == 'running':
                item['stalled'] = time.monotonic() - item.get('_start_clock', time.monotonic()) > REMOTE_STALL_SECONDS
            if item and item['status'] == 'queued':
                waiting = [key for key, value in REMOTE_JOBS.items() if value.get('status') == 'queued']
                if job_id in waiting:
                    item['queue_position'] = waiting.index(job_id) + 1
            if item and len(pieces) == 3 and pieces[2] in ('file', 'last-frame') and item['status'] == 'done':
                record['_downloads'] = record.get('_downloads', 0) + 1
                downloading = True
            else:
                downloading = False
        if item is None:
            self.send_json(404, {'error': 'Unknown job'})
            return
        if len(pieces) == 2:
            for field in tuple(item):
                if field.startswith('_'):
                    item.pop(field)
            self.send_json(200, item)
            return
        if not downloading:
            self.send_json(404, {'error': 'Result not available for this job'})
            return
        try:
            file_path = item.get('_path') if pieces[2] == 'file' else item.get('_last_frame')
            if not file_path or not Path(file_path).is_file():
                self.send_json(404, {'error': 'Output is no longer available'})
                return
            result = Path(file_path)
            self.send_response(200)
            self.send_header('Content-Type', 'application/octet-stream')
            self.send_header('Content-Disposition', f'attachment; filename="{result.name}"')
            self.send_header('Content-Length', str(result.stat().st_size))
            self.end_headers()
            with result.open('rb') as stream:
                shutil.copyfileobj(stream, self.wfile, length=1024 * 1024)
        except (BrokenPipeError, ConnectionResetError):
            pass
        finally:
            with REMOTE_JOB_LOCK:
                record['_downloads'] -= 1

    def do_POST(self):
        if urlsplit(self.path).path != '/generate':
            self.send_json(404, {'error': 'Not found'})
            return
        supplied_id = self.headers.get('X-Request-ID', '').strip()
        try:
            job_id = uuid.UUID(supplied_id).hex if supplied_id else uuid.uuid4().hex
        except ValueError:
            self.send_json(400, {'error': 'X-Request-ID must be a UUID'})
            return
        with REMOTE_JOB_LOCK:
            existing = REMOTE_JOBS.get(job_id)
        if existing:
            self.send_json(202, {'job_id': job_id, 'status_url': f'/jobs/{job_id}',
                                 'download_url': f'/jobs/{job_id}/file', 'reused': True})
            return
        if remote_stalled():
            self.send_json(503, {'error': 'GPU worker appears stalled; inspect the Notebook before accepting more jobs'}, 60)
            return
        gpu_ready, gpu_message, _ = remote_gpu_status()
        if not gpu_ready:
            self.send_json(409, {'error': gpu_message})
            return
        if shutil.disk_usage(REMOTE_ROOT).free < REMOTE_MIN_FREE_BYTES:
            self.send_json(507, {'error': 'Kaggle disk space is low; clean old outputs before retrying'})
            return
        if not REMOTE_UPLOAD_SLOTS.acquire(timeout=1):
            self.send_json(503, {'error': 'Upload slots busy; retry later'}, 5)
            return
        staging = None
        try:
            self.connection.settimeout(120)
            length = int(self.headers.get('Content-Length', '0'))
            if not 0 < length <= 120 * 1024 * 1024:
                raise ValueError('Request must be between 1 byte and 120 MiB')
            body = self.rfile.read(length)
            if len(body) != length:
                raise ValueError('Incomplete request body')
            content_type = self.headers.get('Content-Type', '')
            fields, files, videos, audios = {}, [], [], []
            if content_type.startswith('application/json'):
                fields = json.loads(body)
            elif content_type.startswith('application/x-www-form-urlencoded'):
                fields = {key: values[-1] for key, values in parse_qs(body.decode('utf-8')).items()}
            elif content_type.startswith('multipart/form-data'):
                message = BytesParser(policy=policy.default).parsebytes(
                    b'Content-Type: ' + content_type.encode() + b'\r\nMIME-Version: 1.0\r\n\r\n' + body)
                for part in message.iter_parts():
                    name = part.get_param('name', header='content-disposition')
                    data = part.get_payload(decode=True)
                    if name in ('images', 'videos', 'audios'):
                        collection = {'images': files, 'videos': videos, 'audios': audios}[name]
                        collection.append((part.get_filename() or 'upload', data))
                    elif name:
                        fields[name] = data.decode('utf-8')
            else:
                raise ValueError('Use application/json, form data, or multipart/form-data')
            if not isinstance(fields, dict):
                raise ValueError('Parameters must be a JSON object')
            if 'abc' in fields and len(str(fields['abc']).encode('utf-8')) > 1024 * 1024:
                raise ValueError('ABC score must be 1 MiB or smaller')
            task = str(fields.get('task', REMOTE_ALLOWED_TASKS[0])).lower()
            if task not in REMOTE_ALLOWED_TASKS:
                raise ValueError('Unsupported task: ' + task)
            if len(files) > REMOTE_MAX_IMAGES or len(videos) > 3 or len(audios) > 3:
                raise ValueError(f'Only {REMOTE_MAX_IMAGES} input images are supported')
            if REMOTE_KIND in ('music', 'image_text') and (files or videos or audios):
                raise ValueError('This model does not accept reference images')
            if REMOTE_KIND == 'video':
                minimum, maximum = {'t2v': (0, 0), 'i2v': (1, 1), 'l2v': (1, 1), 'flf2v': (2, 2),
                                    'r2v': (0, REMOTE_MAX_IMAGES)}[task] if task == 'r2v' else {
                                    't2v': (0, 0), 'i2v': (1, 1), 'l2v': (1, 1), 'flf2v': (2, 2)}[task]
                if not minimum <= len(files) <= maximum:
                    raise ValueError('Input image count does not match the selected video task')
            if task != 'r2v' and (videos or audios):
                raise ValueError('Reference videos/audio are supported only for R2V')
            if task == 'r2v' and not (files or videos):
                raise ValueError('R2V requires an image or a video; audio alone is not supported')
            if REMOTE_KIND == 'image_multi':
                minimum, maximum = {'text': (0, 0), 'edit': (1, 1), 'multi': (2, REMOTE_MAX_IMAGES)}[task]
                if not minimum <= len(files) <= maximum:
                    raise ValueError(f'Task {task} requires {minimum}–{maximum} images')
            for key in fields:
                if key not in REMOTE_FIELDS:
                    raise ValueError('Unsupported parameter: ' + key)
            remote_validate_fields(task, dict(fields, _image_count=len(files), _video_count=len(videos), _audio_count=len(audios)))
            if 'quant' in fields and fields['quant']:
                active_model = globals().get('H3_ACTIVE_MODEL', ())
                if fields['quant'] not in active_model:
                    self.send_json(409, {'error': '指定的量化尚未載入；請先在 Notebook 下載並預檢該 GGUF。'})
                    return
            inference_ready, inference_message = remote_inference_ready(task)
            if not inference_ready:
                self.send_json(409, {'error': inference_message})
                return
            for filename, data in files:
                if not isinstance(data, bytes) or len(data) > 20 * 1024 * 1024:
                    raise ValueError('Each image must be 20 MiB or smaller')
                if Path(filename).suffix.lower() not in {'.png', '.jpg', '.jpeg', '.webp'}:
                    raise ValueError('Only PNG/JPEG/WebP images are supported')
                from PIL import Image
                with Image.open(io.BytesIO(data)) as photo:
                    photo.verify()
            if str(fields.get('video_audio', 'false')).lower() not in ('true', 'false'):
                raise ValueError('video_audio must be true or false')
            for filename, data in videos:
                if not data or len(data) > 80 * 1024 * 1024 or Path(filename).suffix.lower() not in ('.mp4', '.mov', '.webm'):
                    raise ValueError('Reference video must be MP4/MOV/WebM and at most 80 MiB')
            for filename, data in audios:
                if not data or len(data) > 30 * 1024 * 1024 or Path(filename).suffix.lower() not in ('.mp3', '.wav', '.flac', '.ogg', '.m4a'):
                    raise ValueError('Reference audio must be an audio file at most 30 MiB')
            staging = Path(tempfile.mkdtemp(prefix='pending_', dir=REMOTE_ROOT))
            image_names, video_names, audio_names = [], [], []
            for index, (filename, data) in enumerate(files, 1):
                target = staging / f'{index:03d}{Path(filename).suffix.lower()}'
                target.write_bytes(data)
                image_names.append(target.name)
            for kind, uploaded, names in (('video', videos, video_names), ('audio', audios, audio_names)):
                for index, (filename, data) in enumerate(uploaded, 1):
                    target = staging / f'{kind}_{index:03d}{Path(filename).suffix.lower()}'
                    target.write_bytes(data)
                    names.append(target.name)

            '''
            Commit the validated upload and queue entry in one critical section.
            在同一個鎖定區段確認請求、保存資料並加入工作佇列。
            '''
            with REMOTE_JOB_LOCK:
                existing = REMOTE_JOBS.get(job_id)
                full = REMOTE_QUEUE.full()
                if not existing and not full:
                    folder = REMOTE_ROOT / job_id
                    staging.rename(folder)
                    staging = None
                    image_paths = [str(folder / name) for name in image_names]
                    fields['_ref_videos'] = [str(folder / name) for name in video_names]
                    fields['_ref_audios'] = [str(folder / name) for name in audio_names]
                    REMOTE_JOBS[job_id] = {'id': job_id, 'task': task, 'status': 'queued',
                                           'created_at': time.time()}
                    REMOTE_QUEUE.put_nowait((job_id, fields, image_paths))
                    position = REMOTE_QUEUE.qsize()
            if existing:
                self.send_json(202, {'job_id': job_id, 'status_url': f'/jobs/{job_id}',
                                     'download_url': f'/jobs/{job_id}/file', 'reused': True})
            elif full:
                self.send_json(429, {'error': 'Queue is full; retry later'}, 15)
            else:
                self.send_json(202, {'job_id': job_id, 'status_url': f'/jobs/{job_id}',
                                     'download_url': f'/jobs/{job_id}/file', 'queue_position': position})
        except (ValueError, KeyError, TypeError, json.JSONDecodeError) as error:
            self.send_json(400, {'error': str(error)})
        except (TimeoutError, ConnectionError, OSError) as error:
            self.send_json(503, {'error': f'Upload temporarily failed: {type(error).__name__}'}, 10)
        except Exception as error:
            self.send_json(500, {'error': type(error).__name__ + ': ' + str(error)[:250]})
        finally:
            if staging is not None:
                shutil.rmtree(staging, ignore_errors=True)
            REMOTE_UPLOAD_SLOTS.release()

'''
Convert numeric HTTP form fields into explicitly supported generator settings.
只轉換白名單中的 HTTP 參數，不執行客戶端提供的任何程式碼。
'''
def remote_generate(job_id, fields, image_paths):
    with REMOTE_ENGINE_LOCK:
        with REMOTE_JOB_LOCK:
            REMOTE_JOBS[job_id].update(status='running', _start_clock=time.monotonic())
        namespace = globals()
        keys = ['TASK', 'PROMPT', 'WIDTH', 'HEIGHT', 'SEED', 'LM_SEED', 'SECONDS', 'STEPS', 'CFG',
                'LYRICS', 'TAGS', 'STYLE', 'DURATION', 'GPU_ID', 'COT', 'ABC_PATH',
                'OUTPUT_NAME', 'REFERENCE_PATHS', 'ACTIVE_JOB', 'TASK_CONFIGS',
                'R2V_REF_DIR', 'NEGATIVE_PROMPT', 'SAMPLER', 'REFERENCE_SIZE',
                'OUTPUTS', 'output', 'OUTPUT_PATH', 'OUTPUT_MP4', 'LAST_OUTPUT', 'RUN_SEED',
                'QUANT', 'ENCODER_VARIANT']
        before = {key: copy.deepcopy(namespace.get(key)) for key in keys if key in namespace}
        missing = set(keys) - set(before)
        try:
            task = fields.get('task', REMOTE_ALLOWED_TASKS[0]).lower()
            numerics = {'seed': int, 'lm_seed': int, 'width': int, 'height': int, 'steps': int,
                        'seconds': float, 'cfg': float, 'duration': int,
                        'image_strength': float, 'reference_size': int, 'audio_shift': float}
            inputs = {}
            for key, value in fields.items():
                inputs[key] = numerics[key](value) if key in numerics else value
            for seed_key in ('seed', 'lm_seed'):
                if seed_key in inputs:
                    if not -1 <= inputs[seed_key] < 2**32:
                        raise ValueError(f'{seed_key} must be -1 or between 0 and 4294967295')
                    if inputs[seed_key] == -1:
                        inputs[seed_key] = secrets.randbelow(2**32)
            with REMOTE_JOB_LOCK:
                for seed_key in ('seed', 'lm_seed'):
                    if seed_key in inputs:
                        REMOTE_JOBS[job_id][seed_key + '_used'] = inputs[seed_key]
            for dimension in ('width', 'height'):
                if dimension in inputs and not 0 < inputs[dimension] <= 4096:
                    raise ValueError('Width and height must be between 1 and 4096')
            for key in ('seconds', 'duration'):
                if key in inputs and not 0 < inputs[key] <= 3600:
                    raise ValueError('Duration must be between 0 and 3600 seconds')
            if 'reference_size' in inputs and not 64 <= inputs['reference_size'] <= 2048:
                raise ValueError('Reference size must be between 64 and 2048')
            if 'ref_image_size' in inputs and inputs['ref_image_size'] not in ('match', 'max'):
                raise ValueError('Reference image mode must be match or max')
            if 'image_strength' in inputs and not 0 <= inputs['image_strength'] <= 1:
                raise ValueError('Image strength must be between 0 and 1')
            if 'steps' in inputs and not 1 <= inputs['steps'] <= 200:
                raise ValueError('Steps must be between 1 and 200')
            if REMOTE_KIND == 'video':
                if 'TASK_CONFIGS' in namespace:
                    if task not in namespace['TASK_CONFIGS']:
                        raise ValueError('Run this task configuration cell before API generation')
                    updated = copy.deepcopy(namespace['TASK_CONFIGS'][task])
                    for field in ('prompt', 'width', 'height', 'seconds', 'steps', 'seed',
                                  'audio_shift', 'ref_image_size', 'quant'):
                        if field in inputs:
                            updated[field] = inputs[field]
                    updated['output_name'] = f'remote_{job_id[:12]}.mp4'
                    updated['quant'] = inputs.get('quant') or namespace.get('QUANT', updated.get('quant'))
                    if task == 'r2v':
                        updated['video_audio'] = str(inputs.get('video_audio', 'false')).lower() == 'true'
                        updated['ref_videos'] = inputs.get('_ref_videos', [])
                        updated['ref_audios'] = inputs.get('_ref_audios', [])
                    if task == 'i2v':
                        updated['first_image'] = image_paths[0]
                    elif task == 'l2v':
                        updated['last_image'] = image_paths[0]
                    elif task == 'flf2v':
                        updated['first_image'], updated['last_image'] = image_paths
                    elif task == 'r2v':
                        updated['ref_images'] = image_paths
                        namespace['R2V_REF_DIR'] = REMOTE_ROOT / job_id
                    namespace['TASK_CONFIGS'][task] = updated
                    namespace['TASK'] = task
                else:
                    namespace['ACTIVE_JOB'] = {'mode': task, 'prompt': inputs.get('prompt', namespace.get('PROMPT', '')),
                                               'run_name': f'remote_{job_id[:12]}',
                                               'first_path': image_paths[0] if image_paths else None,
                                               'last_path': image_paths[1] if len(image_paths) > 1 else None}
                    for key in ('width', 'height', 'seconds', 'seed', 'image_strength'):
                        if key in inputs:
                            namespace[key.upper()] = inputs[key]
            else:
                for key in ('prompt', 'width', 'height', 'seed', 'lm_seed', 'steps', 'cfg',
                            'lyrics', 'tags', 'style', 'duration', 'negative_prompt',
                            'sampler', 'reference_size', 'cot'):
                    if key in inputs:
                        namespace[key.upper()] = inputs[key]
                if REMOTE_KIND == 'image_multi':
                    namespace['TASK'] = task
                    namespace['REFERENCE_PATHS'] = image_paths
                if REMOTE_KIND == 'music':
                    namespace['OUTPUT_NAME'] = f'remote_{job_id[:12]}.mp3'
                    if 'ABC_PATH' in namespace:
                        namespace['ABC_PATH'] = ''
                    if 'abc' in inputs:
                        score = (REMOTE_ROOT / job_id / 'input.abc')
                        score.write_text(str(inputs['abc']), encoding='utf-8')
                        namespace['ABC_PATH'] = str(score)
                if REMOTE_KIND in ('image_multi', 'image_text'):
                    namespace['OUTPUT_NAME'] = f'remote_{job_id[:12]}.png'
            for output_key in ('OUTPUTS', 'output', 'OUTPUT_PATH', 'OUTPUT_MP4', 'LAST_OUTPUT'):
                namespace.pop(output_key, None)
            '''
            Recheck the CUDA backend after queuing to prevent CPU-only generation.
            工作排入佇列後再次確認 CUDA，避免後端異常時改走純 CPU 生成。
            '''
            gpu_ready, gpu_message, _ = remote_gpu_status()
            if not gpu_ready:
                raise RuntimeError(gpu_message)
            inference_ready, inference_message = remote_inference_ready(task)
            if not inference_ready:
                raise RuntimeError(inference_message)
            exec(compile(REMOTE_SCRIPT, '<notebook-generation-cell>', 'exec'), namespace)
            paths = []
            if REMOTE_KIND == 'video':
                paths = [namespace.get('LAST_OUTPUT') or namespace.get('OUTPUT_MP4')]
            elif REMOTE_KIND == 'image_text':
                paths = [namespace.get('OUTPUT_PATH')]
            elif REMOTE_KIND == 'image_multi':
                paths = [namespace.get('output')]
            elif REMOTE_KIND == 'music':
                paths = namespace.get('OUTPUTS', []) if 'OUTPUTS' in namespace else [namespace.get('output')]
            results = [Path(path) for path in paths if path is not None]
            if REMOTE_KIND == 'music' and REMOTE_SERVICE_ID.startswith('06_YuE2_'):
                results += [sidecar for path in list(results)
                            for ext in ('.json', '.abc')
                            if (sidecar := path.with_suffix(ext)).is_file()]
            if not results or any(not path.is_file() for path in results):
                raise FileNotFoundError('Generation finished without a saved result file')
            folder = REMOTE_ROOT / job_id
            if len(results) == 1:
                target = folder / f'result{results[0].suffix.lower()}'
                shutil.copy2(results[0], target)
            else:
                target = folder / 'results.zip'
                with zipfile.ZipFile(target, 'w', compression=zipfile.ZIP_STORED) as archive:
                    for result in results:
                        archive.write(result, arcname=result.name)
            last_frame = None
            if REMOTE_KIND == 'video' and shutil.which('ffprobe') and shutil.which('ffmpeg'):
                try:
                    probe = subprocess.run(['ffprobe', '-v', 'error', '-count_frames',
                                            '-select_streams', 'v:0', '-show_entries',
                                            'stream=nb_read_frames', '-of', 'json', str(target)],
                                           check=True, text=True, capture_output=True, timeout=120)
                    index = int(json.loads(probe.stdout)['streams'][0]['nb_read_frames']) - 1
                    if index >= 0:
                        last_frame = folder / 'last_frame.png'
                        subprocess.run(['ffmpeg', '-y', '-loglevel', 'error', '-i', str(target),
                                        '-vf', rf'select=eq(n\,{index})', '-frames:v', '1',
                                        str(last_frame)], check=True)
                except (OSError, ValueError, KeyError, subprocess.CalledProcessError, subprocess.TimeoutExpired):
                    last_frame = None
            for original in results:
                if original.name.startswith(f'remote_{job_id[:12]}') and original != target:
                    try:
                        original.unlink(missing_ok=True)
                    except OSError:
                        pass
            comfy = globals().get('COMFY')
            if comfy is not None and (Path(comfy) / 'output').exists():
                for artifact in (Path(comfy) / 'output').rglob(f'remote_{job_id[:12]}*'):
                    if artifact.is_file():
                        try:
                            artifact.unlink()
                        except OSError:
                            pass
            with REMOTE_JOB_LOCK:
                REMOTE_JOBS[job_id].update(status='done', filename=target.name,
                                           bytes=target.stat().st_size, _path=str(target),
                                           last_frame_available=bool(last_frame),
                                           _last_frame=str(last_frame) if last_frame else None)
        except Exception as error:
            with REMOTE_JOB_LOCK:
                REMOTE_JOBS[job_id].update(status='error', error=f'{type(error).__name__}: {error}'[:2000])
        finally:
            for key in missing:
                namespace.pop(key, None)
            namespace.update(before)

'''
Use one GPU worker, regardless of the number of HTTP request threads.
不論有多少 HTTP 連線，都只有一個 GPU worker 消費佇列。
'''
def remote_worker():
    while True:
        job_id, fields, image_paths = REMOTE_QUEUE.get()
        try:
            remote_generate(job_id, fields, image_paths)
        except Exception as error:
            with REMOTE_JOB_LOCK:
                if job_id in REMOTE_JOBS:
                    REMOTE_JOBS[job_id].update(status='error', error=f'{type(error).__name__}: {error}'[:2000])
        finally:
            with REMOTE_JOB_LOCK:
                if job_id in REMOTE_JOBS:
                    REMOTE_JOBS[job_id]['finished_at'] = time.time()
            try:
                remote_remove_inputs(job_id)
            except OSError as error:
                print('API input cleanup warning:', str(error)[:180])
            finally:
                REMOTE_QUEUE.task_done()

if not (globals().get('REMOTE_WORKER') and REMOTE_WORKER.is_alive()):
    REMOTE_WORKER = threading.Thread(target=remote_worker, daemon=True)
    REMOTE_WORKER.start()

if not (globals().get('REMOTE_MAINTENANCE') and REMOTE_MAINTENANCE.is_alive()):
    REMOTE_MAINTENANCE = threading.Thread(target=remote_maintenance, daemon=True)
    REMOTE_MAINTENANCE.start()

old_tunnel = globals().get('REMOTE_NGROK_PROC')
if old_tunnel is not None and old_tunnel.poll() is None:
    old_tunnel.terminate()
    try:
        old_tunnel.wait(timeout=5)
    except subprocess.TimeoutExpired:
        old_tunnel.kill()
        old_tunnel.wait(timeout=5)

NGROK_URL = ''

'''
Install the official ngrok agent only if no usable binary exists.
系統尚無 ngrok 才下載官方執行檔，不重複安裝。
'''
ngrok_bin = shutil.which('ngrok') or str(ROOT / 'ngrok')
if not Path(ngrok_bin).is_file():
    url = 'https://bin.ngrok.com/c/bNyj1mQVY4c/ngrok-v3-stable-linux-amd64.tgz'
    with urllib.request.urlopen(url, timeout=90) as response:
        blob = response.read()
    with tarfile.open(fileobj=io.BytesIO(blob), mode='r:gz') as archive:
        entry = next((member for member in archive.getmembers() if Path(member.name).name == 'ngrok' and member.isfile()), None)
        if entry is None:
            raise RuntimeError('ngrok 官方下載檔案缺少執行檔')
        Path(ngrok_bin).write_bytes(archive.extractfile(entry).read())
    Path(ngrok_bin).chmod(0o755)

ngrok_token = getpass('輸入 ngrok Authtoken：').strip()
if not ngrok_token:
    raise ValueError('未輸入 ngrok Authtoken')

result = subprocess.run([ngrok_bin, 'config', 'add-authtoken', ngrok_token], capture_output=True, text=True)
if result.returncode:
    raise RuntimeError('ngrok token 設定失敗（請確認 Token 是否有效）')

del ngrok_token
result = subprocess.run([ngrok_bin, 'config', 'check'], capture_output=True, text=True)
if result.returncode:
    raise RuntimeError('ngrok 設定格式檢查失敗：' + result.stderr[-500:])

server = globals().get('REMOTE_SERVER')
server_thread = globals().get('REMOTE_SERVER_THREAD')
server_is_current = (
    server is not None
    and server_thread is not None
    and server_thread.is_alive()
    and getattr(server, 'notebook_id', None) == REMOTE_SERVICE_ID
)
if server is not None and not server_is_current:
    if server_thread is not None and server_thread.is_alive():
        server.shutdown()
    server.server_close()

if not server_is_current:
    try:
        REMOTE_SERVER = ThreadingHTTPServer(('127.0.0.1', REMOTE_PORT), RemoteHandler)
    except OSError as error:
        raise RuntimeError(f'本機 API Port {REMOTE_PORT} 已被其他服務占用') from error
    REMOTE_SERVER.notebook_id = REMOTE_SERVICE_ID
    REMOTE_SERVER.daemon_threads = True
    REMOTE_SERVER_THREAD = threading.Thread(target=REMOTE_SERVER.serve_forever, daemon=True)
    REMOTE_SERVER_THREAD.start()
    print('已啟動新的本機 API Server。')
else:
    REMOTE_SERVER.RequestHandlerClass = RemoteHandler
    print('已沿用本機 API Server；現有生成工作與結果保持不變。')

with urllib.request.urlopen(f'http://127.0.0.1:{REMOTE_PORT}/health', timeout=4) as response:
    if json.load(response).get('ok') is not True:
        raise RuntimeError('本機 API Server 尚未就緒')

'''
Use the agent inspection API to obtain the actual public URL.
透過 ngrok 本機檢查 API 取得實際對外 HTTPS 網址。
'''
NGROK_LOG = REMOTE_ROOT / 'ngrok.log'
with NGROK_LOG.open('w') as log:
    REMOTE_NGROK_PROC = subprocess.Popen([ngrok_bin, 'http', str(REMOTE_PORT)], stdout=log,
                                         stderr=subprocess.STDOUT, start_new_session=True)

NGROK_URL = ''
for attempt in range(25):
    if REMOTE_NGROK_PROC.poll() is not None:
        break
    try:
        with urllib.request.urlopen('http://127.0.0.1:4040/api/tunnels', timeout=2) as response:
            tunnels = json.load(response).get('tunnels', [])
        NGROK_URL = next((item['public_url'] for item in tunnels
                          if item.get('public_url', '').startswith('https://') and
                          str(REMOTE_PORT) in item.get('config', {}).get('addr', '')), '')
        if NGROK_URL:
            break
    except (OSError, ValueError, KeyError):
        pass
    time.sleep(1)

if not NGROK_URL:
    raise RuntimeError('ngrok URL 建立失敗：\n' + NGROK_LOG.read_text(errors='replace')[-2500:])

print('ngrok HTTPS URL:', NGROK_URL)
print('提醒：公開的 ngrok API URL 不需要額外驗證，請只分享給授課學生。')
print('GET /health | GET /schema | POST /generate | GET /jobs/{id} | GET /jobs/{id}/file')
print('影片輸出另提供：GET /jobs/{id}/last-frame')
print('✓ 第 14 節完成：選用 FIFO／ngrok API 已啟動。')


15. （選用）遠端 API 與 client 使用說明


In [ ]:
'''
Query the active bridge for supported parameters and task names.
從正在執行的橋接服務讀取實際支援的參數與模式。
'''
import json
import urllib.request
request = urllib.request.Request(f'http://127.0.0.1:{REMOTE_PORT}/schema')
with urllib.request.urlopen(request, timeout=5) as response:
    schema = json.load(response)

print(json.dumps(schema, ensure_ascii=False, indent=2))
print('用法：POST NGROK_URL + "/generate"（JSON 或 multipart/images），取得 job_id；')
print('輪詢 GET /jobs/{job_id}；完成後下載 GET /jobs/{job_id}/file。')
print('本機 requests 範例：06_MiniMax_H3_影片生成_client.py（與本 Notebook 同名對應）。')
print('使用前只需在對應的 client.py 填入 API_URL。')
print('本服務使用單一 FIFO worker，遠端請求不會同時啟動多筆 GPU 推論。')
print('✓ 第 15 節完成：API Schema 和 client 說明已顯示。')
